# Etapa 2 — Pré-processamento do CIC-DDoS2019 (ClickHouse + pipeline Python) — v8

**Regra de ouro:** as tabelas originais (`data_*` = treino, `data2_*` = teste) **nunca são alteradas**. Toda escrita só é aceita em tabelas `prep_*` (trava abaixo).

**Novidades da v7 (idempotência e prevenção de erros):**
- Cada tabela `prep_*` só é refeita se **não existir, estiver incompleta ou tiver sido feita com outra configuração** (features, mapa de rótulos). Existir **não** basta: há validador de contagem e assinatura (`construir`). Tabelas de execuções anteriores são adotadas se válidas.
- Resultados de análises (ausentes, constantes, duplicadas, sobreposição, correlação, parâmetros) ficam salvos em `prep_estado`; reexecutar uma célula só reaproveita o resultado. **Reiniciou o kernel? Rode tudo de cima para baixo: o que está pronto é reaproveitado em segundos.**
- Antes de etapas pesadas o notebook espera merges/mutations/consultas anteriores terminarem e adapta as configurações de memória ao servidor; erro de memória (241/452) é repetido com configuração mais conservadora.
- Correção: a trava bloqueava INSERTs cujo texto continha `;` dentro de aspas (ex.: motivos de `prep_selecao_features`); agora literais são ignorados na checagem.
- Correção: `restaurar_selecao` agora restaura também `CONSTANTES`, `DUPLICADAS` e `REMOVIDAS_AUSENTES` (antes dava `NameError` após reiniciar o kernel); `salvar_selecao` não apaga mais `FEATURES_REDUZIDO` ao reexecutar.
- Novo bloco 4.3: investigação dos conflitos de rótulo (diagnóstico com IPs/portas/timestamp da tabela auxiliar).

Para **forçar** refazer algo: `RECRIAR.add("nome_da_tabela_ou_etapa")` (ex.: `"prep_hash"`, `"b7_correlacao_auc"`). O nome é removido de `RECRIAR` automaticamente após refazer.

**Fluxo:**

| # | Bloco | Tabelas criadas |
|---|-------|-----------------|
| 0 | Conexão, trava, configuração, pré-checagens, infra (somente leitura + estado) | `prep_estado` |
| 1 | Camada de preparação: tipos numéricos, rótulo original/binário, origem, ids em tabela auxiliar | `prep_stg`, `prep_aux_ids` |
| 2 | Inválidos: vazio, NaN, ±Inf, texto não numérico → ausente; auditoria e política | `prep_aud_invalidos` |
| 3 | Constantes e colunas exatamente duplicadas (decisão **só no treino**, teste apenas auditoria) | `prep_selecao_features` |
| 4 | Hash das features, duplicatas, conflitos de rótulo, sobreposição treino×teste | `prep_hash`, `prep_grp_treino`, `prep_keep_treino` |
| 5 | Rótulos: `label_original`, `label_multi` (padronizado), `label_binary` | `prep_mapa_rotulos`, `prep_classes_resumo` |
| 6 | Bases finais (treino deduplicado; teste oficial intacto) + views binária/multiclasse | `prep_final_treino`, `prep_final_teste`, `prep_vw_*` |
| 7 | Redundância por correlação e AUC treino×teste (somente treino decide) | `prep_selecao_features` (atualizada) |
| 8 | Parâmetros (mediana, média, desvio) **só do treino** + `PreProc` (RF e SVM) | `prep_parametros` + JSON |
| 9 | Validação final e relatório | — |

**Segurança de carga:** o transporte HTTP não reenvia automaticamente operações de escrita após falha de conexão, pois isso pode duplicar linhas. Se a conexão cair durante uma escrita, o notebook espera a consulta do servidor terminar e **valida a contagem** antes de repetir.


---
**Novidades da v8 (correção do erro de carga e blindagem das etapas pesadas):**
- **Carga atômica:** cada INSERT de `prep_stg`/`prep_aux_ids` grava um bloco único de até `CHUNK_LINHAS` linhas (tudo ou nada) e é conferido pela contagem. Se a conexão cair, espera o servidor terminar e confere se gravou (não duplica nem perde). O erro real do servidor é sempre mostrado.
- **Modo de carga:** `MODO_CARGA = "auto"` sonda se o servidor lê só as linhas pedidas ao filtrar por `_part_offset` (rápido). Se não ler, usa o plano B por hash (lento, como no v7).
- **Etapas pesadas em unidades curtas e retomáveis:** `prep_hash`, `prep_grp_treino`, `prep_keep_treino`, `prep_final_*` e `prep_aud_invalidos` são construídas por unidades; ao reexecutar, só o que falta é feito.
- **Trava entre etapas:** `exigir(...)` no início das células dependentes; se a etapa anterior não terminou com ✔, a célula para com mensagem clara.
- **Correção importante:** `sipHash128` devolve NULL se qualquer argumento Nullable for NULL, e o NULL virava 16 bytes zero. Linhas com valor inválido (∞, vazio, texto) colidiam no mesmo hash e eram deduplicadas em uma só por rótulo. O hash agora usa `ifNull(coluna, 0)` + indicador de ausência.
- A tabela `prep_stg` muda de layout de partição: se existir uma versão antiga, ela é recriada automaticamente (as originais nunca são tocadas).


**v8.1:** `bwd_iat_mean`, `bwd_packets_s`, `active_mean` e `idle_mean` passaram de `Int64` para `Float64` (são médias/taxas com decimais; como inteiras davam NULL). A carga agora detecta mudança de tipo de coluna e recria `prep_stg` sozinha, e checa a conversão já na primeira origem carregada (falha em minutos, não horas).


**v8.2:** `data2_syn` tem 482.799 linhas 100% idênticas (mesmo `unnamed_0`, `flow_id`, IPs, portas e timestamp), o que repetia o `id_linha`. Origens listadas em `ID_FISICO` passam a incluir a posição física da linha no id (único por linha); as demais mantêm o id já carregado. Só essas origens são recarregadas. As linhas repetidas **são mantidas** no teste (a especificação não deduplica o teste).


**v8.3:** erros HTTP agora mostram o código e o corpo do ClickHouse; consultas longas de validação/ausentes/constantes foram divididas em blocos de colunas; adicionados o **Guia visual (antes → depois)** e células **"Exemplo ao vivo"** com linhas reais do seu banco em cada etapa.


**v8.4:** as bases finais usam unidades bem menores (`LINHAS_POR_UNIDADE_FINAL`, `FAIXAS_MIN_TREINO`) e settings de pouca memória, e cada unidade espera (até `ESPERA_MEM_UNIDADE_MAX`) a memória do servidor e os merges internos baixarem, mostrando o progresso, em vez de repetir às cegas. Motivo: o lado direito da junção do treino final era ~1/3 de todas as linhas mantidas (~2,8 milhões) em `data_drdos_dns`.


**v8.4.1** (= v8.4 + salvaguarda de disco + overview final): **nenhuma tabela é apagada**. Se o disco encher (`Code 243` ou pouco espaço livre), o notebook tenta, uma vez e só o necessário, esvaziar **logs internos** do ClickHouse (`LOGS_AUTOMATICOS`, do maior para o menor); se não houver permissão ou não bastar, para na hora com o diagnóstico. A última célula traz o **overview das bases finais** (linhas, colunas, tamanho em disco, memória estimada, classes, marcas).


**v8.4.2:** `rodar_pesado` agora trata o erro de memória (HTTP 500 com `Code: 241`) esperando a memória do servidor baixar e tentando as variantes leves de configuração; a checagem de duplicatas do treino final tem plano B com hash de 64 bits.


**v8.10:** o prefixo `DrDoS_` é removido de todos os rótulos do treino nas views unificadas (`REMOVER_PREFIXO_DRDOS`); `label_original` não muda. **v8.9:** novas seções **11** (unificação de classes **por nome e documentação**, sem olhar o teste para decidir), **12** (seleção de atributos por AUC + informação mútua + correlação) e **13** (validação e guia de views). Nada existente é alterado: as bases finais e as views antigas continuam iguais; as novas views têm sufixos `_multi_unif`, `_binario_sel` e `_multi_unif_sel`.


## Guia visual: o que acontece com os dados, etapa por etapa (antes → depois)

> **Como ler:** os **números de linhas e contagens** são **reais** (da sua EDA e das suas execuções). Os **valores das células** nas tabelas abaixo são **ilustrativos**, no formato real do CIC-DDoS2019 (as tabelas originais guardam **tudo como texto**). Depois de cada etapa há uma célula **"Exemplo ao vivo"** que busca **linhas reais** do seu ClickHouse e mostra o antes e o depois.

| Etapa | Antes (tabela original) | Depois | O que mudou e por quê |
|---|---|---|---|
| **1. Tipos** (`prep_stg`) | `flow_bytes_s = 'Infinity'`, `bwd_iat_mean = '8.5'`, `total_backward_packets = '12'`, `label = ' Syn'` | `flow_bytes_s = NULL` (+ `flow_bytes_s_status = 'pos_inf'`), `bwd_iat_mean = 8.5` (Float64), `total_backward_packets = 12` (Int64), `label_original = 'Syn'` | texto vira número; ∞/vazio/texto viram **NULL com o motivo guardado**; espaços do rótulo saem. Real: 4.092.348 `+Inf` e 249.152 vazios na EDA |
| **2. Identificadores** | `unnamed_0 = '1234'`, `flow_id = '172.16.0.5-192.168.50.1-443-80-6'`, IPs, portas, `timestamp` | saem da matriz; ficam em `prep_aux_ids`; a linha ganha `id_linha` (hash de 64 bits) | o modelo não "decora" IP, porta ou horário; a rastreabilidade é mantida |
| **3. Features inúteis** | 80 features | 80 − constantes − colunas idênticas − ausentes > 50% = `FEATURES_BASE` | ex.: `bwd_psh_flags` é sempre 0 → removida; `subflow_fwd_packets` == `total_fwd_packets` → fica só uma. Real: 13 constantes na EDA |
| **4. Deduplicação (só treino)** | 5 linhas com as **mesmas features e o mesmo rótulo** (`DrDoS_DNS`), mas `flow_id`/`timestamp` diferentes | 1 linha com `n_repeticoes = 5` | Real: 59.927.555 de 70.427.637 linhas (85,09%) eram repetidas em features+rótulo |
| **5. Rótulos** | `UDP-lag` (treino, 366.461 linhas) e `UDPLag` (teste, 1.873 linhas); `BENIGN` | `label_multi = 'UDPLag'` nos dois; `label_binary = 0` para BENIGN e `1` para ataque | `label_original` é sempre preservado |
| **6. Conflitos** | mesmas features aparecem com `BENIGN` **e** com um ataque | `conflito_rotulo = 1` / `conflito_binario = 1` | marcados, **não apagados** (análise de sensibilidade) |
| **7. Sobreposição** | linha do teste com features iguais a uma do treino | `sobrepoe_treino = 1` | mede o vazamento treino → teste; o teste **não** é deduplicado |
| **8. Parâmetros** | `flow_bytes_s = NULL` em uma linha | RF: recebe a **mediana do treino**; SVM: `z = (x − μ) / σ` com μ e σ **do treino** | só o treino define mediana, μ e σ; o teste usa os mesmos |

## 0. Conexão e trava de segurança

In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse-production-e204.up.railway.app:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")


In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


## 0.1 Configuração, constantes da EDA e funções auxiliares

In [4]:
import math, hashlib, time, warnings
warnings.filterwarnings("ignore", message="DataFrame is highly fragmented")


def blocos_de(lista, n=20):
    """Divide uma lista em blocos de n itens (consultas mais curtas = menos risco de o proxy derrubar a conexão)."""
    return [lista[i:i + n] for i in range(0, len(lista), n)]


import numpy as np
from datetime import datetime

# ======================= DECISÕES / PARÂMETROS DO PIPELINE =======================
RECRIAR = set()                 # v7: normalmente NÃO precisa mexer (tabelas incompletas/desatualizadas são detectadas e refeitas sozinhas). Para forçar, ex.:
# RECRIAR = {"prep_stg", "prep_aux_ids"}  # Após erro de contagem na carga inicial; execute desde o bloco 1.
# Para recalcular após mudar a seleção de features, use temporariamente:
# RECRIAR = {"prep_hash", "prep_grp_treino", "prep_keep_treino", "prep_final_treino", "prep_final_teste", "prep_parametros"}
# Depois da execução completa, volte para set(). Não é necessário recriar prep_stg: as colunas brutas não mudaram.
UNIFICAR_UDPLAG = True          # UDP-lag (treino) e UDPLag (teste) -> UDPLag. Revise a evidência do bloco 5 antes de manter True.
MAPA_ROTULOS_EXTRA = {}         # decisões posteriores, ex.: {"DrDoS_LDAP": "LDAP"} (só após ver a evidência do bloco 4)
CLASSES_FORA_MULTI = []         # classes a excluir do multiclasse principal, ex.: ["WebDDoS"] (decisão posterior; nada é excluído automaticamente)
LIMITE_AUSENTES_REMOVER = 0.50  # feature com mais de 50% de ausentes no treino sai da matriz (documentado)
LIMIAR_CORR = 0.95              # |r| para marcar redundância (calculado no treino deduplicado)
AMOSTRA_MOD = 50                # amostra determinística (1/50) para correlação e AUC
SETTINGS_PESADO = ""            # ex.: "SETTINGS max_bytes_before_external_group_by = 20000000000" se faltar memória no servidor
ID_FISICO = {"data2_syn"}      # v8.2: origens com linhas 100% idênticas (inclusive unnamed_0/flow_id/IPs/portas/timestamp): o id_linha inclui a posição física
CHUNK_LINHAS = 100_000          # v8: linhas por INSERT na carga (bloco único = atômico). Menor = menos memória por INSERT
MODO_CARGA = "auto"             # v8: "auto" (sonda o servidor), "offset" (chunks por _part_offset, rápido) ou "hash" (plano B, lento)
LINHAS_POR_UNIDADE = 2_000_000  # v8: linhas por unidade nas etapas hash/finais (cada unidade é um INSERT curto, verificado e retomável)
LIMPAR_LOGS_SE_DISCO_CHEIO = True    # v8.4.1: SÓ se o disco encher, esvazia logs INTERNOS do ClickHouse (system.*; não são dados do TCC). Nenhuma tabela prep_* ou data_* é apagada.
LOGS_BANCO = "system"
LOGS_AUTOMATICOS = ("trace_log", "text_log", "asynchronous_metric_log", "metric_log", "query_thread_log", "processors_profile_log", "opentelemetry_span_log")   # só logs de profiling/métricas/mensagens; query_log, part_log e session_log NÃO são tocados
LOGS_MIN_MIB = 256                   # só limpa se esses logs somarem pelo menos isto (senão não adiantaria)
LOGS_GANHO_MIN_MIB = 128             # considera que liberou espaço se o disco ganhou pelo menos isto
ESPACO_MIN_GIB = None                # espaço livre mínimo antes de cada unidade. None = automático (8% do disco, entre 0,75 e 4 GiB)
LINHAS_POR_UNIDADE_FINAL = 500_000   # v8.4: linhas de origem por unidade das BASES FINAIS (menor = menos memória por INSERT)
FAIXAS_MIN_TREINO = 32               # v8.4: mínimo de faixas por origem no treino final (o lado direito da junção é ~ total_mantidas / faixas)
ESPERA_MEM_UNIDADE_MAX = 5400        # v8.4: espera máxima (s) pela memória do servidor baixar antes de cada unidade
LINHAS_POR_BUCKET = 3_000_000   # v8: grupos por consulta nas etapas de deduplicação/sobreposição (menor = menos memória)
ARTEFATOS = REPO_ROOT / "artefatos_preprocessamento"; ARTEFATOS.mkdir(exist_ok=True)
RELATORIO = {}                  # números e decisões, salvo em JSON no fim

# ======================= FATOS DA EDA (usados só para reconciliar) =======================
ID_COLS = ["unnamed_0", "flow_id", "source_ip", "source_port", "destination_ip", "destination_port", "timestamp"]
FEATURES_ORDEM = ['protocol', 'flow_duration', 'total_fwd_packets', 'total_backward_packets', 'total_length_of_fwd_packets', 'total_length_of_bwd_packets', 'fwd_packet_length_max', 'fwd_packet_length_min', 'fwd_packet_length_mean', 'fwd_packet_length_std', 'bwd_packet_length_max', 'bwd_packet_length_min', 'bwd_packet_length_mean', 'bwd_packet_length_std', 'flow_bytes_s', 'flow_packets_s', 'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min', 'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max', 'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length', 'bwd_header_length', 'fwd_packets_s', 'bwd_packets_s', 'min_packet_length', 'max_packet_length', 'packet_length_mean', 'packet_length_std', 'packet_length_variance', 'fin_flag_count', 'syn_flag_count', 'rst_flag_count', 'psh_flag_count', 'ack_flag_count', 'urg_flag_count', 'cwe_flag_count', 'ece_flag_count', 'down_up_ratio', 'average_packet_size', 'avg_fwd_segment_size', 'avg_bwd_segment_size', 'fwd_header_length_1', 'fwd_avg_bytes_bulk', 'fwd_avg_packets_bulk', 'fwd_avg_bulk_rate', 'bwd_avg_bytes_bulk', 'bwd_avg_packets_bulk', 'bwd_avg_bulk_rate', 'subflow_fwd_packets', 'subflow_fwd_bytes', 'subflow_bwd_packets', 'subflow_bwd_bytes', 'init_win_bytes_forward', 'init_win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward', 'active_mean', 'active_std', 'active_max', 'active_min', 'idle_mean', 'idle_std', 'idle_max', 'idle_min', 'simillarhttp', 'inbound']
FEATURES_INT = set(['protocol', 'total_backward_packets', 'total_length_of_bwd_packets', 'bwd_packet_length_max', 'bwd_packet_length_min', 'flow_iat_min', 'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length', 'bwd_header_length', 'fin_flag_count', 'syn_flag_count', 'rst_flag_count', 'psh_flag_count', 'ack_flag_count', 'urg_flag_count', 'cwe_flag_count', 'ece_flag_count', 'down_up_ratio', 'fwd_header_length_1', 'fwd_avg_bytes_bulk', 'fwd_avg_packets_bulk', 'fwd_avg_bulk_rate', 'bwd_avg_bytes_bulk', 'bwd_avg_packets_bulk', 'bwd_avg_bulk_rate', 'subflow_bwd_packets', 'subflow_bwd_bytes', 'init_win_bytes_forward', 'init_win_bytes_backward', 'min_seg_size_forward', 'active_max', 'active_min', 'idle_max', 'idle_min', 'inbound'])     # v8.1: bwd_iat_mean, bwd_packets_s, active_mean e idle_mean são MÉDIAS/TAXAS com decimais -> Float64
COLS_STATUS = ["flow_bytes_s", "flow_packets_s", "simillarhttp"]    # únicas colunas com inválidos segundo a EDA (05_qualidade_long)
CONSTANTES_EDA = ['bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fin_flag_count', 'psh_flag_count', 'ece_flag_count', 'fwd_avg_bytes_bulk', 'fwd_avg_packets_bulk', 'fwd_avg_bulk_rate', 'bwd_avg_bytes_bulk', 'bwd_avg_packets_bulk', 'bwd_avg_bulk_rate', 'simillarhttp']
PARES_EDA = [('total_fwd_packets', 'subflow_fwd_packets', 1.0), ('total_length_of_fwd_packets', 'subflow_fwd_bytes', 1.0), ('total_backward_packets', 'subflow_bwd_packets', 1.0), ('total_length_of_bwd_packets', 'subflow_bwd_bytes', 1.0), ('fwd_psh_flags', 'rst_flag_count', 1.0), ('bwd_packet_length_mean', 'avg_bwd_segment_size', 1.0), ('fwd_packet_length_mean', 'avg_fwd_segment_size', 1.0), ('fwd_header_length', 'fwd_header_length_1', 1.0), ('flow_packets_s', 'fwd_packets_s', 0.999716), ('fwd_packet_length_min', 'min_packet_length', 0.999709), ('flow_duration', 'fwd_iat_total', 0.998692), ('packet_length_mean', 'avg_fwd_segment_size', 0.998012), ('fwd_packet_length_mean', 'packet_length_mean', 0.998012), ('fwd_packet_length_min', 'fwd_packet_length_mean', 0.997345), ('fwd_packet_length_min', 'avg_fwd_segment_size', 0.997345), ('packet_length_mean', 'average_packet_size', 0.997258), ('fwd_packet_length_mean', 'min_packet_length', 0.99704), ('min_packet_length', 'avg_fwd_segment_size', 0.99704), ('flow_iat_min', 'fwd_iat_min', 0.996656), ('average_packet_size', 'avg_fwd_segment_size', 0.996332), ('fwd_packet_length_mean', 'average_packet_size', 0.996332), ('min_packet_length', 'average_packet_size', 0.995969), ('fwd_packet_length_min', 'average_packet_size', 0.995895), ('flow_iat_max', 'fwd_iat_max', 0.995718), ('min_packet_length', 'packet_length_mean', 0.995126), ('fwd_packet_length_min', 'packet_length_mean', 0.995109), ('idle_mean', 'idle_min', 0.986777), ('fwd_packet_length_max', 'packet_length_mean', 0.986707), ('fwd_packet_length_max', 'fwd_packet_length_mean', 0.986042), ('fwd_packet_length_max', 'avg_fwd_segment_size', 0.986042), ('idle_mean', 'idle_max', 0.985052), ('flow_iat_std', 'fwd_iat_std', 0.983272), ('total_length_of_fwd_packets', 'act_data_pkt_fwd', 0.982658), ('subflow_fwd_bytes', 'act_data_pkt_fwd', 0.982658), ('fwd_packet_length_max', 'average_packet_size', 0.982286), ('fwd_iat_max', 'idle_max', 0.97993), ('flow_iat_max', 'idle_max', 0.978756), ('fwd_packet_length_max', 'fwd_packet_length_min', 0.977507), ('fwd_packet_length_max', 'min_packet_length', 0.977179), ('flow_iat_mean', 'fwd_iat_mean', 0.970816), ('fwd_iat_max', 'idle_mean', 0.965042), ('flow_iat_max', 'idle_mean', 0.964197), ('total_fwd_packets', 'act_data_pkt_fwd', 0.964182), ('subflow_fwd_packets', 'act_data_pkt_fwd', 0.964182), ('flow_iat_std', 'fwd_iat_mean', 0.96035), ('bwd_packet_length_max', 'bwd_packet_length_std', 0.953867), ('fwd_packet_length_max', 'max_packet_length', 0.952314)]                                              # (feature_a, feature_b, r) — EDA sobre treino+teste juntos
ESPERADO_REGISTROS = {'data2_ldap': 2113234, 'data2_mssql': 5775786, 'data2_netbios': 3455899, 'data2_portmap': 191694, 'data2_syn': 4320541, 'data2_udp': 3782206, 'data2_udplag': 725165, 'data_drdos_dns': 5074413, 'data_drdos_ldap': 2181542, 'data_drdos_mssql': 4524498, 'data_drdos_netbios': 4094986, 'data_drdos_ntp': 1217007, 'data_drdos_snmp': 5161377, 'data_drdos_ssdp': 2611374, 'data_drdos_udp': 3136802, 'data_syn': 1582681, 'data_tftp': 20107827, 'data_udplag': 370605}
ESPERADO_CLASSES = {'data2_ldap|NetBIOS': 202919, 'data2_ldap|BENIGN': 5124, 'data2_ldap|LDAP': 1905191, 'data2_mssql|MSSQL': 5763061, 'data2_mssql|BENIGN': 2794, 'data2_mssql|LDAP': 9931, 'data2_netbios|NetBIOS': 3454578, 'data2_netbios|BENIGN': 1321, 'data2_portmap|Portmap': 186960, 'data2_portmap|BENIGN': 4734, 'data2_syn|BENIGN': 35790, 'data2_syn|Syn': 4284751, 'data2_udp|MSSQL': 24392, 'data2_udp|BENIGN': 3134, 'data2_udp|UDP': 3754680, 'data2_udplag|UDPLag': 1873, 'data2_udplag|BENIGN': 4068, 'data2_udplag|Syn': 606749, 'data2_udplag|UDP': 112475, 'data_drdos_dns|BENIGN': 3402, 'data_drdos_dns|DrDoS_DNS': 5071011, 'data_drdos_netbios|BENIGN': 1707, 'data_drdos_netbios|DrDoS_NetBIOS': 4093279, 'data_drdos_ntp|BENIGN': 14365, 'data_drdos_ntp|DrDoS_NTP': 1202642, 'data_drdos_snmp|BENIGN': 1507, 'data_drdos_snmp|DrDoS_SNMP': 5159870, 'data_drdos_udp|BENIGN': 2157, 'data_drdos_udp|DrDoS_UDP': 3134645, 'data_syn|BENIGN': 392, 'data_syn|Syn': 1582289, 'data_udplag|UDP-lag': 366461, 'data_udplag|BENIGN': 3705, 'data_udplag|WebDDoS': 439, 'data_drdos_mssql|BENIGN': 2006, 'data_drdos_mssql|DrDoS_MSSQL': 4522492, 'data_drdos_ssdp|BENIGN': 763, 'data_drdos_ssdp|DrDoS_SSDP': 2610611, 'data_drdos_ldap|BENIGN': 1612, 'data_drdos_ldap|DrDoS_LDAP': 2179930, 'data_tftp|TFTP': 20082580, 'data_tftp|BENIGN': 25247}                                          # "tabela|label" -> registros
EDA_DUP_C = {'data2_ldap': [2113234, 2089294], 'data2_netbios': [3455899, 3444572], 'data2_portmap': [191694, 185603], 'data2_syn': [4320541, 3840360], 'data2_udp': [3782206, 2484521], 'data2_udplag': [725165, 585344], 'data_drdos_dns': [5074413, 4958078], 'data_drdos_ldap': [2181542, 2150031], 'data_drdos_mssql': [4524498, 4315609], 'data_drdos_ntp': [1217007, 90661], 'data_drdos_snmp': [5161377, 5037668], 'data_drdos_ssdp': [2611374, 1719413], 'data_drdos_udp': [3136802, 2059389], 'data_syn': [1582681, 1426770], 'data_udplag': [370605, 277529], 'data2_mssql': [5775786, 5501042], 'data_drdos_netbios': [4094986, 4073854], 'data_tftp': [20107827, 15687817]}                                                # tabela -> [registros, repetidos (features+rótulo, sem índice)]
assert len(FEATURES_ORDEM) == 80 and len(CONSTANTES_EDA) == 13 and len(PARES_EDA) == 47

def conjunto_de(tabela):        # divisão oficial: data_* = treino ; data2_* = teste
    return "teste" if tabela.startswith("data2_") else "treino"

# ======================= AUXILIARES =======================
def q(c): return "`" + c + "`"
def lit(v): return "'" + str(v).replace("\\", "\\\\").replace("'", "\\'") + "'"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def existe(t):
    return int(consultar(f"SELECT toString(count()) AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(t)}")["n"][0]) > 0

def n_linhas(t, where="1"):
    return int(consultar(f"SELECT toString(count()) AS n FROM `{t}` WHERE {where}")["n"][0])

def vai_criar(t):
    if existe(t):
        if t in RECRIAR:
            executar(f"DROP TABLE IF EXISTS `{t}`"); RECRIAR.discard(t); return True
        print(f"  = `{t}` já existe e foi mantida (para refazer: RECRIAR.add('{t}'))"); return False
    return True

def inserir_df(t, df):
    cols = ", ".join(f"`{c}`" for c in df.columns)
    def v(x):
        if x is None or (isinstance(x, float) and math.isnan(x)): return "NULL"
        if isinstance(x, (bool, np.bool_)): return "1" if x else "0"
        if isinstance(x, (int, np.integer)): return str(int(x))
        if isinstance(x, (float, np.floating)): return repr(float(x))
        return lit(x)
    linhas = ",".join("(" + ",".join(v(x) for x in r) + ")" for r in df.itertuples(index=False, name=None))
    executar(f"INSERT INTO `{t}` ({cols}) VALUES {linhas}")

def tipo_ch(c, anulavel=True):
    base = "Int64" if c in FEATURES_INT else "Float64"
    return f"Nullable({base})" if anulavel else base

def expr_label_multi(col):
    """Rótulo multiclasse padronizado a partir de label_original, conforme prep_mapa_rotulos (definido no bloco 5)."""
    mapa = dict(globals().get("MAPA", {}))
    if not mapa: return f"toString({col})"
    ks = "[" + ",".join(lit(k) for k in mapa) + "]"; vs = "[" + ",".join(lit(v) for v in mapa.values()) + "]"
    return f"transform(toString({col}), {ks}, {vs}, toString({col}))"

print("Configuração carregada. Conjuntos:", {"treino": "data_*", "teste": "data2_*"})

Configuração carregada. Conjuntos: {'treino': 'data_*', 'teste': 'data2_*'}


## 0.2 Pré-checagens (somente leitura): tabelas, contagens e colunas

In [5]:
orig = consultar("SELECT name FROM system.tables WHERE database = currentDatabase() AND match(name, '^data2?_') ORDER BY name")["name"].tolist()
faltam, sobram = set(ESPERADO_REGISTROS) - set(orig), set(orig) - set(ESPERADO_REGISTROS)
assert not faltam and not sobram, f"tabelas diferentes da EDA. faltam={faltam} sobram={sobram}"
ORIGENS = sorted(orig)

cont = {t: n_linhas(t) for t in ORIGENS}
cmp_ = pd.DataFrame({"tabela": ORIGENS, "conjunto": [conjunto_de(t) for t in ORIGENS],
                     "registros": [cont[t] for t in ORIGENS], "esperado_eda": [ESPERADO_REGISTROS[t] for t in ORIGENS]})
cmp_["ok"] = cmp_.registros == cmp_.esperado_eda
display(cmp_)
res = cmp_.groupby("conjunto")[["registros"]].sum(); display(res)
assert cmp_.ok.all(), "contagem diferente da EDA"
N_TREINO, N_TESTE = int(res.loc["treino", "registros"]), int(res.loc["teste", "registros"])
print(f"Total: {N_TREINO + N_TESTE:,} (treino {N_TREINO:,} | teste {N_TESTE:,})")

esperadas = set(ID_COLS) | set(FEATURES_ORDEM) | {"label"}
cols_orig = consultar("SELECT `table` AS tabela, name AS coluna, type AS tipo FROM system.columns WHERE database = currentDatabase() AND match(`table`, '^data2?_')")
for t, g in cols_orig.groupby("tabela"):
    assert not (esperadas - set(g["coluna"])), f"{t}: colunas ausentes {esperadas - set(g['coluna'])}"
print("Tipos declarados nas originais:", cols_orig["tipo"].value_counts().to_dict())
RELATORIO["registros"] = {"treino": N_TREINO, "teste": N_TESTE}

,tabela,conjunto,registros,esperado_eda,ok
0,data2_ldap,teste,2113234,2113234,True
1,data2_mssql,teste,5775786,5775786,True
2,data2_netbios,teste,3455899,3455899,True
3,data2_portmap,teste,191694,191694,True
4,data2_syn,teste,4320541,4320541,True
5,data2_udp,teste,3782206,3782206,True
6,data2_udplag,teste,725165,725165,True
7,data_drdos_dns,treino,5074413,5074413,True
8,data_drdos_ldap,treino,2181542,2181542,True
9,data_drdos_mssql,treino,4524498,4524498,True


,registros
conjunto,
teste,20364525
treino,50063112


Total: 70,427,637 (treino 50,063,112 | teste 20,364,525)
Tipos declarados nas originais: {'String': 1584}


## 0.3 Infraestrutura v7: estado persistente, espera do servidor e construção idempotente

- **Nada é refeito se já está pronto e válido.** Cada tabela `prep_*` é construída por `construir(...)`: se ela existe, o código confere (a) a *assinatura* da configuração com que foi feita (features, mapa de rótulos, etc.) e (b) um *validador* de completude (contagens que precisam fechar com o treino/teste). Só refaz se algo não bate.
- **Tabelas de execuções anteriores (sem marca)** são *adotadas* se passarem no validador (não se perdem horas de processamento).
- **Resultados de análises** (constantes, duplicadas, ausentes, sobreposição, correlação, parâmetros) ficam salvos na tabela `prep_estado`; reexecutar a célula só reaproveita o resultado.
- **Espera do servidor:** antes de cada etapa pesada o notebook espera merges/mutations internos e consultas anteriores sobre a mesma tabela terminarem (como no seu outro código: "aguardando terminarem (máx 600s)").
- **Memória:** o limite do servidor é descoberto (cgroup/RAM/`max_server_memory_usage`), e as consultas pesadas usam limites compatíveis; se der erro 241/452 elas são repetidas com configurações mais conservadoras.


In [6]:
# ======================= 0.3 INFRA: estado persistente, espera do servidor e construção idempotente =======================
LIMITE_MERGE_MB = 256          # só espera se os merges em andamento usarem mais memória do que isto (merges minúsculos de logs do sistema não bloqueiam)
ESPERA_MERGES_MAX = 600        # segundos (máx.) esperando merges/mutations terminarem
ESPERA_CONSULTAS_MAX = 7200    # segundos (máx.) esperando uma consulta anterior sobre a mesma tabela terminar
CODIGOS_RECUPERAVEIS = ("Code: 241", "Code: 452", "Code: 159", "MEMORY_LIMIT_EXCEEDED", "SETTING_CONSTRAINT_VIOLATION", "TIMEOUT_EXCEEDED")
HTTP_TIMEOUT = 7200            # consultas pesadas (hash/joins de dezenas de milhões de linhas) podem passar de 30 min
_fechar_conexao()              # a próxima conexão já usa o novo timeout


def fmt_bytes(n):
    n = float(n)
    for u in ("B", "KiB", "MiB", "GiB", "TiB"):
        if n < 1024 or u == "TiB":
            return f"{n:.2f} {u}"
        n /= 1024


def _jd(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.bool_): return bool(o)
    return str(o)


def assinatura(*partes):
    """Impressão digital curta de uma configuração (lista de features, mapa de rótulos, ...)."""
    return hashlib.sha256(json.dumps(partes, sort_keys=True, default=_jd).encode("utf-8")).hexdigest()[:16]


SIG_STG = assinatura("stg", N_TREINO, N_TESTE, FEATURES_ORDEM, sorted(FEATURES_INT), COLS_STATUS, sorted(ID_FISICO))
SIG_FEATS = None


# ---------------- estado persistente (tabela prep_estado; último valor por chave) ----------------
def _garantir_estado():
    if not existe("prep_estado"):
        executar("CREATE TABLE IF NOT EXISTS prep_estado (chave String, versao UInt64, valor String) ENGINE = ReplacingMergeTree(versao) ORDER BY chave")


def estado_set(chave, valor):
    _garantir_estado()
    inserir_df("prep_estado", pd.DataFrame([{"chave": chave, "versao": int(time.time_ns()), "valor": json.dumps(valor, default=_jd)}]))


def estado_get(chave, padrao=None):
    if not existe("prep_estado"):
        return padrao
    r = consultar(f"SELECT argMax(valor, versao) AS v, toString(count()) AS n FROM prep_estado WHERE chave = {lit(chave)}")
    if len(r) == 0 or int(r["n"][0]) == 0:
        return padrao
    v = json.loads(r["v"][0])
    return padrao if v is None else v


# ---------------- memória do servidor -> configurações das consultas pesadas ----------------
def descobrir_memoria():
    """Limite efetivo de memória do servidor em GiB (menor entre RAM do SO e cgroup; respeita max_server_memory_usage). None se desconhecido."""
    total = None
    try:
        am = consultar_num("SELECT metric, value FROM system.asynchronous_metrics WHERE metric IN ('OSMemoryTotal', 'CGroupMemoryTotal')")
        vals = [float(v) / 2 ** 30 for v in am["value"] if float(v) > 0]
        total = min(vals) if vals else None
    except ErroConexao:
        raise
    except Exception:
        pass
    limite = None
    try:
        ss = consultar("SELECT name, value FROM system.server_settings WHERE name IN ('max_server_memory_usage', 'max_server_memory_usage_to_ram_ratio')")
        d = {n: float(v) for n, v in zip(ss["name"], ss["value"])}
        if d.get("max_server_memory_usage", 0) > 0:
            limite = d["max_server_memory_usage"] / 2 ** 30
        elif total:
            limite = total * (d.get("max_server_memory_usage_to_ram_ratio", 0.9) or 0.9)
    except ErroConexao:
        raise
    except Exception:
        if total:
            limite = total * 0.9
    return limite


SERVIDOR_GB = descobrir_memoria()
if SERVIDOR_GB:
    LIMITE_RSS_GB_DINAMICO = max(1.0, 0.75 * SERVIDOR_GB)
    _qb = int(0.6 * SERVIDOR_GB * 2 ** 30)
    VARIANTES_SETTINGS = [
        f"SETTINGS max_memory_usage = {_qb}, max_bytes_before_external_group_by = {_qb // 2}, max_bytes_before_external_sort = {_qb // 2}",
        f"SETTINGS max_bytes_before_external_group_by = {_qb // 3}, max_bytes_before_external_sort = {_qb // 3}, max_threads = 4",
        "SETTINGS max_threads = 2"]
    print(f"Memória efetiva do servidor: {SERVIDOR_GB:.1f} GiB | limite de espera (RSS): {LIMITE_RSS_GB_DINAMICO:.1f} GiB | memória por consulta pesada: {fmt_bytes(_qb)}")
else:
    LIMITE_RSS_GB_DINAMICO = None
    VARIANTES_SETTINGS = ["", "SETTINGS max_threads = 2"]
    print("Memória do servidor não identificada; usando configurações padrão (com repetição conservadora em caso de erro de memória).")
if SETTINGS_PESADO:                       # se você definiu SETTINGS_PESADO manualmente, ele é a primeira tentativa
    VARIANTES_SETTINGS.insert(0, SETTINGS_PESADO)


# ---------------- espera do servidor (merges, mutations, memória residente e consultas anteriores) ----------------
def aguardar_servidor(esperar_consultas=False, tabela=None, max_consultas_s=None):
    """Espera merges/mutations internos e (opcional) consultas anteriores sobre a tabela terminarem. Nunca bloqueia para sempre."""
    max_consultas_s = ESPERA_CONSULTAS_MAX if max_consultas_s is None else max_consultas_s
    limite_rss = LIMITE_RSS_GB_DINAMICO or 2.0
    t0, ultimo_aviso, avisou, desistiu = time.time(), 0.0, False, False
    while True:
        decorrido = time.time() - t0
        try:
            m = consultar_num("SELECT count() AS n, sum(memory_usage) AS mem FROM system.merges")
            mu = consultar_num("SELECT count() AS n FROM system.mutations WHERE is_done = 0 AND database = currentDatabase()")
            rss = consultar_num("SELECT toFloat64(value) AS v FROM system.asynchronous_metrics WHERE metric = 'MemoryResident'")
            mem_m, n_mut = float(m["mem"][0] or 0), int(mu["n"][0])
            rss_gb = float(rss["v"][0]) / 2 ** 30 if len(rss) else 0.0
            motivos, prazo_estourado = [], False
            if mem_m >= LIMITE_MERGE_MB * 2 ** 20:
                if decorrido < ESPERA_MERGES_MAX: motivos.append(f"com merges internos usando {fmt_bytes(mem_m)}; aguardando terminarem (máx {ESPERA_MERGES_MAX}s)...")
                else: prazo_estourado = True
            if n_mut:
                if decorrido < ESPERA_MERGES_MAX: motivos.append(f"com {n_mut} mutation(s) em andamento; aguardando")
                else: prazo_estourado = True
            if rss_gb >= limite_rss:
                if decorrido < ESPERA_MERGES_MAX: motivos.append(f"com {rss_gb:.2f} GiB residentes (limite {limite_rss:.1f}); aguardando baixar")
                else: prazo_estourado = True
            if esperar_consultas:
                padrao = lit(f"%{tabela}%") if tabela else "'%prep_%'"
                qs = consultar_num("SELECT count() AS n, max(elapsed) AS el FROM system.processes WHERE query_id != queryID() "
                                   f"AND query ILIKE {padrao} AND query NOT ILIKE '%system.%'")
                if int(qs["n"][0]) > 0:
                    if decorrido < max_consultas_s: motivos.append(f"com {int(qs['n'][0])} consulta(s) anterior(es) ainda em execução sobre {tabela or 'tabelas prep_*'} (há {float(qs['el'][0]):.0f}s); aguardando (máx {max_consultas_s}s)")
                    else: prazo_estourado = True
        except ErroConexao:
            raise
        except Exception as e:
            print(f"  (não foi possível checar o estado do servidor: {str(e)[:150]}); seguindo.")
            return
        if prazo_estourado and not desistiu:
            print("  ⚠ tempo máximo de espera atingido com atividade ainda em curso no servidor; seguindo mesmo assim."); desistiu = True
        if not motivos:
            if avisou: print("  ✔ servidor livre; seguindo.")
            return
        if time.time() - ultimo_aviso >= 30:
            print("  ClickHouse " + " | ".join(motivos)); ultimo_aviso, avisou = time.time(), True
        time.sleep(5)


def rodar_pesado(sql_modelo):
    """SELECT pesado. Use @@S@@ onde iria o SETTINGS. Em erro de memória/limite repete com a próxima configuração."""
    # v8.4.2: o servidor devolve erro de memória (Code 241) como HTTP 500 com corpo JSON, que o transporte trata como ErroConexao.
    # Antes isso era repassado direto (sem tentar as variantes leves nem esperar a memória baixar). Agora: espera a memória baixar,
    # tenta cada variante de settings e, se todas falharem, espera de novo e faz mais um ciclo completo.
    ultimo = None
    for ciclo in range(2):
        for i, S in enumerate(VARIANTES_SETTINGS):
            try:
                aguardar_servidor(); _esperar_memoria_baixar()
                return consultar_num(sql_modelo.replace("@@S@@", S))
            except PermissionError:
                raise
            except Exception as e:
                ultimo = e
                recuperavel = isinstance(e, ErroConexao) or any(k in str(e) for k in CODIGOS_RECUPERAVEIS)
                if not recuperavel:
                    raise
                if isinstance(e, ErroFatal) or (hasattr(e, "args") and "NOT_ENOUGH_SPACE" in str(e)):
                    raise
                print(f"  ⚠ consulta pesada falhou (ciclo {ciclo + 1}, variante {i + 1}/{len(VARIANTES_SETTINGS)}): {str(e)[:170]}...\n    esperando a memória do servidor baixar e repetindo com outra configuração.")
        if ciclo == 0:
            time.sleep(60)
    raise ultimo


# ---------------- construção idempotente de tabelas ----------------
def _ok(validador):
    try:
        return bool(validador())
    except ErroConexao:
        raise
    except Exception as e:
        print(f"    (validação não pôde ser concluída: {str(e)[:150]})"); return False


def construir(tabela, acoes, assin, validador):
    """Cria `tabela` só se necessário.
    acoes(S): executa DDL/INSERT (S = SETTINGS da tentativa). assin: assinatura da configuração. validador(): True se a tabela está completa.
    Retorna True se construiu, False se reaproveitou."""
    forcar = tabela in RECRIAR
    if existe(tabela) and not forcar:
        marca = estado_get(f"tabela:{tabela}")
        if marca == assin:
            if _ok(validador):
                print(f"  = `{tabela}` já está pronta e válida; não será refeita."); return False
            print(f"  ! `{tabela}` constava como pronta, mas falhou na validação; será refeita.")
        elif marca is None:
            aguardar_servidor(esperar_consultas=True, tabela=tabela)       # pode haver INSERT de uma execução anterior ainda rodando
            if _ok(validador):
                estado_set(f"tabela:{tabela}", assin)
                print(f"  = `{tabela}` já existia (sem marca), passou na validação e foi adotada sem refazer. "
                      f"Se você mudou a seleção de features ou o mapa de rótulos desde a última execução, use RECRIAR.add('{tabela}')."); return False
            print(f"  ! `{tabela}` existe mas está incompleta/inconsistente; será refeita.")
        else:
            print(f"  ! `{tabela}` foi construída com outra configuração (features/rótulos mudaram); será refeita.")
    elif forcar and existe(tabela):
        print(f"  ↻ `{tabela}` está em RECRIAR; será refeita.")
    ultimo = None
    for i, S in enumerate(VARIANTES_SETTINGS):
        aguardar_servidor()
        executar(f"DROP TABLE IF EXISTS `{tabela}`")
        estado_set(f"tabela:{tabela}", None)
        t0 = time.time()
        try:
            acoes(S)
        except ErroConexao as e:
            print(f"  ⚠ a conexão caiu durante `{tabela}` ({str(e)[:150]}). O servidor pode continuar executando; aguardando terminar...")
            aguardar_servidor(esperar_consultas=True, tabela=tabela)
            if existe(tabela) and _ok(validador):
                break
            ultimo = e; continue
        except PermissionError:
            raise
        except Exception as e:
            ultimo = e
            if i + 1 < len(VARIANTES_SETTINGS) and any(k in str(e) for k in CODIGOS_RECUPERAVEIS):
                print(f"  ⚠ `{tabela}`: {str(e)[:160]}...\n    repetindo com outra configuração de memória (tentativa {i + 2}/{len(VARIANTES_SETTINGS)}).")
                continue
            raise
        if _ok(validador):
            break
        ultimo = RuntimeError(f"`{tabela}` criada, mas a validação de completude falhou")
    else:
        raise RuntimeError(f"Não foi possível construir `{tabela}` com nenhuma configuração: {ultimo!r}") from ultimo
    estado_set(f"tabela:{tabela}", assin)
    RECRIAR.discard(tabela)
    print(f"  ✔ `{tabela}` construída e validada: {n_linhas(tabela):,} linhas em {time.time() - t0:,.0f}s")
    return True


def etapa(nome, assin, calcular):
    """Roda calcular() só se não houver resultado salvo com a mesma assinatura. Devolve (resultado, reaproveitado)."""
    if nome not in RECRIAR:
        s = estado_get(f"etapa:{nome}")
        if isinstance(s, dict) and s.get("sig") == assin:
            print(f"  = etapa '{nome}' já concluída em {s.get('em', '?')}; resultado reaproveitado (RECRIAR.add('{nome}') para refazer).")
            return s["res"], True
    res = calcular()
    estado_set(f"etapa:{nome}", {"sig": assin, "res": res, "em": datetime.now().isoformat(timespec="seconds")})
    RECRIAR.discard(nome)
    return res, False


def _soma(tab, col):
    return int(consultar_num(f"SELECT sum({col}) AS s FROM `{tab}`")["s"][0])


def _colunas_de(tab):
    return set(consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(tab)}")["name"])


def garantir_mapa():
    global MAPA
    if "MAPA" not in globals():
        m = consultar("SELECT label_original, label_multi FROM prep_mapa_rotulos")
        MAPA = {a: b for a, b in zip(m.label_original, m.label_multi) if a != b}
    return MAPA


# ---------------- seleção de features: salvar / restaurar (estado + tabela de auditoria) ----------------
def _atualizar_sig():
    global SIG_FEATS
    SIG_FEATS = assinatura(SIG_STG, FEATURES_BASE, COLS_NULL_BASE)


def salvar_selecao(reduzido=None, motivos_extra=None):
    reduzido = list(FEATURES_BASE if reduzido is None else reduzido); motivos_extra = motivos_extra or {}
    linhas = []
    for c in FEATURES_ORDEM:
        if c in CONSTANTES: sit, mot = "removida_constante", "valor único em todo o treino (confirmado em todas as tabelas de treino)"
        elif c in DUPLICADAS: sit, mot = "removida_duplicada", f"idêntica a {DUPLICADAS[c]} no treino (teste usado apenas para auditoria)"
        elif c in REMOVIDAS_AUSENTES: sit, mot = "removida_ausentes", f"ausentes > {LIMITE_AUSENTES_REMOVER:.0%} no treino"
        elif c in reduzido: sit, mot = "mantida", ""
        else: sit, mot = "base_nao_reduzido", motivos_extra.get(c, "redundante por correlação no treino")
        linhas.append({"coluna": c, "situacao": sit, "motivo": mot, "em_base": int(c in FEATURES_BASE), "em_reduzido": int(c in reduzido),
                       "tem_nulos": int(c in COLS_NULL), "tipo": "Int64" if c in FEATURES_INT else "Float64"})
    estado_set("selecao", {"features_base": FEATURES_BASE, "features_reduzido": reduzido, "cols_null": COLS_NULL, "constantes": CONSTANTES,
                           "duplicadas": DUPLICADAS, "removidas_ausentes": REMOVIDAS_AUSENTES, "motivos_extra": motivos_extra})
    executar("DROP TABLE IF EXISTS `prep_selecao_features`")
    executar("CREATE TABLE prep_selecao_features (coluna String, situacao String, motivo String, em_base UInt8, em_reduzido UInt8, tem_nulos UInt8, tipo String) ENGINE = MergeTree ORDER BY coluna")
    inserir_df("prep_selecao_features", pd.DataFrame(linhas))
    _atualizar_sig()


def restaurar_selecao():
    """Restaura TODAS as variáveis da seleção (inclusive CONSTANTES, DUPLICADAS, REMOVIDAS_AUSENTES) a partir do estado salvo."""
    global FEATURES_BASE, FEATURES_REDUZIDO, COLS_NULL, COLS_NULL_BASE, CONSTANTES, DUPLICADAS, REMOVIDAS_AUSENTES
    est = estado_get("selecao")
    if isinstance(est, dict):
        FEATURES_BASE, FEATURES_REDUZIDO, COLS_NULL = est["features_base"], est["features_reduzido"], est["cols_null"]
        CONSTANTES, DUPLICADAS, REMOVIDAS_AUSENTES = est["constantes"], est["duplicadas"], est["removidas_ausentes"]
    elif existe("prep_selecao_features"):          # tabela de uma execução antiga (sem estado)
        s = consultar_num("SELECT * FROM prep_selecao_features")
        ordem = {c: i for i, c in enumerate(FEATURES_ORDEM)}; s = s.sort_values("coluna", key=lambda x: x.map(ordem))
        FEATURES_BASE = s.loc[s.em_base == 1, "coluna"].tolist(); FEATURES_REDUZIDO = s.loc[s.em_reduzido == 1, "coluna"].tolist()
        COLS_NULL = s.loc[s.tem_nulos == 1, "coluna"].tolist()
        CONSTANTES = s.loc[s.situacao == "removida_constante", "coluna"].tolist()
        DUPLICADAS = {}
        for r in s[s.situacao == "removida_duplicada"].itertuples():
            mt = re.search(r"id[eê]ntica a (\S+) no treino", str(r.motivo)); DUPLICADAS[r.coluna] = mt.group(1) if mt else ""
        REMOVIDAS_AUSENTES = s.loc[s.situacao == "removida_ausentes", "coluna"].tolist()
    else:
        raise RuntimeError("Seleção de features ainda não definida: execute os blocos 2 e 3 (células de ausentes, constantes e duplicadas) antes.")
    COLS_NULL_BASE = [c for c in COLS_NULL if c in FEATURES_BASE]
    _atualizar_sig()
    return est


def painel_status():
    t = consultar_num("SELECT name AS tabela, engine AS motor, total_rows AS linhas, total_bytes AS bytes FROM system.tables "
                      "WHERE database = currentDatabase() AND startsWith(name, 'prep_') ORDER BY name")
    if len(t) == 0:
        print("Nenhuma tabela prep_* ainda."); return t
    t["tamanho"] = t["bytes"].map(lambda b: fmt_bytes(b) if pd.notna(b) else "")
    t["marca_pronta"] = ["view" if m == "View" else ("sim" if estado_get(f"tabela:{n}") is not None else "não") for n, m in zip(t.tabela, t.motor)]
    return t[["tabela", "motor", "linhas", "tamanho", "marca_pronta"]]



# ======================= v8: UNIDADES ATÔMICAS, EXIGÊNCIAS DE ETAPA E FAIXAS =======================
ESPERAS_PARTE = (15, 45, 90, 180, 180, 180)     # espera (s) entre tentativas de uma unidade
MAX_TENT_UNID = 5                               # tentativas por unidade antes de parar com erro claro
CODIGOS_DETERMINISTICOS = ("Code: 62", "Code: 47", "Code: 60", "Code: 36", "Code: 53", "Code: 115", "Code: 164", "Code: 497",
                           "Code: 516", "Code: 43", "Code: 44", "Code: 46", "Code: 386")


class ErroFatal(RuntimeError):
    """Erro que NÃO deve ser repetido (ex.: disco cheio que não pôde ser aliviado)."""


ESPERA_ESPACO_MAX = 900     # segundos esperando a remoção de partes antigas antes de parar por disco cheio
_ACOES_LOGS = []


def diagnostico_disco(top=15):
    """SOMENTE LEITURA. Espaço livre no disco do servidor e quais tabelas (inclusive logs internos do ClickHouse) ocupam mais."""
    for titulo, q_ in [
        ("Discos do servidor", "SELECT name, formatReadableSize(free_space) AS livre, formatReadableSize(total_space) AS total, round(free_space / total_space * 100, 1) AS pct_livre FROM system.disks"),
        (f"Maiores tabelas (partes ativas, top {top})", f"SELECT database, `table`, formatReadableSize(sum(bytes_on_disk)) AS tamanho, sum(rows) AS linhas, count() AS partes FROM system.parts WHERE active GROUP BY database, `table` ORDER BY sum(bytes_on_disk) DESC LIMIT {top}"),
        ("Partes inativas (já substituídas por merges; o servidor apaga sozinho em alguns minutos)", "SELECT formatReadableSize(sum(bytes_on_disk)) AS tamanho, count() AS partes FROM system.parts WHERE active = 0"),
    ]:
        try:
            r = consultar(q_); print(f"\n== {titulo} ==\n" + (r.to_string(index=False) if len(r) else "(nenhum)"))
        except Exception as e:
            print(f"\n== {titulo} == indisponível: {str(e).splitlines()[0][:150]}")


def _livre_gib():
    return float(consultar_num("SELECT min(free_space) AS v FROM system.disks")["v"][0]) / 2 ** 30


def _espaco_minimo_gib():
    if ESPACO_MIN_GIB is not None:
        return float(ESPACO_MIN_GIB)
    try:
        total = float(consultar_num("SELECT min(total_space) AS v FROM system.disks")["v"][0]) / 2 ** 30
    except Exception:
        return 4.0
    return max(0.75, min(4.0, 0.08 * total))          # disco de 18,6 GiB -> ~1,5 GiB


def _liberar_espaco_auto(motivo=""):
    """SALVAGUARDA (só quando o disco enche): esvazia, do MAIOR para o MENOR e só até sobrar espaço, logs INTERNOS do ClickHouse (system.*).
    Não toca em nenhuma tabela data_*, prep_* nem em outro banco; só nomes de LOGS_AUTOMATICOS; precisa de permissão (se negada, nada é feito).
    Os logs voltam a crescer sozinhos; só se perde o histórico de profiling/métricas/mensagens do servidor."""
    if not LIMPAR_LOGS_SE_DISCO_CHEIO or len(_ACOES_LOGS) >= 3:
        return False
    try:
        lista = ", ".join(lit(t) for t in LOGS_AUTOMATICOS)
        tam = consultar_num(f"SELECT `table` AS tabela, sum(bytes_on_disk) AS b FROM system.parts WHERE database = {lit(LOGS_BANCO)} AND active "
                            f"AND `table` IN ({lista}) GROUP BY `table` ORDER BY b DESC")
        antes = _livre_gib()
    except Exception as e:
        print(f"  (não foi possível medir os logs internos: {str(e)[:150]}); nada será apagado."); return False
    total_mib = float(tam["b"].sum()) / 2 ** 20 if len(tam) else 0.0
    if total_mib < LOGS_MIN_MIB:
        print(f"  Os logs internos somam só {total_mib:,.0f} MiB (< {LOGS_MIN_MIB}); esvaziá-los não resolveria o disco cheio."); return False
    alvo = max(3 * _espaco_minimo_gib(), 3.0)
    print(f"  ⚠ {motivo}. Esvaziando LOGS INTERNOS do ClickHouse ({LOGS_BANCO}.*, não são dados do TCC), do maior para o menor, até ~{alvo:.1f} GiB livres "
          f"(livre agora: {antes:.2f} GiB; logs somam {total_mib:,.0f} MiB)...")
    feitos = 0
    for t, b in zip(tam["tabela"], tam["b"]):
        try:
            _http(f"TRUNCATE TABLE IF EXISTS {LOGS_BANCO}.{t}"); feitos += 1
            print(f"    ✔ {LOGS_BANCO}.{t} esvaziada ({float(b) / 2 ** 20:,.0f} MiB)")
        except Exception as e:
            print(f"    ✖ {LOGS_BANCO}.{t}: {str(e)[:220]}")
            if "ACCESS_DENIED" in str(e) or "rivileges" in str(e):
                print("    (sem permissão para esvaziar logs internos; interrompendo a limpeza)"); break
        try:
            if _livre_gib() >= alvo:
                break
        except Exception:
            pass
    _ACOES_LOGS.append(motivo)
    try:
        depois = _livre_gib()
    except Exception:
        depois = antes
    print(f"    espaço livre: {antes:.2f} -> {depois:.2f} GiB")
    return feitos > 0 and (depois - antes) * 1024 >= LOGS_GANHO_MIN_MIB


def garantir_espaco(minimo_gib=None, espera_max_s=None):
    """Antes de gravar: confere o espaço livre. Se estiver abaixo do mínimo, tenta liberar logs internos (salvaguarda), espera a remoção de partes antigas
    e, se não melhorar, PARA com instruções (sem repetir às cegas)."""
    minimo = _espaco_minimo_gib() if minimo_gib is None else minimo_gib
    espera_max_s = ESPERA_ESPACO_MAX if espera_max_s is None else espera_max_s
    try:
        livre = _livre_gib()
    except Exception:
        return
    if livre >= minimo:
        return
    _liberar_espaco_auto(f"pouco espaço em disco ({livre:.2f} GiB livres; mínimo {minimo:.1f})")
    t0 = time.time()
    while True:
        try:
            livre = _livre_gib()
        except Exception:
            return
        if livre >= minimo:
            print(f"  ✔ espaço em disco OK: {livre:.2f} GiB livres."); return
        if time.time() - t0 >= espera_max_s:
            break
        print(f"  ⏳ {livre:.2f} GiB livres (mínimo {minimo:.1f}); aguardando a remoção de partes antigas...")
        time.sleep(30)
    diagnostico_disco()
    raise ErroFatal(f"DISCO DO SERVIDOR CHEIO: só {livre:.2f} GiB livres (mínimo {minimo:.1f}). Repetir não resolve. Libere espaço no servidor "
                    f"(os logs internos do ClickHouse não puderam ser esvaziados ou não bastaram; veja o diagnóstico acima) ou aumente o volume; depois rode a célula de novo "
                    f"(o progresso está salvo).")


def _eh_disco_cheio(e):
    return "NOT_ENOUGH_SPACE" in str(e) or "Code: 243" in str(e)


def deterministico(e) -> bool:
    """True = repetir não adianta (sintaxe/coluna/tabela/tipo/permissão ou DISCO CHEIO que não pôde ser aliviado). False = vale repetir."""
    if _eh_disco_cheio(e):
        if _liberar_espaco_auto("o servidor devolveu Code 243 (disco cheio)"):
            return False                                   # espaço liberado: o chamador repete a unidade
        print("\n  ✖ DISCO DO SERVIDOR CHEIO (Code 243). Repetir não resolve. Diagnóstico:"); diagnostico_disco()
        print("\n  → Libere espaço no servidor (logs internos do ClickHouse) ou aumente o volume e rode a célula de novo; o progresso está salvo.")
        return True
    return any(k in str(e) for k in CODIGOS_DETERMINISTICOS)


def exigir(*tabelas):
    """TRAVA DE ETAPA: só deixa a célula seguir se cada tabela existe E foi CONCLUÍDA (marca 'tabela:<nome>' no estado).
    Evita rodar uma etapa sobre dados incompletos se você executar as células fora de ordem ou depois de um erro."""
    faltam = []
    for t in tabelas:
        if not existe(t):
            faltam.append(f"`{t}` (não existe)")
        elif estado_get(f"tabela:{t}") is None:
            faltam.append(f"`{t}` (etapa não concluída)")
    if faltam:
        raise RuntimeError("Etapa anterior NÃO concluída: " + "; ".join(faltam) + ". Rode as células anteriores, em ordem, "
                           "até cada etapa terminar com ✔, e só então execute esta.")


def faixas_id(n, por_unidade, minimo=1):
    """Divide o espaço de id_linha (hash uniforme de 64 bits) em faixas com ~`por_unidade` linhas: [(lo, hi|None)]."""
    B = max(1, int(minimo), -(-int(n) // int(por_unidade)))
    return [(j * 2 ** 64 // B, ((j + 1) * 2 ** 64 // B) if j < B - 1 else None) for j in range(B)]


def cond_id(col, lo, hi):
    return f"{col} >= {lo}" + (f" AND {col} < {hi}" if hi is not None else "")


def faixas_hash(K):
    """K faixas pelo 1º byte de h_feat (FixedString(16)). Devolve [(condição SQL, índice)]; a condição poda a chave primária."""
    K = max(1, min(int(K), 256)); cortes = [j * 256 // K for j in range(K + 1)]; out = []
    for j in range(K):
        lo, hi = cortes[j], cortes[j + 1]; c = []
        if lo > 0: c.append(f"h_feat >= unhex('{lo:02x}{'00' * 15}')")
        if hi < 256: c.append(f"h_feat < unhex('{hi:02x}{'00' * 15}')")
        out.append((" AND ".join(c) or "1", j))
    return out


def _esperar_memoria_baixar(maximo_s=None):
    """Antes de cada unidade pesada: espera RSS e merges internos baixarem (até ESPERA_MEM_UNIDADE_MAX), mostrando o progresso dos merges."""
    maximo_s = ESPERA_MEM_UNIDADE_MAX if maximo_s is None else maximo_s
    limite = LIMITE_RSS_GB_DINAMICO or 2.0
    t0, ult = time.time(), 0.0
    while True:
        try:
            rss = float(consultar_num("SELECT toFloat64(value) AS v FROM system.asynchronous_metrics WHERE metric = 'MemoryResident'")["v"][0]) / 2 ** 30
            mm = consultar_num("SELECT count() AS n, sum(memory_usage) AS mem FROM system.merges"); mem_m = float(mm["mem"][0] or 0) / 2 ** 20
        except Exception:
            return
        if rss < limite and mem_m < LIMITE_MERGE_MB:
            return
        if time.time() - t0 > maximo_s:
            print(f"  ⚠ a memória do servidor continua alta após {int(maximo_s // 60)} min (RSS {rss:.2f} GiB, merges {mem_m:,.0f} MiB); tentando mesmo assim."); return
        if time.time() - ult > 120:
            ult = time.time()
            try:
                det = "; ".join(f"{t} {float(p) * 100:.0f}% ({m})" for t, p, m in consultar("SELECT table, progress, formatReadableSize(memory_usage) AS m FROM system.merges ORDER BY memory_usage DESC LIMIT 3").itertuples(index=False))
            except Exception:
                det = "?"
            print(f"  ⏳ aguardando a memória do servidor baixar: RSS {rss:.2f} GiB (limite {limite:.1f}), merges {mem_m:,.0f} MiB [{det}] — {int((time.time() - t0) // 60)} min de {int(maximo_s // 60)}.")
        time.sleep(15)


def _marcadores(tabela):
    pre = f"unidade:{tabela}:"
    r = consultar(f"SELECT chave, argMax(valor, versao) AS v FROM prep_estado WHERE startsWith(chave, {lit(pre)}) GROUP BY chave")
    return {c[len(pre):]: json.loads(v) for c, v in zip(r.chave, r.v)} if len(r) else {}


def _apagar_marcadores(tabela):
    executar(f"ALTER TABLE prep_estado DELETE WHERE startsWith(chave, {lit(f'unidade:{tabela}:')})")


def _executar_unidade(tabela, u, assin):
    """Uma unidade = UMA partição da tabela: apaga a partição, insere, CONFERE a contagem e só então marca como feita.
    Se a conexão cair, espera o servidor terminar a consulta e confere se ela chegou a gravar tudo (não duplica nem perde)."""
    ultimo = None
    for tent in range(1, MAX_TENT_UNID + 1):
        _vars = u.get("variantes") or VARIANTES_SETTINGS
        S = _vars[min(tent - 1, len(_vars) - 1)]
        erro = None
        try:
            aguardar_servidor(); _esperar_memoria_baixar(); garantir_espaco()
            executar(f"ALTER TABLE `{tabela}` DROP PARTITION {u['particao']}")
            executar(u["sql"].replace("@@S@@", S))
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroFatal):
                raise
            erro = e
            if deterministico(e):
                raise RuntimeError(f"`{tabela}` unidade {u['chave']}: erro que não se resolve repetindo: {str(e)[:600]}") from e
            if isinstance(e, ErroConexao):
                print(f"  ⚠ conexão caiu em {tabela}/{u['chave']}; aguardando o servidor terminar a consulta antes de conferir...")
                aguardar_servidor(esperar_consultas=True, tabela=tabela)
        esperado = u.get("esperado")
        if esperado is None:
            if erro is None:
                estado_set(f"unidade:{tabela}:{u['chave']}", assin); return
        else:
            tem = u["contar"]()
            if tem == esperado:
                estado_set(f"unidade:{tabela}:{u['chave']}", assin); return
            if erro is None:
                erro = RuntimeError(f"contagem divergente: {tem:,} linhas gravadas, esperado {esperado:,}")
        ultimo = erro
        espera = ESPERAS_PARTE[min(tent - 1, len(ESPERAS_PARTE) - 1)]
        print(f"  ⚠ {tabela}/{u['chave']} tentativa {tent}/{MAX_TENT_UNID}: {str(ultimo)[:300]}\n    aguardando {espera}s e refazendo só esta unidade...")
        time.sleep(espera)
    raise RuntimeError(f"`{tabela}` unidade {u['chave']} falhou após {MAX_TENT_UNID} tentativas. Último erro: {ultimo!r}. "
                       f"O que já foi feito está salvo: rode a célula de novo para retomar.")


def construir_por_unidades(tabela, ddl, unidades, assin, validador):
    """Constrói uma tabela PARTICIONADA em unidades curtas e retomáveis. unidades: [{'chave','particao','sql'('@@S@@' = SETTINGS),
    'esperado'(int|None),'contar'(callable)}]. Se já está pronta e válida, não faz nada; se foi interrompida, continua da próxima unidade."""
    forcar = tabela in RECRIAR
    if existe(tabela) and not forcar:
        marca = estado_get(f"tabela:{tabela}")
        if marca == assin:
            if _ok(validador):
                print(f"  = `{tabela}` já está pronta e válida; não será refeita."); return False
            print(f"  ! `{tabela}` constava como pronta, mas falhou na validação; será completada.")
        elif marca is None:
            aguardar_servidor(esperar_consultas=True, tabela=tabela)
            if _ok(validador):
                estado_set(f"tabela:{tabela}", assin)
                print(f"  = `{tabela}` já existia (sem marca), passou na validação e foi adotada sem refazer."); return False
    retomar = existe(tabela) and not forcar and estado_get(f"progresso:{tabela}") == assin
    if not retomar:
        aguardar_servidor(esperar_consultas=True, tabela=tabela)
        executar(f"DROP TABLE IF EXISTS `{tabela}`")
        estado_set(f"tabela:{tabela}", None); _apagar_marcadores(tabela)
        executar(ddl); estado_set(f"progresso:{tabela}", assin)
        print(f"  ▶ construindo `{tabela}` em {len(unidades)} unidade(s).")
    else:
        print(f"  ↻ retomando `{tabela}` de onde parou.")
    feitas = _marcadores(tabela); n = len(unidades); t0 = time.time(); ja = sum(1 for u in unidades if feitas.get(u["chave"]) == assin)
    if ja: print(f"    {ja}/{n} unidades já concluídas.")
    for idx, u in enumerate(unidades, 1):
        if feitas.get(u["chave"]) == assin:
            continue
        _executar_unidade(tabela, u, assin)
        if idx % 5 == 0 or idx == n:
            print(f"    … {tabela}: {idx}/{n} unidades ({time.time() - t0:,.0f}s)")
    if not _ok(validador):
        raise RuntimeError(f"`{tabela}`: todas as unidades foram gravadas, mas a validação de completude falhou. "
                           f"Use RECRIAR.add('{tabela}') e rode de novo.")
    estado_set(f"tabela:{tabela}", assin); estado_set(f"progresso:{tabela}", None); RECRIAR.discard(tabela)
    print(f"  ✔ `{tabela}` construída e validada: {n_linhas(tabela):,} linhas em {time.time() - t0:,.0f}s")
    return True


print("Infra carregada: construir() / construir_por_unidades() / exigir() / etapa() / aguardar_servidor() / rodar_pesado().")


Memória efetiva do servidor: 3.0 GiB | limite de espera (RSS): 2.2 GiB | memória por consulta pesada: 1.80 GiB
Infra carregada: construir() / construir_por_unidades() / exigir() / etapa() / aguardar_servidor() / rodar_pesado().


In [7]:
display(painel_status())


,tabela,motor,linhas,tamanho,marca_pronta
0,prep_aud_invalidos,MergeTree,144.0,14.46 KiB,sim
1,prep_aux_ids,MergeTree,70427637.0,3.32 GiB,sim
2,prep_classes_resumo,MergeTree,18.0,1.14 KiB,sim
3,prep_estado,ReplacingMergeTree,534.0,28.74 KiB,não
4,prep_final_teste,MergeTree,20364525.0,1.28 GiB,sim
5,prep_final_treino,MergeTree,8257343.0,800.20 MiB,sim
6,prep_grp_treino,MergeTree,8041864.0,149.28 MiB,sim
7,prep_hash,MergeTree,70427637.0,783.54 MiB,sim
8,prep_keep_treino,MergeTree,8257343.0,200.91 MiB,sim
9,prep_mapa_rotulos,MergeTree,19.0,864.00 B,sim


## 1. Camada de preparação (`prep_stg` e `prep_aux_ids`)

- Tipos **explícitos**: features inteiras → `Nullable(Int64)`, decimais → `Nullable(Float64)` (não usamos `CREATE TABLE ... AS` das originais, que herdaria `String`).
- Conversão: texto vazio, texto não numérico, `NaN`, `+Inf` e `-Inf` viram `NULL` **e** o motivo fica registrado em `<coluna>_status` (`ok`, `vazio`, `pos_inf`, `neg_inf`, `nan`, `nao_numerico`) para `flow_bytes_s`, `flow_packets_s` e `simillarhttp` (únicas com inválidos na EDA; as outras 77 são verificadas abaixo).
- Origem: `conjunto` (treino/teste) e `origem_tabela`. `id_linha` = hash determinístico da origem + identificadores brutos, usado para rastrear cada registro.
- `label_original` preservado; `label_binary` = 0 para `BENIGN`, 1 para ataque. O rótulo multiclasse padronizado vem no bloco 5 (depois de confirmar UDP-lag × UDPLag).
- Identificadores (`unnamed_0`, `flow_id`, IPs, portas, `timestamp`) e o texto bruto de `simillarhttp` ficam **só** em `prep_aux_ids`, fora da matriz de features.

**v8 — como a carga funciona:** a tabela de origem é dividida em *chunks* (por `_part` e `_part_offset`); cada chunk é um INSERT de bloco único. A contagem de linhas gravadas por origem diz exatamente onde parou: ao reexecutar, a carga continua do chunk seguinte (se a contagem não coincidir com nenhum limite de chunk, a origem é descartada e refeita).

In [8]:
exigir_nada = None   # (a carga é a 1ª etapa: não depende de outra tabela prep_*)


def expr_status(c):
    r, p = q("r_" + c), q("p_" + c)
    return (f"multiIf({r} = '', 'vazio', lowerUTF8({r}) IN ('inf','+inf','infinity','+infinity'), 'pos_inf', "
            f"lowerUTF8({r}) IN ('-inf','-infinity'), 'neg_inf', lowerUTF8({r}) = 'nan', 'nan', isNull({p}), 'nao_numerico', "
            f"ifNull(isInfinite({p}), 0) = 1, if({p} > 0, 'pos_inf', 'neg_inf'), ifNull(isNaN({p}), 0) = 1, 'nan', 'ok')")


def sql_id_linha(t):
    # Origens em ID_FISICO têm linhas 100% idênticas (mesmo unnamed_0, flow_id, IPs, portas e timestamp): o id precisa incluir a posição física
    # (_part, _part_offset) para ser ÚNICO por linha. As demais mantêm a fórmula original (ids já carregados continuam válidos).
    fisico = ", _part, _part_offset" if t in ID_FISICO else ""
    return f"cityHash64({lit(t)}, " + ", ".join(f"toString({q(c)})" for c in ID_COLS) + fisico + ")"


# ---------------- layout das tabelas de carga ----------------
# modo "offset" (padrão): PARTITION BY origem_tabela; cada INSERT grava um BLOCO ÚNICO de ≤ CHUNK_LINHAS linhas (atômico).
# modo "hash" (plano B, mais lento): PARTITION BY (origem, id_linha % N_PARTES), como no v6/v7.
N_PARTES = 64                          # só usado no modo "hash"
SETTINGS_CARGA_HASH = ("SETTINGS max_threads = 1, max_insert_threads = 1, max_block_size = 4096, "
                       "min_insert_block_size_rows = 16384, min_insert_block_size_bytes = 8000000")
MODO_ATUAL = None


def settings_carga(chunk):
    # blocos de inserção MAIORES que o chunk => o INSERT vira UM bloco só (tudo ou nada). Memória ~ chunk x largura da linha.
    return (f"SETTINGS max_threads = 2, max_insert_threads = 1, max_block_size = 8192, "
            f"min_insert_block_size_rows = {chunk * 4}, min_insert_block_size_bytes = 3000000000, max_insert_block_size = {chunk * 4}")


def _particao(modo):
    return "origem_tabela" if modo == "offset" else f"(toString(origem_tabela), toUInt8(id_linha % {N_PARTES}))"


def ddl_stg(modo=None):
    cols = ["id_linha UInt64", "conjunto LowCardinality(String)", "origem_tabela LowCardinality(String)",
            "label_original LowCardinality(String)", "label_binary UInt8"]
    cols += [f"{q(c)} {tipo_ch(c)}" for c in FEATURES_ORDEM]
    cols += [f"{q(c + '_status')} LowCardinality(String)" for c in COLS_STATUS] + ["n_ausentes UInt8"]
    return ("CREATE TABLE prep_stg (" + ", ".join(cols) + f") ENGINE = MergeTree PARTITION BY {_particao(modo or MODO_ATUAL)} "
            "ORDER BY (conjunto, origem_tabela, id_linha)")


def ddl_aux(modo=None):
    return ("CREATE TABLE prep_aux_ids (id_linha UInt64, conjunto LowCardinality(String), origem_tabela LowCardinality(String), "
            "label_original LowCardinality(String), unnamed_0 String, flow_id String, source_ip String, source_port String, "
            "destination_ip String, destination_port String, timestamp_txt String, timestamp_dt Nullable(DateTime64(6)), "
            f"simillarhttp_txt String) ENGINE = MergeTree PARTITION BY {_particao(modo or MODO_ATUAL)} ORDER BY (origem_tabela, id_linha)")


def sql_insert_stg(t, onde=None, settings=""):
    interno = [f"{sql_id_linha(t)} AS id_linha", "trimBoth(toString(`label`)) AS label_original"]
    interno += [f"toFloat64OrNull(trimBoth(toString({q(c)}))) AS {q('p_' + c)}" for c in FEATURES_ORDEM]
    interno += [f"ifNull(trimBoth(toString({q(c)})), '') AS {q('r_' + c)}" for c in COLS_STATUS]
    ext, nomes = [], ["id_linha", "conjunto", "origem_tabela", "label_original", "label_binary"]
    ext += ["id_linha", f"{lit(conjunto_de(t))} AS conjunto", f"{lit(t)} AS origem_tabela", "label_original",
            "toUInt8(label_original != 'BENIGN') AS label_binary"]
    for c in FEATURES_ORDEM:
        p = q("p_" + c)
        ext.append(f"accurateCastOrNull({p}, 'Int64') AS {q(c)}" if c in FEATURES_INT else f"if(isFinite({p}), {p}, NULL) AS {q(c)}")
        nomes.append(q(c))
    for c in COLS_STATUS:
        ext.append(expr_status(c)); nomes.append(q(c + "_status"))
    ext.append("toUInt8(" + " + ".join(f"isNull({q(c)})" for c in FEATURES_ORDEM) + ") AS n_ausentes"); nomes.append("n_ausentes")
    filtro = f" WHERE {onde}" if onde else ""
    return (f"INSERT INTO prep_stg ({', '.join(nomes)}) SELECT {', '.join(ext)} FROM (SELECT {', '.join(interno)} FROM `{t}`{filtro}) {settings}")


def sql_insert_aux(t, onde=None, settings=""):
    filtro = f" WHERE {onde}" if onde else ""
    return (f"INSERT INTO prep_aux_ids SELECT {sql_id_linha(t)}, {lit(conjunto_de(t))}, {lit(t)}, trimBoth(toString(`label`)), "
            + ", ".join(f"toString({q(c)})" for c in ["unnamed_0", "flow_id", "source_ip", "source_port", "destination_ip", "destination_port", "timestamp"])
            + ", parseDateTime64BestEffortOrNull(toString(`timestamp`), 6), toString(`simillarhttp`) "
            + f"FROM `{t}`{filtro} {settings}")


# ---------------- plano de chunks (por _part / _part_offset) ----------------
def _partes_ativas(t):
    p = consultar_num(f"SELECT name AS parte, rows AS linhas FROM system.parts WHERE database = currentDatabase() AND `table` = {lit(t)} "
                      f"AND active ORDER BY name")
    return [(str(a), int(b)) for a, b in zip(p["parte"], p["linhas"])]


def _plano_chunks(t, chunk):
    """Divide a tabela de origem em unidades de ≤ chunk linhas: [[(parte, offset_ini, offset_fim), ...], ...]."""
    partes = _partes_ativas(t)
    if sum(l for _, l in partes) != cont[t]:
        raise RuntimeError(f"{t}: as partes ativas somam {sum(l for _, l in partes):,} linhas, mas a tabela tem {cont[t]:,}. "
                           f"A tabela de origem mudou durante a execução?")
    unidades, atual, n_atual = [], [], 0
    for nome, linhas in partes:
        ini = 0
        while ini < linhas:
            fim = min(linhas, ini + chunk); tam = fim - ini
            if atual and n_atual + tam > chunk:
                unidades.append(atual); atual, n_atual = [], 0
            atual.append((nome, ini, fim)); n_atual += tam; ini = fim
    if atual:
        unidades.append(atual)
    return partes, unidades


def _onde(unidade):
    return " OR ".join(f"(_part = {lit(p)} AND _part_offset >= {a} AND _part_offset < {b})" for p, a, b in unidade)


def _sondar_poda():
    """O servidor lê só ~as linhas pedidas ao filtrar por _part_offset? (poda de granulos). Se não, a carga por chunks ficaria lenta."""
    t = max(ORIGENS, key=lambda x: cont[x])
    _, unidades = _plano_chunks(t, CHUNK_LINHAS)
    if len(unidades) < 6:
        return True
    u = unidades[len(unidades) // 2]; esperado = sum(f - i for _, i, f in u)
    consultar(f"SELECT sum(length(toString(`unnamed_0`))) AS s FROM `{t}` WHERE {_onde(u)}")
    lidas = (ULT_STATS or {}).get("rows_read")
    if lidas is None:
        return True
    ok = int(lidas) <= max(6 * esperado, esperado + 500_000)
    print(f"  sonda de leitura por faixa em {t}: {int(lidas):,} linhas lidas para {esperado:,} pedidas -> "
          f"{'poda OK (modo offset)' if ok else 'SEM poda (usando modo hash)'}")
    return ok


def _chave_particao(t):
    k = consultar(f"SELECT partition_key AS k FROM system.tables WHERE database = currentDatabase() AND name = {lit(t)}")
    return str(k["k"][0]) if len(k) else ""


def _particao_ok(destino):
    chave = _chave_particao(destino)
    if MODO_ATUAL == "offset":
        return chave.replace(" ", "") == "origem_tabela"
    m = re.search(r"%\s*(\d+)", chave)
    return bool(m) and int(m.group(1)) == N_PARTES


def _esquema_ok(destino):
    """prep_stg precisa ter, em cada feature, o tipo que o notebook espera (ex.: se uma coluna saiu de FEATURES_INT, o tipo mudou)."""
    if destino != "prep_stg":
        return True
    r = consultar(f"SELECT name, type FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(destino)}")
    atual = dict(zip(r["name"], r["type"]))
    dif = {c: (atual.get(c), tipo_ch(c)) for c in FEATURES_ORDEM if atual.get(c) != tipo_ch(c)}
    if dif:
        ex = ", ".join(f"{c}: {a} -> {b}" for c, (a, b) in list(dif.items())[:6])
        print(f"  ! `{destino}` tem {len(dif)} coluna(s) com tipo diferente do esperado ({ex}); será recriada.")
    return not dif


def _layout_ok(destino):
    return _particao_ok(destino) and _esquema_ok(destino)


_CHECAGEM_FEITA = []


def _checar_conversao_origem(t):
    """FALHA RÁPIDO: logo na 1ª origem carregada, confere se alguma feature 'normal' ficou com NULL (coluna declarada inteira com valores decimais)."""
    if _CHECAGEM_FEITA:
        return
    normais = [c for c in FEATURES_ORDEM if c not in COLS_STATUS]
    r = consultar_num(f"SELECT {', '.join(f'countIf(isNull({q(c)})) AS {q(c)}' for c in normais)} FROM prep_stg WHERE origem_tabela = {lit(t)}").iloc[0]
    ruins = r[r > 0]
    _CHECAGEM_FEITA.append(t)
    if len(ruins):
        raise RuntimeError(
            f"Conversão inesperada já na origem {t}: estas features ficaram com NULL (valores decimais em coluna declarada como inteira, FEATURES_INT):\n"
            + ruins.to_string() + "\nRemova-as de FEATURES_INT (célula de configuração) e rode de novo: a carga detecta a mudança de tipo e recria prep_stg sozinha. "
            "A carga foi interrompida agora para você não perder horas.")


def _val_carga(destino):
    r = consultar_num(f"SELECT origem_tabela, count() AS n FROM `{destino}` GROUP BY origem_tabela")
    return {str(a): int(b) for a, b in zip(r["origem_tabela"], r["n"])} == {t: int(cont[t]) for t in ORIGENS}


PLANO_PARTES = {}


def _descartar_origem(destino, t):
    aguardar_servidor(esperar_consultas=True, tabela=destino)
    executar(f"ALTER TABLE `{destino}` DROP PARTITION {lit(t)}")
    resto = n_linhas(destino, f"origem_tabela = {lit(t)}")
    if resto:
        raise RuntimeError(f"{destino}/{t}: DROP PARTITION não zerou a origem ({resto:,} linhas). Layout da tabela inesperado: use RECRIAR.add('{destino}').")


def _formula_atual(t):
    return "fisico" if t in ID_FISICO else "padrao"


def _formula_registrada(destino, t):
    return estado_get(f"idformula:{destino}:{t}", "padrao")          # cargas antigas usavam a fórmula padrão


def forcar_recarga_origem(t):
    """Descarta a origem t em prep_stg e prep_aux_ids (e as marcas). Depois rode a célula de carga de novo."""
    for d in ("prep_stg", "prep_aux_ids"):
        if existe(d):
            _descartar_origem(d, t)
        estado_set(f"idformula:{d}:{t}", None); estado_set(f"partes:{d}:{t}", None); estado_set(f"tabela:{d}", None)
    print(f"  origem {t} descartada em prep_stg e prep_aux_ids; rode a carga de novo.")


def _inserir_chunk(destino, t, k, unidades, cum, sql_fn):
    """Grava UM chunk (bloco único = atômico) e confirma pela contagem. Devolve 'ok' ou 'reiniciar' (recomeça a origem)."""
    global CHUNK_LINHAS
    antes, depois = (cum[k - 1] if k else 0), cum[k]
    onde = _onde(unidades[k]); falhas_mem = 0
    for tent in range(1, MAX_TENT_UNID + 2):
        if _partes_ativas(t) != PLANO_PARTES.get(t):
            print(f"    ! as partes de {t} mudaram durante a carga (merge no servidor); refazendo o plano desta origem")
            return "reiniciar"
        erro = None
        try:
            executar(sql_fn(t, onde, settings_carga(CHUNK_LINHAS)))
        except PermissionError:
            raise
        except Exception as e:
            erro = e
            if deterministico(e):
                raise RuntimeError(f"{destino}/{t} chunk {k + 1}: erro que não se resolve repetindo -> {str(e)[:700]}") from e
            if isinstance(e, ErroConexao):
                print(f"    ⚠ conexão caiu em {t} chunk {k + 1}; aguardando o servidor terminar a consulta antes de conferir...")
                aguardar_servidor(esperar_consultas=True, tabela=destino)
        tem = n_linhas(destino, f"origem_tabela = {lit(t)}")
        if tem == depois:
            return "ok"                                   # gravou tudo (inclusive se a resposta se perdeu)
        if tem != antes:
            print(f"    ! {t} chunk {k + 1}: {tem:,} linhas (esperado {antes:,} ou {depois:,}); estado inconsistente -> recomeçando a origem")
            return "reiniciar"
        if erro is None:                                   # respondeu OK mas não gravou: plano de partes desatualizado
            print(f"    ! {t} chunk {k + 1}: o INSERT não gravou linhas; refazendo o plano desta origem")
            return "reiniciar"
        if erro_memoria_ch(erro):
            falhas_mem += 1
            if falhas_mem >= 2 and CHUNK_LINHAS > 10_000:
                CHUNK_LINHAS = max(10_000, CHUNK_LINHAS // 2)
                print(f"    ↓ memória insuficiente 2x: reduzindo CHUNK_LINHAS para {CHUNK_LINHAS:,} e refazendo o plano de {t}")
                return "reiniciar"
        espera = ESPERAS_PARTE[min(tent - 1, len(ESPERAS_PARTE) - 1)]
        print(f"    ⚠ {t} chunk {k + 1}/{len(unidades)} tentativa {tent}/{MAX_TENT_UNID + 1}: {str(erro)[:350]}\n      nada foi gravado neste chunk; "
              f"aguardando {espera}s e repetindo só ele...")
        time.sleep(espera); aguardar_servidor()
    raise RuntimeError(f"{destino}/{t} chunk {k + 1}: falhou após {MAX_TENT_UNID + 1} tentativas. Último erro: {erro!r}. "
                       f"As linhas já gravadas estão corretas; rode a célula de novo para retomar deste ponto.")


def erro_memoria_ch(e):
    return any(k in str(e) for k in ("Code: 241", "MEMORY_LIMIT_EXCEEDED", "memory limit exceeded"))


def _carregar_origem(destino, t, sql_fn):
    for reinicio in range(1, 5):
        partes, unidades = _plano_chunks(t, CHUNK_LINHAS)
        PLANO_PARTES[t] = partes
        if t in ID_FISICO and destino == "prep_aux_ids":
            ref = estado_get(f"partes:prep_stg:{t}")
            if ref is not None and ref != [list(p) for p in partes]:
                raise RuntimeError(f"{t}: as partes físicas da tabela de origem mudaram entre a carga de prep_stg e a de prep_aux_ids; os ids "
                                   f"(que incluem a posição física) não coincidiriam. Execute forcar_recarga_origem('{t}') e rode a carga de novo.")
        cum = [int(x) for x in np.cumsum([sum(f - i for _, i, f in u) for u in unidades])]
        tem = n_linhas(destino, f"origem_tabela = {lit(t)}")
        if tem == 0:
            k0 = 0
        elif tem in cum:
            k0 = cum.index(tem) + 1
            if k0 < len(unidades):
                print(f"    ↻ {t}: retomando do chunk {k0 + 1}/{len(unidades)} ({tem:,} linhas já gravadas)")
        else:
            print(f"    ! {t}: {tem:,} linhas parciais não coincidem com nenhum limite de chunk; descartando a origem e recomeçando")
            _descartar_origem(destino, t); k0 = 0
        t0, resultado = time.time(), "ok"
        for k in range(k0, len(unidades)):
            resultado = _inserir_chunk(destino, t, k, unidades, cum, sql_fn)
            if resultado != "ok":
                break
            if (k + 1) % 25 == 0 or k + 1 == len(unidades):
                print(f"    … {t}: chunk {k + 1}/{len(unidades)} ({cum[k]:,}/{cont[t]:,} linhas, {time.time() - t0:,.0f}s)")
        if resultado == "ok" and n_linhas(destino, f"origem_tabela = {lit(t)}") == cont[t]:
            if t in ID_FISICO:
                estado_set(f"partes:{destino}:{t}", [list(p) for p in partes])
            return
        _descartar_origem(destino, t)
    raise RuntimeError(f"{destino}/{t}: não consegui carregar de forma consistente após 4 reinícios. Veja os avisos acima.")


def carregar_chunks(destino, ddl, sql_fn):
    if existe(destino) and not _layout_ok(destino):
        print(f"  ! `{destino}` tem outro layout (partição ou tipos de coluna); recriando (apenas tabela prep_*).")
        executar(f"DROP TABLE IF EXISTS `{destino}`")
    if not existe(destino):
        executar(ddl)
    for t in ORIGENS:
        if n_linhas(destino, f"origem_tabela = {lit(t)}") == cont[t]:
            if _formula_registrada(destino, t) == _formula_atual(t):
                continue
            print(f"  ! {destino}/{t}: os ids foram gerados com a fórmula '{_formula_registrada(destino, t)}' e agora são '{_formula_atual(t)}'; recarregando só esta origem.")
            _descartar_origem(destino, t)
        aguardar_servidor(esperar_consultas=True, tabela=destino)       # um INSERT de execução anterior pode ainda estar rodando
        t0 = time.time()
        _carregar_origem(destino, t, sql_fn)
        estado_set(f"idformula:{destino}:{t}", _formula_atual(t))
        print(f"  ✔ {destino} <- {t}: {n_linhas(destino, f'origem_tabela = {lit(t)}'):,} linhas em {time.time() - t0:,.0f}s")
        if destino == "prep_stg":
            _checar_conversao_origem(t)


# ---------------- plano B: carga por hash (v6/v7), só se o servidor não podar por _part_offset ----------------
def _cont_parte(destino, t, i, n):
    return n_linhas(destino, f"origem_tabela = {lit(t)} AND id_linha % {n} = {i}")


def _limpar_parte(destino, t, i):
    executar(f"ALTER TABLE `{destino}` DROP PARTITION ({lit(t)}, {i})")
    resto = _cont_parte(destino, t, i, N_PARTES)
    if resto != 0:
        raise RuntimeError(f"{destino}/{t} parte {i}: DROP PARTITION não zerou a parte ({resto:,} linhas).")


def carregar_hash(destino, ddl, sql_fn):
    if existe(destino) and not _layout_ok(destino):
        print(f"  ! `{destino}` tem outro layout de partição; recriando."); executar(f"DROP TABLE IF EXISTS `{destino}`")
    if not existe(destino):
        executar(ddl)
    for t in ORIGENS:
        esperado = cont[t]
        if n_linhas(destino, f"origem_tabela = {lit(t)}") == esperado:
            if _formula_registrada(destino, t) == _formula_atual(t):
                continue
            print(f"  ! {destino}/{t}: fórmula de id mudou; recarregando só esta origem.")
            for i_ in range(N_PARTES):
                executar(f"ALTER TABLE `{destino}` DROP PARTITION ({lit(t)}, {i_})")
        aguardar_servidor(esperar_consultas=True, tabela=destino)
        ep = consultar_num(f"SELECT {sql_id_linha(t)} % {N_PARTES} AS p, count() AS n FROM `{t}` GROUP BY p")
        esp_parte = {int(p): int(n) for p, n in zip(ep.p, ep.n)}
        t0 = time.time()
        for i in range(N_PARTES):
            alvo = esp_parte.get(i, 0)
            for tent in range(1, MAX_TENT_UNID + 2):
                tem = _cont_parte(destino, t, i, N_PARTES)
                if tem == alvo:
                    break
                if tem != 0:
                    print(f"    descartando resíduo parcial de {t} parte {i} ({tem:,} de {alvo:,})"); _limpar_parte(destino, t, i)
                aguardar_servidor()
                try:
                    executar(sql_fn(t, f"{sql_id_linha(t)} % {N_PARTES} = {i}", SETTINGS_CARGA_HASH))
                except PermissionError:
                    raise
                except Exception as e:
                    if deterministico(e):
                        raise RuntimeError(f"{destino}/{t} parte {i}: erro que não se resolve repetindo -> {str(e)[:700]}") from e
                    espera = ESPERAS_PARTE[min(tent - 1, len(ESPERAS_PARTE) - 1)]
                    print(f"    ⚠ {t} parte {i}, tentativa {tent}: {str(e)[:350]}\n      aguardando {espera}s...")
                    if isinstance(e, ErroConexao):
                        aguardar_servidor(esperar_consultas=True, tabela=destino)
                    time.sleep(espera)
            else:
                tem = _cont_parte(destino, t, i, N_PARTES)
                if tem != alvo:
                    raise RuntimeError(f"{destino}/{t} parte {i}: {tem:,} linhas (esperado {alvo:,}) após {MAX_TENT_UNID + 1} tentativas (ver erros acima).")
        if n_linhas(destino, f"origem_tabela = {lit(t)}") != esperado:
            raise RuntimeError(f"{destino}/{t}: contagem final diferente do esperado ({esperado:,}).")
        estado_set(f"idformula:{destino}:{t}", _formula_atual(t))
        print(f"  ✔ {destino} <- {t}: {esperado:,} linhas em {time.time() - t0:,.0f}s")
        if destino == "prep_stg":
            _checar_conversao_origem(t)


# ---------------- orquestração idempotente ----------------
def garantir_carga(destino, ddl_fn):
    sig = assinatura(SIG_STG, MODO_ATUAL, "carga2")
    pronta = lambda: _layout_ok(destino) and _val_carga(destino)
    forcar = destino in RECRIAR
    if existe(destino) and not forcar:
        marca = estado_get(f"tabela:{destino}")
        if marca == sig:
            if _ok(pronta):
                print(f"  = `{destino}` já está pronta e válida ({n_linhas(destino):,} linhas); não será refeita."); return False
            print(f"  ! `{destino}` constava como pronta, mas falhou na validação; será completada.")
        elif marca is None:
            aguardar_servidor(esperar_consultas=True, tabela=destino)
            if _ok(pronta):
                estado_set(f"tabela:{destino}", sig)
                print(f"  = `{destino}` já existia (sem marca), passou na validação e foi adotada sem refazer."); return False
    elif forcar and existe(destino):
        print(f"  ↻ `{destino}` está em RECRIAR; será refeita.")
        executar(f"DROP TABLE IF EXISTS `{destino}`")
    estado_set(f"tabela:{destino}", None)
    t0 = time.time()
    if MODO_ATUAL == "offset":
        carregar_chunks(destino, ddl_fn(), sql_insert_stg if destino == "prep_stg" else sql_insert_aux)
    else:
        carregar_hash(destino, ddl_fn(), sql_insert_stg if destino == "prep_stg" else sql_insert_aux)
    if not _val_carga(destino):
        raise RuntimeError(f"`{destino}`: carga terminou, mas a contagem por origem não bate com as tabelas de origem.")
    estado_set(f"tabela:{destino}", sig); RECRIAR.discard(destino)
    print(f"✔ `{destino}`: {n_linhas(destino):,} linhas (esperado {N_TREINO + N_TESTE:,}) em {time.time() - t0:,.0f}s")
    return True


MODO_ATUAL = _sondar_poda() and "offset" or "hash" if MODO_CARGA == "auto" else MODO_CARGA
print(f"Modo de carga: {MODO_ATUAL} | CHUNK_LINHAS = {CHUNK_LINHAS:,}")
garantir_carga("prep_stg", ddl_stg)
garantir_carga("prep_aux_ids", ddl_aux)

  sonda de leitura por faixa em data_tftp: 100,000 linhas lidas para 100,000 pedidas -> poda OK (modo offset)
Modo de carga: offset | CHUNK_LINHAS = 100,000
  = `prep_stg` já está pronta e válida (70,427,637 linhas); não será refeita.
  = `prep_aux_ids` já está pronta e válida (70,427,637 linhas); não será refeita.


False

### 1.1 Validações da camada de preparação (conversão, unicidade do `id_linha`, rótulos)

In [9]:
exigir("prep_stg", "prep_aux_ids")


def _validar_stg():
    normais = [c for c in FEATURES_ORDEM if c not in COLS_STATUS]
    exprs = [f"countIf(isNull({q(c)})) AS {q(c)}" for c in normais]
    for c in COLS_STATUS:
        exprs.append(f"countIf(isNull({q(c)}) AND {q(c + '_status')} = 'ok') AS {q('sem_motivo_' + c)}")
        exprs.append(f"countIf(isNotNull({q(c)}) AND {q(c + '_status')} != 'ok') AS {q('motivo_sem_null_' + c)}")
    soma = None
    for o in ORIGENS:        # (a) consultas curtas: uma origem por vez, em blocos de 20 colunas
        r = pd.concat([consultar_num(f"SELECT {', '.join(b)} FROM prep_stg WHERE origem_tabela = {lit(o)}").iloc[0] for b in blocos_de(exprs, 20)])
        soma = r if soma is None else soma + r
    ruins = soma[soma > 0]
    assert ruins.empty, ("Conversão inesperada (falha de tipo, não explicada pela EDA):\n" + ruins.to_string() +
                         "\n-> são colunas declaradas como inteiras (FEATURES_INT) com valores decimais: remova-as de FEATURES_INT e rode de novo (a carga recria prep_stg sozinha ao ver a mudança de tipo).")
    print("✔ Conversões consistentes: as 77 features normais não geraram NULL; nas 3 colunas com status, NULL <=> status != 'ok'.")

    # (b) id_linha único, conferido por origem (uniqExact de uma origem cabe na memória; o global não caberia)
    n_tot = u_tot = 0; repetidos = {}
    for o in ORIGENS:
        u = rodar_pesado(f"SELECT count() AS n, uniqExact(id_linha) AS distintos FROM prep_stg WHERE origem_tabela = {lit(o)} @@S@@").iloc[0]
        n_tot += int(u.n); u_tot += int(u.distintos)
        if u.n != u.distintos: repetidos[o] = int(u.n - u.distintos)
    print(f"id_linha: {n_tot:,} linhas, {u_tot:,} distintos (conferido por origem)")
    if repetidos:
        raise AssertionError(f"colisão/repetição de id_linha em {repetidos}: o id é hash dos identificadores brutos; se houver linhas repetidas no próprio CSV, "
                             "a junção por id_linha duplicaria linhas. Adicione essas origens a ID_FISICO (célula de configuração) e rode de novo: só elas serão recarregadas "
                             "(o id passa a incluir a posição física da linha).")

    # (c) rótulos do banco == rótulos da EDA (por tabela)
    r = consultar_num("SELECT origem_tabela, label_original, count() AS registros FROM prep_stg GROUP BY origem_tabela, label_original")
    obtido = {f"{a}|{b}": int(n) for a, b, n in zip(r.origem_tabela, r.label_original, r.registros)}
    assert obtido == ESPERADO_CLASSES, "rótulos/contagens por tabela divergem da EDA (verifique espaços/caixa)"
    print(f"✔ {len(obtido)} pares (tabela, rótulo) idênticos à EDA. Rótulos distintos: {sorted(set(r.label_original))}")
    return {"ok": True, "pares": len(obtido)}


_res_b1, _ = etapa("b1_validacao", SIG_STG, _validar_stg)

  = etapa 'b1_validacao' já concluída em 2026-10-14T19:52:11; resultado reaproveitado (RECRIAR.add('b1_validacao') para refazer).


### Exemplo (etapa 1 — tipos e identificadores): antes → depois
**Antes:** a tabela original guarda **tudo como texto** (`String`): `'12'`, `'8.5'`, `'Infinity'`, `''`, `' Syn'`.
**Depois (`prep_stg`):** cada coluna ganha o **tipo certo** (Int64 ou Float64, anuláveis); os identificadores (`flow_id`, IPs, portas, `timestamp`, `unnamed_0`) saem da matriz e vão para `prep_aux_ids`; a linha ganha um `id_linha` único.
A célula abaixo pega **uma linha real** da menor tabela original e mostra os dois lados.

In [10]:
def exemplo_tipos():
    o = min(ORIGENS, key=lambda t: cont[t])                       # menor origem: a consulta é rápida
    cols = [c for c in ["flow_duration", "total_fwd_packets", "total_backward_packets", "bwd_iat_mean", "bwd_packets_s",
                        "flow_bytes_s", "flow_packets_s", "simillarhttp", "protocol", "inbound"] if c in FEATURES_ORDEM]
    sel = ", ".join(f"toString({q(c)}) AS {q(c)}" for c in cols)
    bruto = consultar(f"SELECT toString(unnamed_0) AS unnamed_0, {sel}, toString(`label`) AS label_texto FROM `{o}` LIMIT 1")
    u0 = bruto["unnamed_0"][0]
    aux = consultar(f"SELECT toString(id_linha) AS id_linha, flow_id, source_ip, source_port, destination_ip, destination_port, timestamp_txt "
                    f"FROM prep_aux_ids WHERE origem_tabela = {lit(o)} AND unnamed_0 = {lit(u0)} LIMIT 1")
    idl = aux["id_linha"][0]
    stg = consultar(f"SELECT {sel}, label_original, toString(label_binary) AS label_binary FROM prep_stg WHERE origem_tabela = {lit(o)} AND id_linha = {idl}")
    linhas = [{"coluna": c, "ANTES (texto na original)": repr(bruto[c][0]), "DEPOIS (valor em prep_stg)": ("NULL" if stg[c][0] is None else stg[c][0]),
               "tipo em prep_stg": tipo_ch(c)} for c in cols]
    linhas.append({"coluna": "label", "ANTES (texto na original)": repr(bruto["label_texto"][0]), "DEPOIS (valor em prep_stg)": stg["label_original"][0],
                   "tipo em prep_stg": "LowCardinality(String), sem espaços"})
    linhas.append({"coluna": "label_binary (nova)", "ANTES (texto na original)": "—", "DEPOIS (valor em prep_stg)": stg["label_binary"][0], "tipo em prep_stg": "UInt8 (0 = BENIGN)"})
    print(f"Linha real da origem `{o}` (unnamed_0 = {u0}):"); display(pd.DataFrame(linhas))
    print("Identificadores: saíram da matriz de features e ficaram em prep_aux_ids (com id_linha = hash único):")
    display(aux.T.rename(columns={0: "valor"}))

In [11]:
try:
    exemplo_tipos()
except Exception as _e:
    print('(exemplo ao vivo indisponível agora:', str(_e)[:250], ')')

Linha real da origem `data2_portmap` (unnamed_0 = 24):


,coluna,ANTES (texto na original),DEPOIS (valor em prep_stg),tipo em prep_stg
0,flow_duration,'114456999',114456999,Nullable(Float64)
1,total_fwd_packets,'45',45,Nullable(Float64)
2,total_backward_packets,'0',0,Nullable(Int64)
3,bwd_iat_mean,'0.0',0,Nullable(Float64)
4,bwd_packets_s,'0.0',0,Nullable(Float64)
5,flow_bytes_s,'0.0',0,Nullable(Float64)
6,flow_packets_s,'0.3931607537604581',0.3931607537604581,Nullable(Float64)
7,simillarhttp,'0',0,Nullable(Float64)
8,protocol,'0',0,Nullable(Int64)
9,inbound,'0',0,Nullable(Int64)


Identificadores: saíram da matriz de features e ficaram em prep_aux_ids (com id_linha = hash único):


,valor
id_linha,3767656306684942068
flow_id,192.168.50.254-224.0.0.5-0-0-0
source_ip,192.168.50.254
source_port,0
destination_ip,224.0.0.5
destination_port,0
timestamp_txt,2018-11-03 09:18:16.964447


## 2. Valores inválidos (`NaN`, `±Inf`, vazios, texto)

Não imputamos nada com zero nem com o maior valor finito. Aqui: **contabilizar** por coluna, tabela, conjunto e classe, **investigar a causa** e **definir a política**. A imputação (mediana do treino) só é calculada no bloco 8, depois da separação treino/teste.

In [12]:
exigir("prep_stg", "prep_aux_ids")
DDL_AUD = ("CREATE TABLE prep_aud_invalidos (conjunto LowCardinality(String), origem_tabela LowCardinality(String), label_original LowCardinality(String), "
           "coluna String, status LowCardinality(String), registros UInt64) ENGINE = MergeTree PARTITION BY origem_tabela ORDER BY (conjunto, origem_tabela, coluna)")
_u_aud = []
for _o in ORIGENS:
    _p = [f"SELECT conjunto, origem_tabela, label_original, {lit(c)} AS coluna, {q(c + '_status')} AS status, count() AS registros "
          f"FROM prep_stg WHERE origem_tabela = {lit(_o)} AND {q(c + '_status')} != 'ok' GROUP BY conjunto, origem_tabela, label_original, status" for c in COLS_STATUS]
    _u_aud.append({"chave": _o, "particao": lit(_o), "sql": "INSERT INTO prep_aud_invalidos " + " UNION ALL ".join(_p) + " @@S@@", "esperado": None})


def _v_aud():
    esperado = sum(n_linhas("prep_stg", f"{q(c + '_status')} != 'ok'") for c in COLS_STATUS)
    return _soma("prep_aud_invalidos", "registros") == esperado


construir_por_unidades("prep_aud_invalidos", DDL_AUD, _u_aud, assinatura(SIG_STG, "aud_invalidos2"), _v_aud)
aud = consultar_num("SELECT * FROM prep_aud_invalidos")

print("== Por coluna x status x conjunto ==")
display(aud.pivot_table(index=["coluna", "status"], columns="conjunto", values="registros", aggfunc="sum", fill_value=0, margins=True))

tot = aud.groupby("status")["registros"].sum()
print("\nReconciliação com a EDA (+Inf=4.092.348 | vazio=249.152 | não numérico=6.145):",
      {k: int(tot.get(k, 0)) for k in ["pos_inf", "neg_inf", "nan", "vazio", "nao_numerico"]})

lin = consultar_num("""SELECT conjunto, count() AS total,
    countIf(flow_packets_s_status != 'ok' OR flow_bytes_s_status != 'ok') AS linhas_taxas_invalidas,
    countIf(simillarhttp_status != 'ok') AS linhas_texto_simillarhttp, countIf(n_ausentes > 0) AS linhas_com_algum_ausente
    FROM prep_stg GROUP BY conjunto""")
display(lin); print("EDA: linhas afetadas = 2.170.750 (tabela 05_linhas_afetadas)")

print("\n== % de linhas com taxa inválida por classe (flow_bytes_s ou flow_packets_s) ==")
por_classe = consultar_num("""SELECT conjunto, label_original, count() AS total,
    countIf(flow_packets_s_status != 'ok' OR flow_bytes_s_status != 'ok') AS invalidas FROM prep_stg GROUP BY conjunto, label_original""")
por_classe["pct"] = (100 * por_classe.invalidas / por_classe.total).round(2)
display(por_classe.sort_values(["conjunto", "pct"], ascending=[True, False]).reset_index(drop=True))

print("\n== Causa provável: Inf em taxas por duração zero? ==")
causa = consultar_num("""SELECT conjunto,
    countIf(flow_packets_s_status = 'pos_inf') AS pkts_inf, countIf(flow_packets_s_status = 'pos_inf' AND flow_duration = 0) AS pkts_inf_dur0,
    countIf(flow_bytes_s_status = 'pos_inf') AS bytes_inf, countIf(flow_bytes_s_status = 'pos_inf' AND flow_duration = 0) AS bytes_inf_dur0,
    countIf(flow_bytes_s_status = 'vazio') AS bytes_vazio, countIf(flow_bytes_s_status = 'vazio' AND flow_duration = 0) AS bytes_vazio_dur0
    FROM prep_stg GROUP BY conjunto""")
display(causa)

print("\n== simillarhttp: texto bruto por classe (risco de artefato de captura) ==")
display(consultar_num("SELECT label_original, count() AS linhas_com_texto FROM prep_stg WHERE simillarhttp_status = 'nao_numerico' GROUP BY label_original ORDER BY linhas_com_texto DESC"))
RELATORIO["invalidos"] = {"por_status": {k: int(v) for k, v in tot.items()}, "linhas": lin.to_dict("records")}


  = `prep_aud_invalidos` já está pronta e válida; não será refeita.
== Por coluna x status x conjunto ==


conjunto                       teste   treino      All
coluna         status                                 
flow_bytes_s   pos_inf        807386  1114212  1921598
               vazio             128   249024   249152
flow_packets_s pos_inf        807514  1363236  2170750
simillarhttp   nao_numerico     1738     4407     6145
All                          1616766  2730879  4347645


Reconciliação com a EDA (+Inf=4.092.348 | vazio=249.152 | não numérico=6.145): {'pos_inf': 4092348, 'neg_inf': 0, 'nan': 0, 'vazio': 249152, 'nao_numerico': 6145}
  conexão instável (Exception: HTTP 500 [código ClickHouse: 241] corpo='{\n\t"meta":\n\t[\n\t\t{\n\t\t\t"name": "conjunto",\n\t\t\t"type": "LowCardinality(String)"\n\t\t},\n\t\t{\n\t\t\t"name); nova tentativa em 5s...


,conjunto,total,linhas_taxas_invalidas,linhas_texto_simillarhttp,linhas_com_algum_ausente
0,treino,50063112,1363236,4407,1367643
1,teste,20364525,807514,1738,809252


EDA: linhas afetadas = 2.170.750 (tabela 05_linhas_afetadas)

== % de linhas com taxa inválida por classe (flow_bytes_s ou flow_packets_s) ==


,conjunto,label_original,total,invalidas,pct
0,teste,Syn,4891500,332667,6.80
1,teste,Portmap,186960,9763,5.22
2,teste,NetBIOS,3657497,141603,3.87
3,teste,MSSQL,5787453,202982,3.51
4,teste,LDAP,1915122,42983,2.24
5,teste,UDP,3867155,76857,1.99
6,teste,BENIGN,56965,659,1.16
7,teste,UDPLag,1873,0,0.00
8,treino,Syn,1582289,202306,12.79
9,treino,UDP-lag,366461,36382,9.93



== Causa provável: Inf em taxas por duração zero? ==
  conexão instável (Exception: HTTP 500 [código ClickHouse: 241] corpo='{\n\t"meta":\n\t[\n\t\t{\n\t\t\t"name": "conjunto",\n\t\t\t"type": "LowCardinality(String)"\n\t\t},\n\t\t{\n\t\t\t"name); nova tentativa em 5s...


,conjunto,pkts_inf,pkts_inf_dur0,bytes_inf,bytes_inf_dur0,bytes_vazio,bytes_vazio_dur0
0,treino,1363236,1363236,1114212,1114212,249024,249024
1,teste,807514,807514,807386,807386,128,128



== simillarhttp: texto bruto por classe (risco de artefato de captura) ==


,label_original,linhas_com_texto
0,BENIGN,3781
1,TFTP,1026
2,DrDoS_NTP,567
3,DrDoS_DNS,146
4,DrDoS_SNMP,130
5,DrDoS_UDP,121
6,WebDDoS,117
7,DrDoS_MSSQL,72
8,DrDoS_SSDP,64
9,DrDoS_NetBIOS,64


### Exemplo (etapa 2 — valores inválidos): antes → depois
**Antes:** a célula da tabela original pode ser `'Infinity'`, `'-Infinity'`, `'NaN'`, vazia (`''`) ou texto (`'abc'`).
**Depois:** vira **NULL** em `prep_stg` e o **motivo** fica em `<coluna>_status` (`pos_inf`, `neg_inf`, `nan`, `vazio`, `nao_numerico`). Nada é "chutado" para 0 nem para o maior valor.
Real (EDA): 4.092.348 valores `+Inf`, 249.152 células vazias em uma feature e 2.170.750 linhas afetadas; a causa dos infinitos é a duração zero (taxa = bytes ÷ 0).
A célula abaixo mostra **linhas reais** de cada tipo de problema que existir nos seus dados.

In [13]:
def exemplo_invalidos():
    linhas = []
    for st in ("pos_inf", "neg_inf", "vazio", "nan", "nao_numerico"):
        for col in COLS_STATUS:
            r = consultar(f"SELECT toString(id_linha) AS id_linha, origem_tabela FROM prep_stg WHERE {q(col + '_status')} = {lit(st)} LIMIT 1")
            if not len(r):
                continue
            o, idl = r["origem_tabela"][0], r["id_linha"][0]
            u0 = consultar(f"SELECT unnamed_0 FROM prep_aux_ids WHERE origem_tabela = {lit(o)} AND id_linha = {idl} LIMIT 1")
            if not len(u0):
                continue
            antes = consultar(f"SELECT toString({q(col)}) AS v FROM `{o}` WHERE toString(unnamed_0) = {lit(u0['unnamed_0'][0])} LIMIT 1")
            dep = consultar(f"SELECT toString({q(col)}) AS v, {q(col + '_status')} AS s FROM prep_stg WHERE origem_tabela = {lit(o)} AND id_linha = {idl}")
            linhas.append({"origem": o, "coluna": col, "ANTES (texto na original)": repr(antes["v"][0]) if len(antes) else "?",
                           "DEPOIS (valor em prep_stg)": "NULL" if dep["v"][0] is None else dep["v"][0], "status guardado": dep["s"][0]})
            break
    if linhas:
        display(pd.DataFrame(linhas))
    else:
        print("Nenhum valor inválido encontrado nas colunas com status.")

In [14]:
try:
    exemplo_invalidos()
except Exception as _e:
    print('(exemplo ao vivo indisponível agora:', str(_e)[:250], ')')

,origem,coluna,ANTES (texto na original),DEPOIS (valor em prep_stg),status guardado
0,data2_portmap,flow_bytes_s,'Infinity',NULL,pos_inf
1,data_drdos_udp,flow_bytes_s,'7.66E8',NULL,vazio
2,data_drdos_mssql,simillarhttp,'0',NULL,nao_numerico


### 2.1 Política para ausentes (decidida com a taxa **do treino**)

- Feature com ausentes ≤ `LIMITE_AUSENTES_REMOVER` → manter, **imputar pela mediana do treino** (bloco 8) e criar coluna indicadora `<feature>_ausente` (opcional nos experimentos).
- Feature acima do limite → sai da matriz, com a decisão registrada.
- Nenhum registro é removido por causa de valor inválido (só duplicatas do treino, bloco 4).

In [15]:
exigir("prep_stg")


def _calc_ausentes():
    exprs = [f"countIf(isNull({q(c)})) AS {q(c)}" for c in FEATURES_ORDEM]
    acc = {}
    for o in ORIGENS:        # consultas curtas por origem (blocos de 20 colunas), somadas por conjunto
        r = pd.concat([consultar_num(f"SELECT {'count() AS total, ' if k == 0 else ''}{', '.join(b)} FROM prep_stg WHERE origem_tabela = {lit(o)}").iloc[0]
                       for k, b in enumerate(blocos_de(exprs, 20))])
        cj = conjunto_de(o); acc[cj] = r if cj not in acc else acc[cj] + r
    linhas = []
    for c in FEATURES_ORDEM:
        linhas.append({"coluna": c, "nulos_treino": int(acc["treino"][c]), "nulos_teste": int(acc["teste"][c]),
                       "taxa_treino": float(acc["treino"][c] / acc["treino"]["total"]), "taxa_teste": float(acc["teste"][c] / acc["teste"]["total"])})
    return linhas


_linhas_aus, _ = etapa("b2_ausentes", assinatura(SIG_STG, "ausentes"), _calc_ausentes)
taxa = pd.DataFrame(_linhas_aus)
COLS_NULL = taxa.loc[(taxa.nulos_treino > 0) | (taxa.nulos_teste > 0), "coluna"].tolist()        # colunas que continuam Nullable
REMOVIDAS_AUSENTES = taxa.loc[taxa.taxa_treino > LIMITE_AUSENTES_REMOVER, "coluna"].tolist()
display(taxa[taxa.coluna.isin(COLS_NULL)])
print("Colunas com ausentes (viram Nullable + indicador):", COLS_NULL)
print("Removidas por excesso de ausentes:", REMOVIDAS_AUSENTES or "nenhuma")
RELATORIO["ausentes"] = {"colunas": COLS_NULL, "removidas_por_excesso": REMOVIDAS_AUSENTES,
                         "taxas_treino": {c: float(taxa.set_index("coluna").loc[c, "taxa_treino"]) for c in COLS_NULL}}


  = etapa 'b2_ausentes' já concluída em 2026-10-14T19:55:52; resultado reaproveitado (RECRIAR.add('b2_ausentes') para refazer).


,coluna,nulos_treino,nulos_teste,taxa_treino,taxa_teste
14,flow_bytes_s,1363236,807514,0.027230,0.039653
15,flow_packets_s,1363236,807514,0.027230,0.039653
78,simillarhttp,4407,1738,0.000088,0.000085


Colunas com ausentes (viram Nullable + indicador): ['flow_bytes_s', 'flow_packets_s', 'simillarhttp']
Removidas por excesso de ausentes: nenhuma


## 3. Features constantes e colunas exatamente duplicadas (decisões aprendidas só no treino)

- **Constantes:** valor único (ignorando ausentes) em todo o treino. O relatório mostra individualmente as 13 features apontadas pela EDA, informa quais foram confirmadas e também lista constantes novas.
- **Duplicadas:** a decisão de remover uma coluna é tomada **exclusivamente com os dados de treino**. Para cada par candidato, a primeira coluna (`mantida`) é a referência e a segunda (`candidata_a_remover`) só é removida se forem idênticas valor a valor no treino (comparação nula-segura).
- O teste é usado apenas para **auditoria posterior**: se uma coluna removida por duplicidade no treino diferir no teste, isso é registrado como alerta, mas não altera a seleção aprendida no treino.
- Correlação alta, mas não perfeita, não remove features automaticamente; é avaliada no bloco 7 para um experimento reduzido separado.

In [16]:
exigir("prep_stg")


def _calc_constantes():
    exprs = []
    for c in FEATURES_ORDEM:
        exprs += [f"min({q(c)}) AS {q('mn_' + c)}", f"max({q(c)}) AS {q('mx_' + c)}", f"countIf(isNotNull({q(c)})) AS {q('nv_' + c)}"]
    _partes = []
    for o in ORIGENS:        # uma origem por vez, em blocos de 10 colunas (30 expressões)
        r = pd.concat([consultar_num(f"SELECT {', '.join(b)} FROM prep_stg WHERE origem_tabela = {lit(o)}").iloc[0] for b in blocos_de(exprs, 30)]).to_frame().T.reset_index(drop=True)
        r = pd.concat([pd.DataFrame({"conjunto": [conjunto_de(o)], "origem_tabela": [o]}), r], axis=1); _partes.append(r)
    df = pd.concat(_partes, ignore_index=True)
    tr, te = df[df.conjunto == "treino"], df[df.conjunto == "teste"]
    linhas = []
    for c in FEATURES_ORDEM:
        nv_tr, nv_te = tr[f"nv_{c}"].sum(), te[f"nv_{c}"].sum()
        mn_tr, mx_tr, mn_te, mx_te = tr[f"mn_{c}"].min(), tr[f"mx_{c}"].max(), te[f"mn_{c}"].min(), te[f"mx_{c}"].max()
        linhas.append({"coluna": c, "validos_treino": int(nv_tr), "min_treino": mn_tr, "max_treino": mx_tr,
                       "constante_treino": bool(nv_tr > 0 and mn_tr == mx_tr), "sem_valor_valido_treino": bool(nv_tr == 0),
                       "validos_teste": int(nv_te), "min_teste": mn_te, "max_teste": mx_te,
                       "constante_teste": bool(nv_te > 0 and mn_te == mx_te), "na_eda": c in CONSTANTES_EDA})
    return linhas


_linhas_const, _ = etapa("b3_constantes", SIG_STG, _calc_constantes)
aud_const = pd.DataFrame(_linhas_const)
CONSTANTES = aud_const.loc[aud_const.constante_treino | aud_const.sem_valor_valido_treino, "coluna"].tolist()
aud_const["situacao"] = np.select(
    [aud_const.coluna.isin(CONSTANTES_EDA) & aud_const.coluna.isin(CONSTANTES),
     aud_const.coluna.isin(CONSTANTES_EDA) & ~aud_const.coluna.isin(CONSTANTES),
     ~aud_const.coluna.isin(CONSTANTES_EDA) & aud_const.coluna.isin(CONSTANTES)],
    ["EDA: confirmada constante no treino", "EDA: NÃO constante no treino; será mantida se não houver outro critério", "Constante adicional encontrada no treino; será removida"],
    default="Não indicada como constante na EDA")
# Exibir todas as 13 features reportadas pela EDA e todas as constantes novas encontradas.
relatorio_constantes = aud_const[aud_const.na_eda | aud_const.coluna.isin(CONSTANTES)].copy()
display(relatorio_constantes[["coluna", "na_eda", "constante_treino", "sem_valor_valido_treino", "constante_teste", "situacao"]])
so_eda = sorted(set(CONSTANTES_EDA) - set(CONSTANTES))
so_novo = sorted(set(CONSTANTES) - set(CONSTANTES_EDA))
print(f"Features constantes removidas: {len(CONSTANTES)}; constantes indicadas pela EDA: {len(CONSTANTES_EDA)}.")
print(f"Das {len(CONSTANTES_EDA)} features da EDA, {len(CONSTANTES_EDA) - len(so_eda)} foram confirmadas como constantes/sem valores válidos no treino.")
print("Na EDA, mas NÃO constantes no treino (não removidas por constância):", so_eda or "nenhuma")
print("Constantes adicionais fora da EDA (removidas):", so_novo or "nenhuma")
muda_teste = aud_const[aud_const.constante_treino & ~aud_const.constante_teste & (aud_const.validos_teste > 0)].coluna.tolist()
print("Constantes no treino que variam no teste (removidas da matriz por decisão baseada no treino):", muda_teste or "nenhuma")
RELATORIO["constantes"] = {"eda_total": len(CONSTANTES_EDA), "eda_confirmadas": len(CONSTANTES_EDA) - len(so_eda),
                          "removidas": CONSTANTES, "so_eda": so_eda, "so_novo": so_novo, "variam_no_teste": muda_teste,
                          "auditoria_eda": relatorio_constantes[["coluna", "constante_treino", "sem_valor_valido_treino", "constante_teste", "situacao"]].to_dict("records")}


  = etapa 'b3_constantes' já concluída em 2026-10-14T20:52:57; resultado reaproveitado (RECRIAR.add('b3_constantes') para refazer).


,coluna,na_eda,constante_treino,sem_valor_valido_treino,constante_teste,situacao
31,bwd_psh_flags,True,True,False,True,EDA: confirmada constante no treino
32,fwd_urg_flags,True,True,False,True,EDA: confirmada constante no treino
33,bwd_urg_flags,True,True,False,True,EDA: confirmada constante no treino
43,fin_flag_count,True,True,False,True,EDA: confirmada constante no treino
46,psh_flag_count,True,True,False,True,EDA: confirmada constante no treino
50,ece_flag_count,True,True,False,True,EDA: confirmada constante no treino
56,fwd_avg_bytes_bulk,True,True,False,True,EDA: confirmada constante no treino
57,fwd_avg_packets_bulk,True,True,False,True,EDA: confirmada constante no treino
58,fwd_avg_bulk_rate,True,True,False,True,EDA: confirmada constante no treino
59,bwd_avg_bytes_bulk,True,True,False,True,EDA: confirmada constante no treino


Features constantes removidas: 13; constantes indicadas pela EDA: 13.
Das 13 features da EDA, 13 foram confirmadas como constantes/sem valores válidos no treino.
Na EDA, mas NÃO constantes no treino (não removidas por constância): nenhuma
Constantes adicionais fora da EDA (removidas): nenhuma
Constantes no treino que variam no teste (removidas da matriz por decisão baseada no treino): nenhuma


In [17]:
# --- Colunas exatamente duplicadas: a decisão de seleção usa SOMENTE o treino ---
# Os pares candidatos vêm da EDA; a EDA serve apenas para propor pares a verificar.
cand = [(a, b) for a, b, r in PARES_EDA if abs(r) >= 0.99999 and a not in CONSTANTES and b not in CONSTANTES]


exigir("prep_stg")


def _calc_duplicadas():
    partes = []
    for i, (a, b) in enumerate(cand):
        eq = f"ifNull({q(a)} = {q(b)}, isNull({q(a)}) AND isNull({q(b)}) )"
        partes.append(f"countIf({eq} = 0) AS dif_{i}")
        partes.append(f"countIf(ifNull(abs({q(a)} - {q(b)}) > 1e-9 * greatest(1, abs({q(a)})), isNull({q(a)}) != isNull({q(b)})) = 1) AS tol_{i}")
    acc = {}
    for o in ORIGENS:        # uma consulta curta por origem, somada por conjunto
        r = consultar_num(f"SELECT count() AS n, {', '.join(partes)} FROM prep_stg WHERE origem_tabela = {lit(o)}").iloc[0]
        cj = conjunto_de(o); acc[cj] = r if cj not in acc else acc[cj] + r
    res = pd.DataFrame(acc).T
    linhas = []
    for i, (a, b) in enumerate(cand):
        dif_tr, dif_te = int(res.loc["treino", f"dif_{i}"]), int(res.loc["teste", f"dif_{i}"])
        linhas.append({"mantida": a, "candidata_a_remover": b,
                       "difere_treino": dif_tr, "difere_teste_auditoria": dif_te,
                       "difere_treino_tol1e-9": int(res.loc["treino", f"tol_{i}"]),
                       "difere_teste_tol1e-9_auditoria": int(res.loc["teste", f"tol_{i}"]),
                       "duplicada_no_treino": dif_tr == 0,
                       "difere_no_teste_alerta": dif_tr == 0 and dif_te > 0})
    return linhas


_linhas_dup, _ = etapa("b3_duplicadas", assinatura(SIG_STG, CONSTANTES, cand), _calc_duplicadas)
aud_dup = pd.DataFrame(_linhas_dup)
display(aud_dup)

# Só o resultado do treino decide a remoção. O teste não participa desta condição.
DUPLICADAS = {}
for a, b in aud_dup.loc[aud_dup.duplicada_no_treino, ["mantida", "candidata_a_remover"]].itertuples(index=False):
    representante = DUPLICADAS.get(a, a)
    DUPLICADAS[b] = representante
alertas_dup_teste = aud_dup.loc[aud_dup.difere_no_teste_alerta, ["mantida", "candidata_a_remover", "difere_teste_auditoria"]].to_dict("records")
print("Colunas removidas por duplicidade decidida no TREINO (coluna removida -> representante mantido):", DUPLICADAS or "nenhuma")
print("Pares candidatos que não são exatamente iguais no treino:", aud_dup.loc[~aud_dup.duplicada_no_treino, ["mantida", "candidata_a_remover"]].values.tolist() or "nenhum")
if alertas_dup_teste:
    print("ALERTA DE AUDITORIA: há colunas idênticas no treino, mas diferentes no teste. A seleção NÃO foi alterada:")
    display(pd.DataFrame(alertas_dup_teste))
else:
    print("Auditoria do teste: nenhum par removido por duplicidade no treino apresentou diferenças no teste.")
RELATORIO["duplicadas_colunas"] = {"removidas_decisao_treino": DUPLICADAS,
                                    "alertas_diferenca_teste": alertas_dup_teste,
                                    "pares_auditados": aud_dup.to_dict("records")}


  = etapa 'b3_duplicadas' já concluída em 2026-10-14T20:54:44; resultado reaproveitado (RECRIAR.add('b3_duplicadas') para refazer).


,mantida,candidata_a_remover,difere_treino,difere_teste_auditoria,difere_treino_tol1e-9,difere_teste_tol1e-9_auditoria,duplicada_no_treino,difere_no_teste_alerta
0,total_fwd_packets,subflow_fwd_packets,0,0,0,0,True,False
1,total_length_of_fwd_packets,subflow_fwd_bytes,0,0,0,0,True,False
2,total_backward_packets,subflow_bwd_packets,0,0,0,0,True,False
3,total_length_of_bwd_packets,subflow_bwd_bytes,0,0,0,0,True,False
4,fwd_psh_flags,rst_flag_count,0,0,0,0,True,False
5,bwd_packet_length_mean,avg_bwd_segment_size,4889,5612,0,0,False,False
6,fwd_packet_length_mean,avg_fwd_segment_size,61437,10703,0,0,False,False
7,fwd_header_length,fwd_header_length_1,0,0,0,0,True,False


Colunas removidas por duplicidade decidida no TREINO (coluna removida -> representante mantido): {'subflow_fwd_packets': 'total_fwd_packets', 'subflow_fwd_bytes': 'total_length_of_fwd_packets', 'subflow_bwd_packets': 'total_backward_packets', 'subflow_bwd_bytes': 'total_length_of_bwd_packets', 'rst_flag_count': 'fwd_psh_flags', 'fwd_header_length_1': 'fwd_header_length'}
Pares candidatos que não são exatamente iguais no treino: [['bwd_packet_length_mean', 'avg_bwd_segment_size'], ['fwd_packet_length_mean', 'avg_fwd_segment_size']]
Auditoria do teste: nenhum par removido por duplicidade no treino apresentou diferenças no teste.


In [18]:
# --- matriz base (F0) e registro das decisões (estado + prep_selecao_features) ---
_faltam = [v for v in ('COLS_NULL', 'CONSTANTES', 'DUPLICADAS', 'REMOVIDAS_AUSENTES') if v not in globals()]
assert not _faltam, f"Etapa anterior NÃO concluída: faltam {_faltam}. Execute, em ordem, as células de ausentes (bloco 2.1), constantes e duplicadas (bloco 3)."
FEATURES_BASE = [c for c in FEATURES_ORDEM if c not in CONSTANTES and c not in DUPLICADAS and c not in REMOVIDAS_AUSENTES]
COLS_NULL_BASE = [c for c in COLS_NULL if c in FEATURES_BASE]

_est = estado_get("selecao")
if (isinstance(_est, dict) and _est.get("features_base") == FEATURES_BASE and _est.get("cols_null") == COLS_NULL
        and _est.get("constantes") == CONSTANTES and _est.get("duplicadas") == DUPLICADAS
        and _est.get("removidas_ausentes") == REMOVIDAS_AUSENTES
        and existe("prep_selecao_features") and n_linhas("prep_selecao_features") == len(FEATURES_ORDEM)):
    restaurar_selecao()
    print("= seleção de features já registrada e idêntica; mantida (inclui FEATURES_REDUZIDO, se o bloco 7 já rodou).")
else:
    FEATURES_REDUZIDO = list(FEATURES_BASE)
    salvar_selecao(FEATURES_BASE)
    print("✔ seleção de features registrada (estado + prep_selecao_features).")
print(f"Matriz base: {len(FEATURES_BASE)} features (de 80): -{len(CONSTANTES)} constantes, -{len(DUPLICADAS)} duplicadas, -{len(REMOVIDAS_AUSENTES)} por ausentes.")
print("Colunas base que mantêm ausentes (imputação + indicador):", COLS_NULL_BASE)
RELATORIO["features_base"] = FEATURES_BASE


= seleção de features já registrada e idêntica; mantida (inclui FEATURES_REDUZIDO, se o bloco 7 já rodou).
Matriz base: 61 features (de 80): -13 constantes, -6 duplicadas, -0 por ausentes.
Colunas base que mantêm ausentes (imputação + indicador): ['flow_bytes_s', 'flow_packets_s']


### Exemplo (etapa 3 — features removidas): antes → depois
**Antes:** 80 features candidatas. **Depois:** `FEATURES_BASE` = 80 − constantes − colunas idênticas − ausentes > 50%.
- **Constante** (ex.: `bwd_psh_flags`): vale o mesmo número em **todas** as linhas do treino → não ajuda a separar classes → removida.
- **Idêntica** (ex.: `subflow_fwd_packets` == `total_fwd_packets`): o mesmo valor em todas as linhas → mantém só uma.
A decisão usa **somente o treino**; a mesma lista é aplicada ao teste. A célula abaixo mostra os valores **reais**.

In [19]:
def exemplo_features():
    print(f"Candidatas: {len(FEATURES_ORDEM)} | constantes no treino: {len(CONSTANTES)} | idênticas a outra coluna: {len(DUPLICADAS)} | "
          f"removidas por ausentes: {len(REMOVIDAS_AUSENTES)} | FEATURES_BASE: {len(FEATURES_BASE)}")
    for c in list(CONSTANTES)[:4]:
        r = consultar(f"SELECT toString(min({q(c)})) AS mn, toString(max({q(c)})) AS mx, toString(count({q(c)})) AS n FROM prep_stg WHERE conjunto = 'treino'")
        print(f"  CONSTANTE  {c}: mínimo = {r['mn'][0]}, máximo = {r['mx'][0]} em {int(r['n'][0]):,} linhas do treino  ->  removida")
    for b, a in list(DUPLICADAS.items())[:3]:
        r = consultar(f"SELECT toString({q(a)}) AS a, toString({q(b)}) AS b FROM prep_stg WHERE conjunto = 'treino' AND isNotNull({q(a)}) LIMIT 3")
        print(f"  IDÊNTICA   {b} == {a}. 3 linhas reais -> {a}: {list(r['a'])} | {b}: {list(r['b'])}  ->  fica só {a}")

In [20]:
try:
    exemplo_features()
except Exception as _e:
    print('(exemplo ao vivo indisponível agora:', str(_e)[:250], ')')

Candidatas: 80 | constantes no treino: 13 | idênticas a outra coluna: 6 | removidas por ausentes: 0 | FEATURES_BASE: 61
  CONSTANTE  bwd_psh_flags: mínimo = 0, máximo = 0 em 50,063,112 linhas do treino  ->  removida
  CONSTANTE  fwd_urg_flags: mínimo = 0, máximo = 0 em 50,063,112 linhas do treino  ->  removida
  CONSTANTE  bwd_urg_flags: mínimo = 0, máximo = 0 em 50,063,112 linhas do treino  ->  removida
  CONSTANTE  fin_flag_count: mínimo = 0, máximo = 0 em 50,063,112 linhas do treino  ->  removida
  IDÊNTICA   subflow_fwd_packets == total_fwd_packets. 3 linhas reais -> total_fwd_packets: ['2', '2', '2'] | subflow_fwd_packets: ['2', '2', '2']  ->  fica só total_fwd_packets
  IDÊNTICA   subflow_fwd_bytes == total_length_of_fwd_packets. 3 linhas reais -> total_length_of_fwd_packets: ['0', '0', '0'] | subflow_fwd_bytes: ['0', '0', '0']  ->  fica só total_length_of_fwd_packets
  IDÊNTICA   subflow_bwd_packets == total_backward_packets. 3 linhas reais -> total_backward_packets: ['0', '0', 

## 4. Duplicatas, conflitos de rótulo e sobreposição treino × teste

Chave de comparação = **as mesmas features da matriz base** (sem identificadores, constantes nem colunas duplicadas) + indicador de ausência nas colunas anuláveis (para que `NULL` nunca se confunda com `0`). Hash de 128 bits (`sipHash128`).

Regras:
1. **Treino:** mantém **uma** ocorrência por (features, rótulo) — a de menor `id_linha`, para ser reprodutível — e guarda `n_repeticoes`.
2. **Conflitos** (mesmas features, rótulos diferentes): **não** se escolhe rótulo nem se descarta em massa; os grupos ficam marcados (`conflito_rotulo`, `conflito_binario`) para análise de sensibilidade.
3. **Teste:** **não** é deduplicado (avaliação oficial preservada). A sobreposição com o treino é medida e marcada em cada linha do teste (`sobrepoe_treino`, `sobrepoe_treino_mesmo_rotulo`), permitindo um experimento extra sem sobreposição.
4. Treino e teste **nunca** são deduplicados em conjunto.

In [21]:
restaurar_selecao()
exigir("prep_stg")
# ATENÇÃO: sipHash128 devolve NULL se QUALQUER argumento Nullable for NULL, e NULL em FixedString(16) vira 16 bytes zero => todas as linhas com
# valor inválido colidiriam no mesmo hash. Por isso cada feature entra como ifNull(coluna, 0) + um indicador isNull(coluna) para as colunas com ausentes.
args = [f"ifNull({q(c)}, 0)" for c in FEATURES_BASE] + [f"isNull({q(c)})" for c in COLS_NULL_BASE]
SIG_HASH = assinatura(SIG_FEATS, "hash3-nullsafe", LINHAS_POR_UNIDADE)
DDL_HASH = ("CREATE TABLE prep_hash (id_linha UInt64, conjunto LowCardinality(String), origem_tabela LowCardinality(String), "
            "label_original LowCardinality(String), label_binary UInt8, h_feat FixedString(16), b UInt8) "
            "ENGINE = MergeTree PARTITION BY (origem_tabela, b) ORDER BY (conjunto, h_feat)")
_u_hash = []
for _o in ORIGENS:
    for _j, (_lo, _hi) in enumerate(faixas_id(cont[_o], LINHAS_POR_UNIDADE)):
        _cond = f"origem_tabela = {lit(_o)} AND {cond_id('id_linha', _lo, _hi)}"
        _u_hash.append({"chave": f"{_o}#{_j}", "particao": f"({lit(_o)}, {_j})", "esperado": n_linhas("prep_stg", _cond),
                        "contar": (lambda o=_o, j=_j: n_linhas("prep_hash", f"origem_tabela = {lit(o)} AND b = {j}")),
                        "sql": (f"INSERT INTO prep_hash SELECT id_linha, conjunto, origem_tabela, label_original, label_binary, "
                                f"sipHash128({', '.join(args)}), toUInt8({_j}) FROM prep_stg WHERE {_cond} @@S@@")})
construir_por_unidades("prep_hash", DDL_HASH, _u_hash, SIG_HASH,
                       lambda: n_linhas("prep_hash") == N_TREINO + N_TESTE and "b" in _colunas_de("prep_hash"))
print(f"prep_hash: {n_linhas('prep_hash'):,} linhas (esperado {N_TREINO + N_TESTE:,}) | hash sobre {len(FEATURES_BASE)} features + {len(COLS_NULL_BASE)} indicadores")

  = `prep_hash` já está pronta e válida; não será refeita.
prep_hash: 70,427,637 linhas (esperado 70,427,637) | hash sobre 61 features + 2 indicadores


### 4.1 Sobreposição treino × teste e evidência sobre os rótulos (antes de definir o mapa de rótulos)

In [22]:
restaurar_selecao()


exigir("prep_hash")


def _calc_sobreposicao():
    K = max(1, -(-N_TREINO // LINHAS_POR_BUCKET)); ss, cc = [], []
    for cond_b, _j in faixas_hash(K):        # cada faixa tem ~1/K dos hashes: o conjunto do IN e a junção ficam pequenos
        ss.append(rodar_pesado(f"""
            SELECT origem_tabela, label_original, count() AS linhas_teste,
                   countIf(h_feat IN (SELECT h_feat FROM prep_hash WHERE conjunto = 'treino' AND {cond_b})) AS com_features_no_treino
            FROM prep_hash WHERE conjunto = 'teste' AND {cond_b} GROUP BY origem_tabela, label_original @@S@@"""))
        cc.append(rodar_pesado(f"""
            SELECT te.label_original AS rotulo_teste, tr.label_original AS rotulo_treino, count() AS linhas_teste
            FROM (SELECT h_feat, label_original FROM prep_hash WHERE conjunto = 'teste' AND {cond_b}) AS te
            INNER JOIN (SELECT h_feat, label_original FROM prep_hash WHERE conjunto = 'treino' AND {cond_b} GROUP BY h_feat, label_original) AS tr ON te.h_feat = tr.h_feat
            GROUP BY rotulo_teste, rotulo_treino @@S@@"""))
    sobre = pd.concat(ss).groupby(["origem_tabela", "label_original"], as_index=False).sum()
    cruz = pd.concat(cc).groupby(["rotulo_teste", "rotulo_treino"], as_index=False).sum()
    return {"sobre": sobre.to_dict("records"), "cruz": cruz.to_dict("records")}


_res_sobre, _ = etapa("b4_sobreposicao", assinatura(SIG_HASH, SIG_FEATS, "sobreposicao"), _calc_sobreposicao)
sobre, cruz = pd.DataFrame(_res_sobre["sobre"]), pd.DataFrame(_res_sobre["cruz"])
sobre["pct"] = (100 * sobre.com_features_no_treino / sobre.linhas_teste).round(2)
display(sobre.sort_values(["origem_tabela", "linhas_teste"], ascending=[True, False]).reset_index(drop=True))
tot_t, tot_s = int(sobre.linhas_teste.sum()), int(sobre.com_features_no_treino.sum())
print(f"Teste: {tot_s:,} de {tot_t:,} linhas ({100 * tot_s / tot_t:.2f}%) têm features idênticas a alguma linha do treino.")
RELATORIO["sobreposicao_teste_com_treino"] = {"linhas_teste": tot_t, "com_features_no_treino": tot_s}

print("\n== Quando as features coincidem, que rótulos aparecem em cada lado? (linhas do teste x rótulo no treino) ==")
display(cruz.sort_values("linhas_teste", ascending=False).head(40).reset_index(drop=True))
print("Leitura: se 'LDAP' (teste) casa sobretudo com 'DrDoS_LDAP' (treino), isso sustenta MAPA_ROTULOS_EXTRA = {'DrDoS_LDAP': 'LDAP'}. "
      "O mapa NÃO é aplicado automaticamente (exceto UDP-lag, no bloco 5).")
cruz.to_csv(ARTEFATOS / "evidencia_rotulos_treino_x_teste.csv", index=False)


  = etapa 'b4_sobreposicao' já concluída em 2026-10-14T21:27:22; resultado reaproveitado (RECRIAR.add('b4_sobreposicao') para refazer).


,origem_tabela,label_original,linhas_teste,com_features_no_treino,pct
0,data2_ldap,LDAP,1905191,1793296,94.13
1,data2_ldap,NetBIOS,202919,200630,98.87
2,data2_ldap,BENIGN,5124,213,4.16
3,data2_mssql,MSSQL,5763061,4912367,85.24
4,data2_mssql,LDAP,9931,9335,94.00
5,data2_mssql,BENIGN,2794,87,3.11
6,data2_netbios,NetBIOS,3454578,3365692,97.43
7,data2_netbios,BENIGN,1321,13,0.98
8,data2_portmap,Portmap,186960,184258,98.55
9,data2_portmap,BENIGN,4734,55,1.16


Teste: 13,145,336 de 20,364,525 linhas (64.55%) têm features idênticas a alguma linha do treino.

== Quando as features coincidem, que rótulos aparecem em cada lado? (linhas do teste x rótulo no treino) ==


,rotulo_teste,rotulo_treino,linhas_teste
0,MSSQL,DrDoS_MSSQL,4697674
1,MSSQL,DrDoS_NetBIOS,3759818
2,MSSQL,DrDoS_DNS,3668200
3,NetBIOS,DrDoS_NetBIOS,3565846
4,NetBIOS,DrDoS_SNMP,3541665
5,UDP,DrDoS_UDP,2562460
6,UDP,DrDoS_SSDP,2492015
7,NetBIOS,DrDoS_DNS,2440432
8,LDAP,DrDoS_LDAP,1783115
9,UDP,UDP-lag,1766715


Leitura: se 'LDAP' (teste) casa sobretudo com 'DrDoS_LDAP' (treino), isso sustenta MAPA_ROTULOS_EXTRA = {'DrDoS_LDAP': 'LDAP'}. O mapa NÃO é aplicado automaticamente (exceto UDP-lag, no bloco 5).


### Exemplo (etapa 7 — sobreposição treino × teste): antes → depois
**Antes:** o teste é uma tabela à parte (`data2_*`), nunca deduplicada.
**Depois:** cada linha do teste ganha `sobrepoe_treino = 1` se existe no treino uma linha com **exatamente as mesmas features**, e `sobrepoe_treino_mesmo_rotulo = 1` se o rótulo também coincide. Isso **mede** o vazamento (não o remove): fluxos idênticos nos dois conjuntos podem inflar as métricas.
Abaixo, os números **reais** por tabela de teste e rótulo.

In [23]:
def exemplo_sobreposicao():
    d = pd.DataFrame(sobre).copy()
    d["% do teste que também está no treino"] = (d["com_features_no_treino"] / d["linhas_teste"] * 100).round(2)
    display(d.sort_values("com_features_no_treino", ascending=False).head(10).reset_index(drop=True))
    print(f"Total: {int(d.com_features_no_treino.sum()):,} de {int(d.linhas_teste.sum()):,} linhas de teste ({d.com_features_no_treino.sum() / d.linhas_teste.sum() * 100:.2f}%) têm features idênticas a alguma do treino.")

In [24]:
try:
    exemplo_sobreposicao()
except Exception as _e:
    print('(exemplo ao vivo indisponível agora:', str(_e)[:250], ')')

,origem_tabela,label_original,linhas_teste,com_features_no_treino,pct,% do teste que também está no treino
0,data2_mssql,MSSQL,5763061,4912367,85.24,85.24
1,data2_netbios,NetBIOS,3454578,3365692,97.43,97.43
2,data2_udp,UDP,3754680,2579201,68.69,68.69
3,data2_ldap,LDAP,1905191,1793296,94.13,94.13
4,data2_ldap,NetBIOS,202919,200630,98.87,98.87
5,data2_portmap,Portmap,186960,184258,98.55,98.55
6,data2_udplag,UDP,112475,73208,65.09,65.09
7,data2_udp,MSSQL,24392,20844,85.45,85.45
8,data2_mssql,LDAP,9931,9335,94.00,94.00
9,data2_syn,Syn,4284751,4333,0.10,0.10


Total: 13,145,336 de 20,364,525 linhas de teste (64.55%) têm features idênticas a alguma do treino.


### 4.2 Grupos de features no treino, conflitos e deduplicação
(O mapa de rótulos é definido no bloco 5; execute o bloco 5.1 **antes** deste bloco 4.2. Ele só depende de `label_original` e da sobreposição acima.)

## 5. Rótulos: original, multiclasse padronizado e binário

- `label_original`: nunca alterado. `label_binary`: 0 = `BENIGN`, 1 = qualquer ataque.
- `label_multi`: nome padronizado. Única unificação automática: **UDP-lag → UDPLag** (`UNIFICAR_UDPLAG`), porque o treino usa a grafia `UDP-lag` e o teste `UDPLag`. A evidência abaixo (perfil das features e sobreposição) deve ser revista antes de manter essa decisão.
- `DrDoS_X` × `X` **não** são unificados sem decisão sua (use `MAPA_ROTULOS_EXTRA`); a tabela de classes mostra quais existem só em um lado.

In [25]:
exigir("prep_stg", "prep_hash")
print("== Perfil médio das classes relacionadas a UDP (evidência UDP-lag × UDPLag) ==")
perfil = consultar_num("""SELECT label_original, conjunto, count() AS linhas, round(avg(flow_duration)) AS dur_media,
    round(avg(total_fwd_packets), 2) AS pkts_fwd_media, round(avg(fwd_packet_length_mean), 1) AS tam_pkt_medio,
    round(avg(init_win_bytes_forward)) AS janela_ini, round(avg(protocol), 1) AS protocolo_medio
    FROM prep_stg WHERE label_original IN ('UDP-lag', 'UDPLag', 'UDP', 'DrDoS_UDP', 'Syn') GROUP BY label_original, conjunto ORDER BY label_original""")
display(perfil)
if "cruz" not in globals():
    cruz = pd.DataFrame(estado_get("etapa:b4_sobreposicao")["res"]["cruz"])
ev = cruz[(cruz.rotulo_teste == "UDPLag") | (cruz.rotulo_treino == "UDP-lag")]
print("Cruzamento por features idênticas (linhas do teste):"); display(ev)
print("As origens também sugerem a mesma classe: treino 'data_udplag' contém 'UDP-lag'; teste 'data2_udplag' contém 'UDPLag'.")

MAPA = {}
if UNIFICAR_UDPLAG: MAPA["UDP-lag"] = "UDPLag"
MAPA.update(MAPA_ROTULOS_EXTRA)
rotulos = sorted(consultar("SELECT DISTINCT label_original AS r FROM prep_stg")["r"].tolist())
mapa_df = pd.DataFrame({"label_original": rotulos})
mapa_df["label_multi"] = mapa_df.label_original.map(lambda r: MAPA.get(r, r))
mapa_df["motivo"] = mapa_df.label_original.map(lambda r: "padronização de grafia (UDP-lag = UDPLag)" if MAPA.get(r) == "UDPLag" and r == "UDP-lag"
                                              else "decisão do usuário (MAPA_ROTULOS_EXTRA)" if r in MAPA_ROTULOS_EXTRA else "sem alteração")
mapa_df["label_binary"] = (mapa_df.label_original != "BENIGN").astype(int)


def _acoes_mapa(S):
    executar("CREATE TABLE prep_mapa_rotulos (label_original String, label_multi String, motivo String, label_binary UInt8) ENGINE = MergeTree ORDER BY label_original")
    inserir_df("prep_mapa_rotulos", mapa_df)


def _v_mapa():
    t = consultar("SELECT label_original, label_multi FROM prep_mapa_rotulos")
    return dict(zip(t.label_original, t.label_multi)) == dict(zip(mapa_df.label_original, mapa_df.label_multi))


construir("prep_mapa_rotulos", _acoes_mapa, assinatura(SIG_STG, MAPA, rotulos), _v_mapa)
display(mapa_df)
RELATORIO["mapa_rotulos"] = MAPA


== Perfil médio das classes relacionadas a UDP (evidência UDP-lag × UDPLag) ==


,label_original,conjunto,linhas,dur_media,pkts_fwd_media,tam_pkt_medio,janela_ini,protocolo_medio
0,DrDoS_UDP,treino,3134645,90551,3.77,372.8,6,17.0
1,Syn,teste,4891500,5376758,2.85,6.5,5835,6.0
2,Syn,treino,1582289,8086316,2.98,0.1,5838,6.0
3,UDP,teste,3867155,85414,3.76,371.6,-1,17.0
4,UDP-lag,treino,366461,5208263,2.93,65.8,5061,7.5
5,UDPLag,teste,1873,623687,2.71,362.9,838,16.0


Cruzamento por features idênticas (linhas do teste):


,rotulo_teste,rotulo_treino,linhas_teste
20,LDAP,UDP-lag,602393
31,MSSQL,UDP-lag,158946
42,NetBIOS,UDP-lag,787
53,Portmap,UDP-lag,345
65,Syn,UDP-lag,2113
78,UDP,UDP-lag,1766715
79,UDPLag,DrDoS_DNS,361
80,UDPLag,DrDoS_LDAP,2
81,UDPLag,DrDoS_MSSQL,198
82,UDPLag,DrDoS_NTP,9


As origens também sugerem a mesma classe: treino 'data_udplag' contém 'UDP-lag'; teste 'data2_udplag' contém 'UDPLag'.
  = `prep_mapa_rotulos` já está pronta e válida; não será refeita.


,label_original,label_multi,motivo,label_binary
0,BENIGN,BENIGN,sem alteração,0
1,DrDoS_DNS,DrDoS_DNS,sem alteração,1
2,DrDoS_LDAP,DrDoS_LDAP,sem alteração,1
3,DrDoS_MSSQL,DrDoS_MSSQL,sem alteração,1
4,DrDoS_NTP,DrDoS_NTP,sem alteração,1
5,DrDoS_NetBIOS,DrDoS_NetBIOS,sem alteração,1
6,DrDoS_SNMP,DrDoS_SNMP,sem alteração,1
7,DrDoS_SSDP,DrDoS_SSDP,sem alteração,1
8,DrDoS_UDP,DrDoS_UDP,sem alteração,1
9,LDAP,LDAP,sem alteração,1


### Exemplo (etapa 5 — rótulos): antes → depois
**Antes:** `label` (texto) com nomes diferentes para o mesmo ataque, por dia de captura: `UDP-lag` (treino, 366.461 linhas) e `UDPLag` (teste, 1.873 linhas).
**Depois:** `label_original` (**sempre preservado**), `label_multi` (nome padronizado, ex.: `UDPLag` nos dois conjuntos) e `label_binary` (0 = BENIGN, 1 = ataque).
Abaixo, **contagens reais** antes (por `label_original`) e depois (por `label_multi`).

In [26]:
def exemplo_rotulos():
    a = consultar_num("SELECT conjunto, label_original, count() AS linhas FROM prep_stg GROUP BY conjunto, label_original")
    a["label_multi"] = a["label_original"].map(lambda x: MAPA.get(x, x)); a["label_binary"] = (a["label_original"] != "BENIGN").astype(int)
    print("ANTES -> DEPOIS (só os rótulos cujo nome muda, e BENIGN):")
    mud = a[(a.label_original != a.label_multi) | (a.label_original == "BENIGN")].sort_values(["label_original", "conjunto"])
    display(mud.reset_index(drop=True))
    print("DEPOIS: linhas por label_multi e conjunto:")
    display(a.pivot_table(index="label_multi", columns="conjunto", values="linhas", aggfunc="sum", fill_value=0).astype(int))

In [27]:
try:
    exemplo_rotulos()
except Exception as _e:
    print('(exemplo ao vivo indisponível agora:', str(_e)[:250], ')')

  conexão instável (Exception: HTTP 500 [código ClickHouse: 241] corpo='{\n\t"meta":\n\t[\n\t\t{\n\t\t\t"name": "conjunto",\n\t\t\t"type": "LowCardinality(String)"\n\t\t},\n\t\t{\n\t\t\t"name); nova tentativa em 5s...
ANTES -> DEPOIS (só os rótulos cujo nome muda, e BENIGN):


,conjunto,label_original,linhas,label_multi,label_binary
0,teste,BENIGN,56965,BENIGN,0
1,treino,BENIGN,56863,BENIGN,0
2,treino,UDP-lag,366461,UDPLag,1


DEPOIS: linhas por label_multi e conjunto:


conjunto,teste,treino
label_multi,,
BENIGN,56965,56863
DrDoS_DNS,0,5071011
DrDoS_LDAP,0,2179930
DrDoS_MSSQL,0,4522492
DrDoS_NTP,0,1202642
DrDoS_NetBIOS,0,4093279
DrDoS_SNMP,0,5159870
DrDoS_SSDP,0,2610611
DrDoS_UDP,0,3134645


### 4.2 (continuação) Grupos de features, conflitos de rótulo e deduplicação do treino

In [28]:
restaurar_selecao(); garantir_mapa()
exigir("prep_hash", "prep_mapa_rotulos")
LM = expr_label_multi("label_original")
SIG_DEDUP = assinatura(SIG_HASH, SIG_FEATS, MAPA, "dedup2", LINHAS_POR_BUCKET)
FH = faixas_hash(max(1, -(-N_TREINO // LINHAS_POR_BUCKET)))          # faixas pelo 1º byte do hash (poda a chave primária)
DDL_GRP = ("CREATE TABLE prep_grp_treino (h_feat FixedString(16), n_linhas UInt64, n_rotulos UInt64, n_binarios UInt64, rotulos Array(String), b UInt8) "
           "ENGINE = MergeTree PARTITION BY b ORDER BY h_feat")
DDL_KEEP = ("CREATE TABLE prep_keep_treino (h_feat FixedString(16), label_multi LowCardinality(String), id_manter UInt64, n_repeticoes UInt64, b UInt8) "
            "ENGINE = MergeTree PARTITION BY b ORDER BY (h_feat, label_multi)")
_u_grp = [{"chave": str(j), "particao": str(j), "esperado": None, "sql": (
    f"INSERT INTO prep_grp_treino SELECT h_feat, count(), uniqExact(lm), uniqExact(label_binary), groupUniqArray(lm), toUInt8({j}) "
    f"FROM (SELECT h_feat, label_binary, {LM} AS lm FROM prep_hash WHERE conjunto = 'treino' AND {c}) GROUP BY h_feat @@S@@")} for c, j in FH]
_u_keep = [{"chave": str(j), "particao": str(j), "esperado": None, "sql": (
    f"INSERT INTO prep_keep_treino SELECT h_feat, lm, min(id_linha), count(), toUInt8({j}) "
    f"FROM (SELECT id_linha, h_feat, {LM} AS lm FROM prep_hash WHERE conjunto = 'treino' AND {c}) GROUP BY h_feat, lm @@S@@")} for c, j in FH]

# validadores: cada linha do treino precisa estar contada exatamente uma vez
construir_por_unidades("prep_grp_treino", DDL_GRP, _u_grp, SIG_DEDUP, lambda: _soma("prep_grp_treino", "n_linhas") == N_TREINO)
construir_por_unidades("prep_keep_treino", DDL_KEEP, _u_keep, SIG_DEDUP, lambda: _soma("prep_keep_treino", "n_repeticoes") == N_TREINO)

n_unicos = n_linhas("prep_keep_treino"); n_grupos = n_linhas("prep_grp_treino")
print(f"TREINO: {N_TREINO:,} linhas -> {n_unicos:,} únicas (features+rótulo) | repetidas removidas: {N_TREINO - n_unicos:,} ({100 * (N_TREINO - n_unicos) / N_TREINO:.2f}%) | grupos de features: {n_grupos:,}")


# reconciliação com a EDA (por tabela, repetições de features+rótulo, nível C)
def _calc_rec():
    linhas = []
    for o in ORIGENS:        # uniqExact por origem (o global não caberia na memória do servidor)
        r = rodar_pesado(f"""SELECT origem_tabela, conjunto, count() AS registros, count() - uniqExact(h_feat, {LM}) AS repetidas_aqui
                             FROM prep_hash WHERE origem_tabela = {lit(o)} GROUP BY origem_tabela, conjunto @@S@@""")
        linhas += r.to_dict("records")
    return linhas


_rec, _ = etapa("b4_reconciliacao_eda", SIG_DEDUP, _calc_rec)
rec = pd.DataFrame(_rec)
rec["repetidas_eda"] = rec.origem_tabela.map(lambda t: EDA_DUP_C[t][1])
rec["diferenca"] = rec.repetidas_aqui - rec.repetidas_eda
display(rec)
# Separar explicitamente o número histórico da EDA dos resultados medidos pelo pipeline atual.
EDA_DUP_TOTAL = 59_927_555
EDA_BASE_TOTAL = 70_427_637
EDA_DUP_PCT = 100 * EDA_DUP_TOTAL / EDA_BASE_TOTAL
repetidas_treino_pipeline = N_TREINO - n_unicos
pct_treino_pipeline = 100 * repetidas_treino_pipeline / N_TREINO if N_TREINO else 0.0
print("\n== EDA histórica (referência; não é o resultado da deduplicação do treino) ==")
print(f"EDA consolidada: {EDA_DUP_TOTAL:,} repetidas de {EDA_BASE_TOTAL:,} registros ({EDA_DUP_PCT:.2f}%). O percentual antigo de 67,10% não é utilizado.")
print("== Resultado efetivo do pipeline ==")
print(f"Treino: {N_TREINO:,} linhas antes -> {n_unicos:,} após deduplicação global (features + rótulo); removidas {repetidas_treino_pipeline:,} ({pct_treino_pipeline:.2f}% do treino).")
print(f"Teste: {N_TESTE:,} linhas oficiais; nenhuma linha é removida por deduplicação.")
print(f"Auditoria por tabela — soma das contagens recalculadas nesta etapa: {int(rec.repetidas_aqui.sum()):,}; a comparação com os números da EDA está na tabela acima.")
print("Observação: o número efetivo do treino é calculado globalmente e pode incluir repetições entre tabelas de treino; não deve ser confundido com o percentual agregado da EDA.")
RELATORIO["duplicatas_eda_vs_pipeline"] = {"eda_repetidas_referencia": EDA_DUP_TOTAL, "eda_total_referencia": EDA_BASE_TOTAL,
    "eda_percentual_referencia": EDA_DUP_PCT, "treino_antes": N_TREINO, "treino_depois": n_unicos,
    "treino_removidas_efetivas": repetidas_treino_pipeline, "treino_percentual_removido": pct_treino_pipeline,
    "teste_registros_preservados": N_TESTE, "teste_deduplicado": False}

print("\n== Conflitos de rótulo no treino (mesmas features, rótulos diferentes) ==")
conf = consultar_num("SELECT n_rotulos, n_binarios, count() AS grupos, sum(n_linhas) AS linhas FROM prep_grp_treino WHERE n_rotulos > 1 GROUP BY n_rotulos, n_binarios ORDER BY n_rotulos")
display(conf)
top = consultar_num("""SELECT arrayStringConcat(arraySort(rotulos), ' | ') AS rotulos_txt, count() AS grupos, sum(n_linhas) AS linhas
                       FROM prep_grp_treino WHERE n_rotulos > 1 GROUP BY rotulos_txt ORDER BY linhas DESC LIMIT 25""")
display(top); top.to_csv(ARTEFATOS / "conflitos_rotulo_treino_top.csv", index=False)
print("Política: nada é corrigido nem descartado automaticamente. Os grupos ficam marcados na base final (conflito_rotulo / conflito_binario) "
      "para uma análise de sensibilidade (treinar com e sem eles). Investigue os pares mais frequentes acima antes de qualquer correção.")
RELATORIO["duplicatas_treino"] = {"antes": N_TREINO, "depois": n_unicos, "removidas": N_TREINO - n_unicos}
RELATORIO["conflitos_treino"] = conf.to_dict("records")


  = `prep_grp_treino` já está pronta e válida; não será refeita.
  = `prep_keep_treino` já está pronta e válida; não será refeita.
TREINO: 50,063,112 linhas -> 8,257,343 únicas (features+rótulo) | repetidas removidas: 41,805,769 (83.51%) | grupos de features: 8,041,864
  = etapa 'b4_reconciliacao_eda' já concluída em 2026-10-14T21:58:58; resultado reaproveitado (RECRIAR.add('b4_reconciliacao_eda') para refazer).


,origem_tabela,conjunto,registros,repetidas_aqui,repetidas_eda,diferenca
0,data2_ldap,teste,2113234,2089294,2089294,0
1,data2_mssql,teste,5775786,5501042,5501042,0
2,data2_netbios,teste,3455899,3444572,3444572,0
3,data2_portmap,teste,191694,185603,185603,0
4,data2_syn,teste,4320541,3840360,3840360,0
5,data2_udp,teste,3782206,2484589,2484521,68
6,data2_udplag,teste,725165,585344,585344,0
7,data_drdos_dns,treino,5074413,4958078,4958078,0
8,data_drdos_ldap,treino,2181542,2150031,2150031,0
9,data_drdos_mssql,treino,4524498,4315609,4315609,0



== EDA histórica (referência; não é o resultado da deduplicação do treino) ==
EDA consolidada: 59,927,555 repetidas de 70,427,637 registros (85.09%). O percentual antigo de 67,10% não é utilizado.
== Resultado efetivo do pipeline ==
Treino: 50,063,112 linhas antes -> 8,257,343 após deduplicação global (features + rótulo); removidas 41,805,769 (83.51% do treino).
Teste: 20,364,525 linhas oficiais; nenhuma linha é removida por deduplicação.
Auditoria por tabela — soma das contagens recalculadas nesta etapa: 59,934,887; a comparação com os números da EDA está na tabela acima.
Observação: o número efetivo do treino é calculado globalmente e pode incluir repetições entre tabelas de treino; não deve ser confundido com o percentual agregado da EDA.

== Conflitos de rótulo no treino (mesmas features, rótulos diferentes) ==


,n_rotulos,n_binarios,grupos,linhas
0,2,1,170440,4147225
1,2,2,9,279
2,3,2,8,159
3,3,1,15938,7188383
4,4,2,3,128
5,4,1,2366,5326733
6,5,1,1159,3778950
7,6,2,1,58
8,6,1,166,6716265
9,7,1,41,1736075


,rotulos_txt,grupos,linhas
0,DrDoS_DNS | DrDoS_MSSQL | DrDoS_NetBIOS | DrDoS_SSDP | DrDoS_UDP | TFTP,11,5446114
1,DrDoS_DNS | DrDoS_LDAP | DrDoS_MSSQL | DrDoS_NetBIOS | DrDoS_SNMP | DrDoS_SSDP | DrDoS_UDP | TFTP,8,2662961
2,DrDoS_DNS | DrDoS_LDAP | DrDoS_MSSQL | DrDoS_NetBIOS | DrDoS_SNMP | DrDoS_SSDP | DrDoS_UDP | TFTP | UDPLag,8,1770822
3,Syn | TFTP | UDPLag,1266,1763568
4,DrDoS_DNS | DrDoS_MSSQL | DrDoS_NetBIOS,2371,1616109
5,DrDoS_DNS | DrDoS_NetBIOS | DrDoS_SNMP | DrDoS_SSDP,15,1516239
6,DrDoS_DNS | DrDoS_NetBIOS | DrDoS_SNMP,170,1513175
7,DrDoS_DNS | DrDoS_LDAP | DrDoS_MSSQL | DrDoS_NetBIOS | DrDoS_SNMP | DrDoS_SSDP | DrDoS_UDP,9,1325391
8,DrDoS_SSDP | DrDoS_UDP,136809,1212660
9,DrDoS_DNS | DrDoS_LDAP | DrDoS_MSSQL | DrDoS_NetBIOS | DrDoS_SNMP | DrDoS_SSDP | DrDoS_UDP | Syn | TFTP | UDPLag,6,1205340


Política: nada é corrigido nem descartado automaticamente. Os grupos ficam marcados na base final (conflito_rotulo / conflito_binario) para uma análise de sensibilidade (treinar com e sem eles). Investigue os pares mais frequentes acima antes de qualquer correção.


### Exemplo (etapa 4 — deduplicação do treino): antes → depois
**Antes:** várias linhas com **as mesmas features e o mesmo rótulo**, mas identificadores diferentes (`unnamed_0`, `flow_id`, `timestamp`). Para o modelo, são a mesma observação repetida.
**Depois:** **uma** linha em `prep_final_treino`, com `n_repeticoes = N` dizendo quantas existiam.
Real: 59.927.555 de 70.427.637 linhas eram repetidas em features+rótulo (85,09%). A deduplicação é **só no treino**: o teste fica intacto.
A célula abaixo pega **o grupo real com mais repetições** e mostra suas linhas.

In [29]:
def exemplo_dedup():
    top = consultar("SELECT toString(id_manter) AS id_manter, label_multi, toString(n_repeticoes) AS n FROM prep_keep_treino ORDER BY n_repeticoes DESC LIMIT 1")
    if not len(top):
        print("Sem dados de deduplicação."); return
    idm, n = top["id_manter"][0], int(top["n"][0])
    ids = consultar(f"SELECT toString(id_linha) AS id_linha FROM prep_hash WHERE conjunto = 'treino' AND h_feat = "
                    f"(SELECT h_feat FROM prep_keep_treino WHERE id_manter = {idm} LIMIT 1) LIMIT 4")["id_linha"].tolist()
    lista = ", ".join(ids)
    feats = [c for c in ["flow_duration", "total_fwd_packets", "total_backward_packets", "protocol"] if c in FEATURES_BASE][:3]
    sel = ", ".join(f"toString({q(c)}) AS {q(c)}" for c in feats)
    stg = consultar(f"SELECT toString(id_linha) AS id_linha, origem_tabela, label_original, {sel} FROM prep_stg WHERE id_linha IN ({lista})")
    aux = consultar(f"SELECT toString(id_linha) AS id_linha, unnamed_0, flow_id, timestamp_txt FROM prep_aux_ids WHERE id_linha IN ({lista}) AND origem_tabela IN "
                    f"({', '.join(lit(x) for x in stg['origem_tabela'].unique())})")
    antes = stg.merge(aux, on="id_linha")
    antes["fica?"] = np.where(antes["id_linha"] == idm, "SIM (mantida)", "removida")
    print(f"ANTES: este grupo tem {n:,} linhas com as MESMAS features e rótulo '{top['label_multi'][0]}' (primeiras {len(antes)} abaixo; só os identificadores diferem):")
    display(antes)
    print(f"DEPOIS: 1 linha em prep_final_treino (id_linha = {idm}) com n_repeticoes = {n:,}.")

In [30]:
try:
    exemplo_dedup()
except Exception as _e:
    print('(exemplo ao vivo indisponível agora:', str(_e)[:250], ')')

ANTES: este grupo tem 3,710,350 linhas com as MESMAS features e rótulo 'TFTP' (primeiras 4 abaixo; só os identificadores diferem):


,id_linha,origem_tabela,label_original,flow_duration,total_fwd_packets,total_backward_packets,unnamed_0,flow_id,timestamp_txt,fica?
0,6179430434439951186,data_drdos_mssql,DrDoS_MSSQL,1,2,0,138228,172.16.0.5-192.168.50.1-49343-62680-17,2018-12-01 11:45:27.939831,removida
1,7630859575681144498,data_drdos_netbios,DrDoS_NetBIOS,1,2,0,58347,172.16.0.5-192.168.50.1-55579-51605-17,2018-12-01 11:47:12.153594,removida
2,9761912515278305876,data_drdos_netbios,DrDoS_NetBIOS,1,2,0,140845,172.16.0.5-192.168.50.1-61162-11951-17,2018-12-01 11:47:13.092811,removida
3,11039403798939274957,data_drdos_netbios,DrDoS_NetBIOS,1,2,0,50746,172.16.0.5-192.168.50.1-39662-27176-17,2018-12-01 11:47:09.412575,removida


DEPOIS: 1 linha em prep_final_treino (id_linha = 4490035471702) com n_repeticoes = 3,710,350.


### 4.3 Investigação dos conflitos de rótulo (benigno × ataque com features idênticas)
Cruza os grupos conflitantes do treino com a tabela auxiliar (IPs, portas, timestamp) **somente para diagnóstico** — esses campos continuam fora da matriz de features. Serve para decidir se o conflito parece (a) fluxos legítimos distintos que viraram estatísticas idênticas ao remover os identificadores, (b) erro de rotulagem ou (c) arredondamento. Nada é corrigido ou descartado automaticamente.


In [31]:
def _calc_conflitos():
    d = rodar_pesado("""
        SELECT h.label_original AS rotulo, h.origem_tabela AS tabela, count() AS linhas,
               uniqExact(a.source_ip) AS ips_origem, uniqExact(a.destination_port) AS portas_destino,
               topK(3)(a.destination_port) AS top_portas_destino,
               toString(min(a.timestamp_dt)) AS primeiro, toString(max(a.timestamp_dt)) AS ultimo
        FROM prep_aux_ids AS a
        INNER JOIN (SELECT id_linha, label_original, origem_tabela FROM prep_hash
                    WHERE conjunto = 'treino' AND h_feat IN (SELECT h_feat FROM prep_grp_treino WHERE n_binarios > 1)) AS h
        ON a.id_linha = h.id_linha
        GROUP BY rotulo, tabela ORDER BY linhas DESC LIMIT 30 @@S@@""")
    return d.to_dict("records")


try:
    _conf_inv, _ = etapa("b4_conflitos_investigacao", assinatura(SIG_DEDUP, "conflitos"), _calc_conflitos)
    display(pd.DataFrame(_conf_inv))
    print("Leitura: se os conflitos se concentram em poucas portas/IPs e em janelas de tempo curtas, a causa provável é perda de informação ao remover os "
          "identificadores (fluxos distintos com as mesmas estatísticas). Sem evidência clara de erro de rotulagem, os grupos permanecem marcados "
          "(conflito_rotulo / conflito_binario) para análise de sensibilidade.")
    RELATORIO["conflitos_investigacao"] = _conf_inv
except (PermissionError, ErroConexao):
    raise
except Exception as _e:
    print(f"⚠ Investigação opcional dos conflitos não pôde ser concluída ({str(_e)[:200]}). O pipeline não depende dela; segue normalmente.")


  = etapa 'b4_conflitos_investigacao' já concluída em 2026-10-14T22:02:16; resultado reaproveitado (RECRIAR.add('b4_conflitos_investigacao') para refazer).


,rotulo,tabela,linhas,ips_origem,portas_destino,top_portas_destino,primeiro,ultimo
0,DrDoS_DNS,data_drdos_dns,362,2,356,"[24526, 51483, 13369]",2018-12-01 10:58:33.193434,2018-12-01 11:18:13.136368
1,BENIGN,data_tftp,67,6,6,"[443, 53, 61473]",2018-12-01 13:38:18.174286,2018-12-01 16:45:05.036801
2,DrDoS_UDP,data_drdos_udp,48,2,3,"[137, 49153, 22]",2018-12-01 12:40:39.635769,2018-12-01 13:03:40.110381
3,BENIGN,data_drdos_mssql,31,3,1,[443],2018-12-01 11:32:51.117483,2018-12-01 11:45:46.191731
4,BENIGN,data_drdos_ntp,29,11,13,"[443, 50453, 59017]",2018-12-01 09:17:12.634569,2018-12-01 10:49:06.236251
5,DrDoS_SNMP,data_drdos_snmp,23,2,12,"[162, 137, 57176]",2018-12-01 12:06:40.703016,2018-12-01 12:23:11.113546
6,DrDoS_LDAP,data_drdos_ldap,12,1,12,"[8884, 11479, 31333]",2018-12-01 11:23:23.821642,2018-12-01 11:32:32.186851
7,TFTP,data_tftp,10,2,8,"[51220, 137, 41992]",2018-12-01 13:34:37.807535,2018-12-01 17:06:13.429294
8,BENIGN,data_drdos_ldap,7,5,6,"[443, 57198, 58628]",2018-12-01 11:30:25.398181,2018-12-01 11:32:19.391037
9,BENIGN,data_drdos_dns,7,4,4,"[443, 52057, 55380]",2018-12-01 11:10:41.416479,2018-12-01 11:20:09.317389


Leitura: se os conflitos se concentram em poucas portas/IPs e em janelas de tempo curtas, a causa provável é perda de informação ao remover os identificadores (fluxos distintos com as mesmas estatísticas). Sem evidência clara de erro de rotulagem, os grupos permanecem marcados (conflito_rotulo / conflito_binario) para análise de sensibilidade.


### Exemplo (etapa 6 — conflitos de rótulo): antes → depois
**Antes:** as **mesmas features** aparecem com rótulos **diferentes** (ex.: `BENIGN` e `Syn`). Não dá para saber, só olhando as features, qual rótulo está certo.
**Depois:** as linhas **não são apagadas**: ficam marcadas com `conflito_rotulo = 1` (rótulos multiclasse diferentes) e `conflito_binario = 1` (BENIGN × ataque). Você pode excluí-las depois, como análise de sensibilidade.
A célula abaixo mostra **um grupo real** em conflito, se houver.

In [32]:
def exemplo_conflito():
    r = consultar("SELECT toString(id_linha) AS id_linha, label_original FROM prep_hash WHERE conjunto = 'treino' AND h_feat IN "
                  "(SELECT h_feat FROM prep_grp_treino WHERE n_rotulos > 1 LIMIT 1) ORDER BY label_original LIMIT 1 BY label_original LIMIT 6")
    if not len(r):
        print("Nenhum conflito de rótulo no treino."); return
    lista = ", ".join(r["id_linha"])
    feats = [c for c in ["flow_duration", "total_fwd_packets", "total_backward_packets"] if c in FEATURES_BASE][:3]
    sel = ", ".join(f"toString({q(c)}) AS {q(c)}" for c in feats)
    stg = consultar(f"SELECT toString(id_linha) AS id_linha, origem_tabela, label_original, {sel} FROM prep_stg WHERE id_linha IN ({lista})")
    print("Mesmas features, rótulos diferentes (uma linha por rótulo):"); display(stg)
    print("Em prep_final_treino essas linhas ficam com conflito_rotulo = 1 (e conflito_binario = 1 se misturam BENIGN e ataque).")

In [33]:
try:
    exemplo_conflito()
except Exception as _e:
    print('(exemplo ao vivo indisponível agora:', str(_e)[:250], ')')

Mesmas features, rótulos diferentes (uma linha por rótulo):


,id_linha,origem_tabela,label_original,flow_duration,total_fwd_packets,total_backward_packets
0,1350549836328046577,data_drdos_udp,DrDoS_UDP,108168,4,0
1,12343339093886538454,data_drdos_ssdp,DrDoS_SSDP,108168,4,0


Em prep_final_treino essas linhas ficam com conflito_rotulo = 1 (e conflito_binario = 1 se misturam BENIGN e ataque).


### 5.2 Resumo de classes (treino bruto, treino deduplicado, teste) e o caso `WebDDoS`
Nada é excluído automaticamente. A tabela mostra se cada classe tem treino suficiente e representação no teste, para a decisão posterior (`CLASSES_FORA_MULTI`).

In [34]:
exigir("prep_keep_treino", "prep_grp_treino")
garantir_mapa()
LMk = expr_label_multi("label_original")
bruto = consultar_num(f"SELECT {LMk} AS label_multi, conjunto, count() AS n FROM prep_stg GROUP BY label_multi, conjunto")
ded = consultar_num("SELECT label_multi, count() AS n_treino_dedup, sum(n_repeticoes) AS n_treino_bruto FROM prep_keep_treino GROUP BY label_multi")
cl = bruto.pivot_table(index="label_multi", columns="conjunto", values="n", aggfunc="sum", fill_value=0).reset_index()
cl = cl.merge(ded, on="label_multi", how="left").fillna(0)
cl["so_em"] = np.where((cl.treino > 0) & (cl.teste == 0), "só treino", np.where((cl.treino == 0) & (cl.teste > 0), "só teste", "ambos"))
cl["alerta"] = np.where(cl.so_em != "ambos", "sem avaliação/treino correspondente", np.where(cl.n_treino_dedup < 1000, "poucas amostras únicas no treino", ""))
cl = cl[["label_multi", "treino", "n_treino_dedup", "teste", "so_em", "alerta"]].sort_values("treino", ascending=False).reset_index(drop=True)
for c in ["treino", "n_treino_dedup", "teste"]: cl[c] = cl[c].astype(int)
display(cl)


def _acoes_classes(S):
    executar("CREATE TABLE prep_classes_resumo (label_multi String, treino UInt64, n_treino_dedup UInt64, teste UInt64, so_em String, alerta String) ENGINE = MergeTree ORDER BY label_multi")
    inserir_df("prep_classes_resumo", cl)


def _v_classes():
    t = consultar_num("SELECT label_multi, treino, n_treino_dedup, teste FROM prep_classes_resumo").sort_values("label_multi").reset_index(drop=True)
    c = cl[["label_multi", "treino", "n_treino_dedup", "teste"]].sort_values("label_multi").reset_index(drop=True)
    return len(t) == len(c) and (t.astype({"treino": int, "n_treino_dedup": int, "teste": int}).values.tolist() == c.values.tolist())


construir("prep_classes_resumo", _acoes_classes, assinatura(SIG_DEDUP, "classes", cl.values.tolist()), _v_classes)
wd = cl[cl.label_multi == "WebDDoS"]
print("\nWebDDoS:", wd.to_dict("records"))
print("Leitura: sem exemplos no teste oficial e com poucas amostras únicas no treino, a classe não é avaliável no multiclasse principal. "
      "Documente a limitação e, se decidir, use CLASSES_FORA_MULTI = ['WebDDoS'] (afeta só a view multiclasse; os dados continuam na base).")
RELATORIO["classes"] = cl.to_dict("records")


,label_multi,treino,n_treino_dedup,teste,so_em,alerta
0,TFTP,20082580,4396919,0,só treino,sem avaliação/treino correspondente
1,DrDoS_SNMP,5159870,115316,0,só treino,sem avaliação/treino correspondente
2,DrDoS_DNS,5071011,113259,0,só treino,sem avaliação/treino correspondente
3,DrDoS_MSSQL,4522492,207003,0,só treino,sem avaliação/treino correspondente
4,DrDoS_NetBIOS,4093279,19490,0,só treino,sem avaliação/treino correspondente
5,DrDoS_UDP,3134645,1075337,0,só treino,sem avaliação/treino correspondente
6,DrDoS_SSDP,2610611,891221,0,só treino,sem avaliação/treino correspondente
7,DrDoS_LDAP,2179930,30099,0,só treino,sem avaliação/treino correspondente
8,Syn,1582289,155515,4891500,ambos,
9,DrDoS_NTP,1202642,1112936,0,só treino,sem avaliação/treino correspondente


  = `prep_classes_resumo` já está pronta e válida; não será refeita.

WebDDoS: [{'label_multi': 'WebDDoS', 'treino': 439, 'n_treino_dedup': 414, 'teste': 0, 'so_em': 'só treino', 'alerta': 'sem avaliação/treino correspondente'}]
Leitura: sem exemplos no teste oficial e com poucas amostras únicas no treino, a classe não é avaliável no multiclasse principal. Documente a limitação e, se decidir, use CLASSES_FORA_MULTI = ['WebDDoS'] (afeta só a view multiclasse; os dados continuam na base).


## 6. Bases finais (treino deduplicado e teste oficial) e views binária / multiclasse

- `prep_final_treino`: uma linha por (features, rótulo) + `n_repeticoes`, `conflito_rotulo`, `conflito_binario`.
- `prep_final_teste`: **todas** as linhas oficiais + `sobrepoe_treino`, `sobrepoe_treino_mesmo_rotulo`.
- Mesmas colunas de features nos dois (matriz base), mesmos tipos. Nenhum identificador na matriz. Valores ausentes continuam `NULL` (a imputação usa parâmetros do treino, bloco 8); colunas com ausentes ganham `<coluna>_ausente`.
- As duas views (binária e multiclasse) leem a **mesma** tabela: nada é binarizado de forma irreversível.

In [35]:
restaurar_selecao(); garantir_mapa()
base_cols = [f"{q(c)} {tipo_ch(c, anulavel=(c in COLS_NULL))}" for c in FEATURES_BASE]
ind_cols = [f"{q(c + '_ausente')} UInt8" for c in COLS_NULL_BASE]
meta = ["id_linha UInt64", "origem_tabela LowCardinality(String)", "label_original LowCardinality(String)", "label_multi LowCardinality(String)", "label_binary UInt8"]
extra_tr = ["n_repeticoes UInt32", "conflito_rotulo UInt8", "conflito_binario UInt8"]
extra_te = ["sobrepoe_treino UInt8", "sobrepoe_treino_mesmo_rotulo UInt8"]
feat_sel = lambda: [(f"s.{q(c)}" if c in COLS_NULL else f"assumeNotNull(s.{q(c)})") + f" AS {q(c)}" for c in FEATURES_BASE]
ind_sel = lambda: [f"toUInt8(isNull(s.{q(c)})) AS {q(c + '_ausente')}" for c in COLS_NULL_BASE]
nomes_ins = lambda extra: [m.split()[0] for m in meta + extra] + [q(c) for c in FEATURES_BASE] + [q(c + "_ausente") for c in COLS_NULL_BASE]
cols_esperadas = lambda extra: {m.split()[0] for m in meta + extra} | set(FEATURES_BASE) | {c + "_ausente" for c in COLS_NULL_BASE}
SIG_FINAL = assinatura(SIG_DEDUP, SIG_FEATS, MAPA, COLS_NULL, "final3", LINHAS_POR_UNIDADE_FINAL, FAIXAS_MIN_TREINO)
exigir("prep_keep_treino", "prep_grp_treino", "prep_hash", "prep_mapa_rotulos")
TREINO = [t for t in ORIGENS if conjunto_de(t) == "treino"]
TESTE = [t for t in ORIGENS if conjunto_de(t) == "teste"]
cols_esperadas = lambda extra: {m.split()[0] for m in meta + extra} | set(FEATURES_BASE) | {c + "_ausente" for c in COLS_NULL_BASE} | {"b"}
_cols_def = lambda extra: ", ".join(meta + extra + base_cols + ind_cols + ["b UInt8"])
DDL_FT = f"CREATE TABLE prep_final_treino ({_cols_def(extra_tr)}) ENGINE = MergeTree PARTITION BY (origem_tabela, b) ORDER BY id_linha"
DDL_FE = f"CREATE TABLE prep_final_teste ({_cols_def(extra_te)}) ENGINE = MergeTree PARTITION BY (origem_tabela, b) ORDER BY id_linha"
LMs = expr_label_multi('s.label_original')
_sel = ", ".join(feat_sel() + ind_sel())


def _sql_ft(o, j, lo, hi):
    cond = f"origem_tabela = {lit(o)} AND {cond_id('id_linha', lo, hi)}"; ck = cond_id("id_manter", lo, hi)
    return (f"INSERT INTO prep_final_treino ({', '.join(nomes_ins(extra_tr))}, b) "
            f"SELECT s.id_linha, s.origem_tabela, s.label_original, k.label_multi, s.label_binary, k.n_repeticoes, toUInt8(g.n_rotulos > 1), toUInt8(g.n_binarios > 1), "
            f"{_sel}, toUInt8({j}) FROM (SELECT * FROM prep_stg WHERE {cond}) AS s "
            f"INNER JOIN (SELECT h_feat, label_multi, id_manter, n_repeticoes FROM prep_keep_treino WHERE {ck}) AS k ON s.id_linha = k.id_manter "
            f"INNER JOIN (SELECT h_feat, n_rotulos, n_binarios FROM prep_grp_treino WHERE h_feat IN (SELECT h_feat FROM prep_keep_treino WHERE {ck})) AS g "
            f"ON k.h_feat = g.h_feat @@S@@")


def _sql_fe(o, j, lo, hi):
    cond = f"origem_tabela = {lit(o)} AND {cond_id('id_linha', lo, hi)}"
    return (f"INSERT INTO prep_final_teste ({', '.join(nomes_ins(extra_te))}, b) "
            f"SELECT s.id_linha, s.origem_tabela, s.label_original, {LMs}, s.label_binary, toUInt8(notEmpty(g.rotulos)), toUInt8(has(g.rotulos, {LMs})), "
            f"{_sel}, toUInt8({j}) FROM (SELECT * FROM prep_stg WHERE {cond}) AS s "
            f"INNER JOIN (SELECT id_linha, h_feat FROM prep_hash WHERE {cond}) AS h ON s.id_linha = h.id_linha "
            f"LEFT JOIN (SELECT h_feat, rotulos FROM prep_grp_treino WHERE h_feat IN (SELECT h_feat FROM prep_hash WHERE {cond})) AS g "
            f"ON h.h_feat = g.h_feat @@S@@")


VARIANTES_MAGRAS = ["SETTINGS max_threads = 1, max_block_size = 4096", "SETTINGS max_threads = 1, max_block_size = 2048"]
_u_ft, _u_fe = [], []
for _o in TREINO:
    for _j, (_lo, _hi) in enumerate(faixas_id(cont[_o], LINHAS_POR_UNIDADE_FINAL, FAIXAS_MIN_TREINO)):
        _u_ft.append({"chave": f"{_o}#{_j}", "particao": f"({lit(_o)}, {_j})", "sql": _sql_ft(_o, _j, _lo, _hi), "esperado": None, "variantes": VARIANTES_MAGRAS})
for _o in TESTE:
    for _j, (_lo, _hi) in enumerate(faixas_id(cont[_o], LINHAS_POR_UNIDADE_FINAL)):
        _cond = f"origem_tabela = {lit(_o)} AND {cond_id('id_linha', _lo, _hi)}"
        _u_fe.append({"chave": f"{_o}#{_j}", "particao": f"({lit(_o)}, {_j})", "sql": _sql_fe(_o, _j, _lo, _hi), "esperado": n_linhas("prep_stg", _cond), "variantes": VARIANTES_MAGRAS,
                      "contar": (lambda o=_o, j=_j: n_linhas("prep_final_teste", f"origem_tabela = {lit(o)} AND b = {j}"))})

construir_por_unidades("prep_final_treino", DDL_FT, _u_ft, SIG_FINAL,
                       lambda: n_linhas("prep_final_treino") == n_linhas("prep_keep_treino") and _colunas_de("prep_final_treino") == cols_esperadas(extra_tr))
construir_por_unidades("prep_final_teste", DDL_FE, _u_fe, SIG_FINAL,
                       lambda: n_linhas("prep_final_teste") == N_TESTE and _colunas_de("prep_final_teste") == cols_esperadas(extra_te))

n_unicos = n_linhas("prep_keep_treino")
n_ft, n_fe = n_linhas("prep_final_treino"), n_linhas("prep_final_teste")
print(f"prep_final_treino: {n_ft:,} (esperado {n_unicos:,}) | prep_final_teste: {n_fe:,} (esperado {N_TESTE:,})")
assert n_ft == n_unicos and n_fe == N_TESTE

# views binária e multiclasse (mesma tabela, alvo diferente) — baratas, sempre recriadas
feats_v = ", ".join([q(c) for c in FEATURES_BASE] + [q(c + "_ausente") for c in COLS_NULL_BASE])
fora = ", ".join(lit(c) for c in CLASSES_FORA_MULTI)
filtro_multi = f" WHERE label_multi NOT IN ({fora})" if CLASSES_FORA_MULTI else ""
for conj, tab in [("treino", "prep_final_treino"), ("teste", "prep_final_teste")]:
    executar(f"DROP VIEW IF EXISTS `prep_vw_{conj}_binario`")
    executar(f"CREATE VIEW prep_vw_{conj}_binario AS SELECT id_linha, label_binary AS y, {feats_v} FROM {tab}")
    executar(f"DROP VIEW IF EXISTS `prep_vw_{conj}_multi`")
    executar(f"CREATE VIEW prep_vw_{conj}_multi AS SELECT id_linha, label_multi AS y, {feats_v} FROM {tab}{filtro_multi}")
print("Views criadas: prep_vw_{treino,teste}_{binario,multi}. Classes fora do multiclasse:", CLASSES_FORA_MULTI or "nenhuma (decisão posterior)")
RELATORIO["final"] = {"treino": n_ft, "teste": n_fe, "features": len(FEATURES_BASE), "indicadores": COLS_NULL_BASE}


  conexão instável (Exception: HTTP 500 [código ClickHouse: 241] corpo='{\n\t"meta":\n\t[\n\t\t{\n\t\t\t"name": "n",\n\t\t\t"type": "String"\n\t\t}\n\t],\n\n\t"data":\n\t[\n\n\t],\n\n\t"rows"); nova tentativa em 5s...
  = `prep_final_treino` já está pronta e válida; não será refeita.
  = `prep_final_teste` já está pronta e válida; não será refeita.
prep_final_treino: 8,257,343 (esperado 8,257,343) | prep_final_teste: 20,364,525 (esperado 20,364,525)
Views criadas: prep_vw_{treino,teste}_{binario,multi}. Classes fora do multiclasse: nenhuma (decisão posterior)


### Exemplo (resultado das etapas 1–7): uma linha das bases finais
**Antes:** as tabelas originais (88 colunas de texto). **Depois:** `prep_final_treino` e `prep_final_teste`: as colunas de **metadados** (`id_linha`, `origem_tabela`, `label_original`, `label_multi`, `label_binary`, marcas de repetição/conflito/sobreposição), as **features numéricas** de `FEATURES_BASE` (NULL onde havia ∞/vazio/texto) e as colunas `<feature>_ausente` (1 = estava ausente).
Abaixo, **uma linha real** de cada base (primeiras colunas).

In [36]:
def exemplo_linha_final():
    for t in ("prep_final_treino", "prep_final_teste"):
        r = consultar(f"SELECT * FROM {t} LIMIT 1")
        print(f"{t}: 1 linha real, {r.shape[1]} colunas (primeiras 24):"); display(r.T.rename(columns={0: "valor"}).head(24))

In [37]:
try:
    exemplo_linha_final()
except Exception as _e:
    print('(exemplo ao vivo indisponível agora:', str(_e)[:250], ')')

prep_final_treino: 1 linha real, 72 colunas (primeiras 24):


,valor
id_linha,9223614819634979104
origem_tabela,data_udplag
label_original,UDP-lag
label_multi,UDPLag
label_binary,1
n_repeticoes,1
conflito_rotulo,1
conflito_binario,0
protocol,17
flow_duration,108538


prep_final_teste: 1 linha real, 71 colunas (primeiras 24):


,valor
id_linha,13835061114718178508
origem_tabela,data2_mssql
label_original,MSSQL
label_multi,MSSQL
label_binary,1
sobrepoe_treino,0
sobrepoe_treino_mesmo_rotulo,0
protocol,17
flow_duration,2
total_fwd_packets,2


## 7. Redundância por correlação e verificação de AUC (decide **só o treino**)

- A EDA calculou correlação com treino + teste juntos. Aqui recalculamos **no treino deduplicado** (amostra determinística de 1/`AMOSTRA_MOD`).
- Correlação alta **não** remove nada automaticamente da base: gera uma lista `FEATURES_REDUZIDO` (uma feature de cada grupo redundante, a mais "redundante" sai) para um experimento comparativo, especialmente para o SVM. A base completa continua disponível.
- AUC isolado > 0,90 (`min_packet_length`, `fwd_packet_length_min`, `inbound`) não prova vazamento; verificamos se o efeito se mantém no teste oficial. Isso é diagnóstico, não critério de remoção.

In [38]:
def roc_auc_score(y, x):
    """AUC (Mann-Whitney por postos), igual ao do scikit-learn para alvo binário; evita depender do scikit-learn."""
    d = pd.DataFrame({"y": np.asarray(y), "x": np.asarray(x, dtype="float64")}).dropna()
    n1, n0 = int((d.y == 1).sum()), int((d.y == 0).sum())
    if n1 == 0 or n0 == 0:
        return np.nan
    r = d.x.rank(method="average")
    return float((r[d.y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


exigir("prep_final_treino", "prep_final_teste")
restaurar_selecao(); garantir_mapa()
cols_q = ", ".join(q(c) for c in FEATURES_BASE)


def _calc_corr_auc():
    amo = consultar_num(f"SELECT label_binary, {cols_q} FROM prep_final_treino WHERE cityHash64(id_linha) % {AMOSTRA_MOD} = 0")
    C = amo[FEATURES_BASE].corr(method="pearson").abs()
    pares = [(a, b, float(C.loc[a, b])) for i, a in enumerate(FEATURES_BASE) for b in FEATURES_BASE[i + 1:] if C.loc[a, b] > LIMIAR_CORR]
    pares_df = pd.DataFrame(pares, columns=["feature_a", "feature_b", "abs_r_treino"]).sort_values("abs_r_treino", ascending=False).reset_index(drop=True)
    eda_set = {frozenset((a, b)) for a, b, r in PARES_EDA}
    pares_df["na_eda"] = [frozenset((a, b)) in eda_set for a, b in zip(pares_df.feature_a, pares_df.feature_b)]
    # redução gulosa: em cada par ainda ativo sai a feature com maior correlação média com as demais
    media_corr = C.mean(); removidas_corr = {}
    for a, b, r in pares:
        if a in removidas_corr or b in removidas_corr: continue
        sai, fica = (a, b) if media_corr[a] >= media_corr[b] else (b, a)
        removidas_corr[sai] = f"|r|={r:.3f} com {fica} no treino (maior correlação média)"
    tst = consultar_num(f"SELECT label_binary, {cols_q} FROM prep_final_teste WHERE cityHash64(id_linha) % {AMOSTRA_MOD} = 0")

    def auc_por_feature(d):
        out = {}
        for c in FEATURES_BASE:
            x = d[c].fillna(d[c].median()); out[c] = roc_auc_score(d["label_binary"], x) if d["label_binary"].nunique() == 2 else np.nan
        return pd.Series(out)

    auc = pd.DataFrame({"auc_treino": auc_por_feature(amo), "auc_teste": auc_por_feature(tst)})
    auc["forca_treino"] = (auc.auc_treino - 0.5).abs(); auc["forca_teste"] = (auc.auc_teste - 0.5).abs()
    return {"n_amostra": int(len(amo)), "pares": pares_df.to_dict("records"), "removidas_corr": removidas_corr,
            "auc": auc.reset_index().rename(columns={"index": "feature"}).to_dict("records")}


_sig_b7 = assinatura(SIG_FEATS, MAPA, COLS_NULL, LIMIAR_CORR, AMOSTRA_MOD, n_linhas("prep_final_treino"))
_res7, _ = etapa("b7_correlacao_auc", _sig_b7, _calc_corr_auc)
pares_df = pd.DataFrame(_res7["pares"]); removidas_corr = _res7["removidas_corr"]
auc = pd.DataFrame(_res7["auc"]).set_index("feature")
print(f"Amostra do treino deduplicado: {_res7['n_amostra']:,} linhas")
display(pares_df)
print(f"{len(pares_df)} pares com |r| > {LIMIAR_CORR} no treino deduplicado (EDA, treino+teste: 47; alguns pares da EDA já saíram por constância/duplicidade exata).")
pares_df.to_csv(ARTEFATOS / "pares_correlacionados_treino.csv", index=False)

FEATURES_REDUZIDO = [c for c in FEATURES_BASE if c not in removidas_corr]
print(f"FEATURES_REDUZIDO: {len(FEATURES_REDUZIDO)} (base {len(FEATURES_BASE)}). Saem (candidatas): {list(removidas_corr)}")
salvar_selecao(FEATURES_REDUZIDO, removidas_corr)
RELATORIO["reduzido"] = {"mantidas": FEATURES_REDUZIDO, "candidatas_removidas": removidas_corr}

display(auc.sort_values("forca_treino", ascending=False).head(12).round(4))
for c in ["min_packet_length", "fwd_packet_length_min", "inbound"]:
    if c in auc.index: print(f"{c}: AUC treino {auc.loc[c, 'auc_treino']:.3f} | teste {auc.loc[c, 'auc_teste']:.3f}")
    else: print(f"{c}: não está na matriz base (removida como duplicata/constante)")
print("Leitura: AUC alto que cai (ou inverte) no teste indica dependência do período/captura; AUC alto que persiste indica separação real. "
      "Verifique também, na documentação do CICFlowMeter, se essas features usam o rótulo (não usam, mas confirme).")
auc.round(5).to_csv(ARTEFATOS / "auc_treino_x_teste.csv")


  = etapa 'b7_correlacao_auc' já concluída em 2026-10-17T00:52:41; resultado reaproveitado (RECRIAR.add('b7_correlacao_auc') para refazer).
Amostra do treino deduplicado: 166,089 linhas


,feature_a,feature_b,abs_r_treino,na_eda
0,bwd_packet_length_mean,avg_bwd_segment_size,1.000000,True
1,fwd_packet_length_mean,avg_fwd_segment_size,1.000000,True
2,flow_packets_s,fwd_packets_s,0.999849,True
3,flow_iat_min,fwd_iat_min,0.999767,True
4,flow_duration,fwd_iat_total,0.999754,True
5,fwd_packet_length_min,min_packet_length,0.999692,True
6,total_length_of_fwd_packets,act_data_pkt_fwd,0.999065,True
7,flow_iat_max,fwd_iat_max,0.998083,True
8,packet_length_mean,avg_fwd_segment_size,0.996536,True
9,fwd_packet_length_mean,packet_length_mean,0.996536,True


43 pares com |r| > 0.95 no treino deduplicado (EDA, treino+teste: 47; alguns pares da EDA já saíram por constância/duplicidade exata).
FEATURES_REDUZIDO: 40 (base 61). Saem (candidatas): ['flow_duration', 'total_backward_packets', 'act_data_pkt_fwd', 'fwd_packet_length_mean', 'packet_length_mean', 'fwd_packet_length_max', 'min_packet_length', 'fwd_packet_length_min', 'bwd_packet_length_mean', 'flow_packets_s', 'flow_iat_mean', 'flow_iat_max', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max', 'flow_iat_min', 'bwd_iat_std', 'avg_fwd_segment_size', 'active_mean', 'idle_max', 'idle_mean']


,auc_treino,auc_teste,forca_treino,forca_teste
feature,,,,
min_packet_length,0.9786,0.8913,0.4786,0.3913
fwd_packet_length_min,0.9770,0.8902,0.4770,0.3902
avg_fwd_segment_size,0.9688,0.8151,0.4688,0.3151
fwd_packet_length_mean,0.9688,0.8151,0.4688,0.3151
average_packet_size,0.9383,0.7982,0.4383,0.2982
packet_length_mean,0.9339,0.7879,0.4339,0.2879
bwd_packets_s,0.0677,0.2046,0.4323,0.2954
total_backward_packets,0.0718,0.1803,0.4282,0.3197
inbound,0.9202,0.9093,0.4202,0.4093


min_packet_length: AUC treino 0.979 | teste 0.891
fwd_packet_length_min: AUC treino 0.977 | teste 0.890
inbound: AUC treino 0.920 | teste 0.909
Leitura: AUC alto que cai (ou inverte) no teste indica dependência do período/captura; AUC alto que persiste indica separação real. Verifique também, na documentação do CICFlowMeter, se essas features usam o rótulo (não usam, mas confirme).


## 8. Parâmetros aprendidos **só no treino** + pré-processador reutilizável (RF e SVM)

- **Mediana** (imputação) das colunas com ausentes: calculada na base de treino deduplicada, nunca no teste.
- **Média e desvio-padrão** (z-score) de todas as features da base, calculados no treino **depois** da imputação, desvio populacional (como o `StandardScaler`). Desvio zero vira 1.
- Os parâmetros são calculados por agregação **sobre `prep_final_treino`** (somente treino, depois da separação), o que é equivalente a ajustar o escalador no treino e evita carregar dezenas de milhões de linhas na memória. Ficam em `prep_parametros` e em JSON versionado (com hash SHA-256).
- **Random Forest:** imputação (sem z-score). **SVM:** imputação + z-score. O mesmo teste oficial para ambos. O balanceamento (Hybrid Sampling) fica para a etapa seguinte e deve ser aplicado **somente** ao treino, depois destes parâmetros.

In [39]:
restaurar_selecao(); garantir_mapa()


def _calc_params():
    med = {}
    if COLS_NULL_BASE:
        r = consultar_num("SELECT " + ", ".join(f"quantileExactInclusive(0.5)({q(c)}) AS {q(c)}" for c in COLS_NULL_BASE) + " FROM prep_final_treino").iloc[0]
        med = {c: float(r[c]) for c in COLS_NULL_BASE}
    def expr_imp(c): return f"ifNull({q(c)}, {med[c]!r})" if c in med else q(c)
    agg = ", ".join(f"avg({expr_imp(c)}) AS {q('m_' + c)}, stddevPopStable({expr_imp(c)}) AS {q('s_' + c)}, countIf(isNull({q(c)})) AS {q('a_' + c)}" if c in COLS_NULL_BASE
                    else f"avg({q(c)}) AS {q('m_' + c)}, stddevPopStable({q(c)}) AS {q('s_' + c)}, toUInt64(0) AS {q('a_' + c)}" for c in FEATURES_BASE)
    st = consultar_num(f"SELECT count() AS n, {agg} FROM prep_final_treino").iloc[0]
    linhas = []
    for c in FEATURES_BASE:
        sd = float(st[f"s_{c}"]); linhas.append({"coluna": c, "mediana": med.get(c), "media": float(st[f"m_{c}"]), "desvio": sd,
                                               "escala": sd if sd > 0 else 1.0, "ausentes_treino": int(st[f"a_{c}"]), "usa_indicador": int(c in COLS_NULL_BASE)})
    return {"versao": datetime.now().strftime("%Y%m%d_%H%M%S"), "n": int(st["n"]), "medianas": med, "params": linhas}


exigir("prep_final_treino", "prep_final_teste")
_sig_params = assinatura(SIG_FEATS, MAPA, COLS_NULL, FEATURES_REDUZIDO, n_linhas("prep_final_treino"), "mediana_inclusiva")
_resp, _reutil = etapa("b8_parametros", _sig_params, _calc_params)
PARAMS = pd.DataFrame(_resp["params"]); med = _resp["medianas"]; versao = _resp["versao"]
assert not PARAMS[["media", "desvio"]].isna().any().any(), "parâmetro NaN: investigue colunas com variância nula/ausente"
display(PARAMS.round(4).head(15)); print(f"... {len(PARAMS)} features | linhas de treino usadas: {_resp['n']:,}")
print("Desvio zero (escala=1):", PARAMS.loc[PARAMS.desvio <= 0, "coluna"].tolist() or "nenhuma")

caminho_params = ARTEFATOS / f"prep_params_{versao}.json"
_tab_ok = existe("prep_parametros") and n_linhas("prep_parametros", f"versao = {lit(versao)}") == len(PARAMS)
if _reutil and _tab_ok and caminho_params.exists() and (ARTEFATOS / "prep_params_ULTIMO.json").exists():
    print(f"= parâmetros da versão {versao} já gravados (tabela prep_parametros + JSON); nada a refazer.")
else:
    artefato = {"versao": versao, "ajustado_em": "treino (prep_final_treino, deduplicado)", "linhas_treino": int(_resp["n"]),
                "features": FEATURES_BASE, "features_reduzido": FEATURES_REDUZIDO, "colunas_com_ausentes": COLS_NULL_BASE, "mapa_rotulos": MAPA,
                "parametros": PARAMS.to_dict("records")}
    corpo = json.dumps(artefato, sort_keys=True, ensure_ascii=False, default=_jd)
    artefato["sha256"] = hashlib.sha256(corpo.encode("utf-8")).hexdigest()
    caminho_params.write_text(json.dumps(artefato, indent=1, ensure_ascii=False, default=_jd), encoding="utf-8")
    (ARTEFATOS / "prep_params_ULTIMO.json").write_text(json.dumps(artefato, indent=1, ensure_ascii=False, default=_jd), encoding="utf-8")
    executar("DROP TABLE IF EXISTS `prep_parametros`")
    executar("CREATE TABLE prep_parametros (versao String, coluna String, mediana Nullable(Float64), media Float64, desvio Float64, escala Float64, ausentes_treino UInt64, usa_indicador UInt8) ENGINE = MergeTree ORDER BY coluna")
    inserir_df("prep_parametros", PARAMS.assign(versao=versao)[["versao", "coluna", "mediana", "media", "desvio", "escala", "ausentes_treino", "usa_indicador"]])
    print(f"✔ Parâmetros salvos: {caminho_params.name} (sha256 {artefato['sha256'][:12]}...) e tabela prep_parametros")
RELATORIO["parametros"] = {"versao": versao, "medianas": med}


  = etapa 'b8_parametros' já concluída em 2026-10-17T00:52:50; resultado reaproveitado (RECRIAR.add('b8_parametros') para refazer).


,coluna,mediana,media,desvio,escala,ausentes_treino,usa_indicador
0,protocol,NaN,1.663000e+01,1.988800e+00,1.988800e+00,0,0
1,flow_duration,NaN,4.540482e+06,1.281078e+07,1.281078e+07,0,0
2,total_fwd_packets,NaN,1.835900e+01,6.016669e+02,6.016669e+02,0,0
3,total_backward_packets,NaN,8.930000e-02,3.673200e+00,3.673200e+00,0,0
4,total_length_of_fwd_packets,NaN,6.379800e+03,1.572545e+04,1.572545e+04,0,0
5,total_length_of_bwd_packets,NaN,3.613260e+01,9.826648e+03,9.826648e+03,0,0
6,fwd_packet_length_max,NaN,4.792402e+02,1.817958e+02,1.817958e+02,0,0
7,fwd_packet_length_min,NaN,4.515357e+02,1.988079e+02,1.988079e+02,0,0
8,fwd_packet_length_mean,NaN,4.687621e+02,1.852447e+02,1.852447e+02,0,0
9,fwd_packet_length_std,NaN,1.126710e+01,2.412760e+01,2.412760e+01,0,0


... 61 features | linhas de treino usadas: 8,257,343
Desvio zero (escala=1): nenhuma
= parâmetros da versão 20261017_005250 já gravados (tabela prep_parametros + JSON); nada a refazer.


In [40]:
class PreProc:
    """Aplica os parâmetros do TREINO a qualquer DataFrame (treino, teste, amostras). RF: imputa. SVM: imputa + z-score."""
    def __init__(self, artefato):
        p = pd.DataFrame(artefato["parametros"]).set_index("coluna")
        self.features, self.reduzido = artefato["features"], artefato["features_reduzido"]
        self.com_ausentes = artefato["colunas_com_ausentes"]
        self.mediana, self.media, self.escala = p["mediana"].to_dict(), p["media"], p["escala"]
        self.versao, self.sha256 = artefato["versao"], artefato.get("sha256")

    @classmethod
    def carregar(cls, caminho=None):
        return cls(json.loads(Path(caminho or ARTEFATOS / "prep_params_ULTIMO.json").read_text(encoding="utf-8")))

    def transformar(self, df, modelo="rf", usar_indicadores=True, features=None):
        feats = list(features or self.features)
        X = df[feats].astype("float64").replace([np.inf, -np.inf], np.nan)
        ind = {f"{c}_ausente": X[c].isna().astype("uint8") for c in self.com_ausentes if c in feats}
        for c in self.com_ausentes:
            if c in feats: X[c] = X[c].fillna(self.mediana[c])
        assert not X.isna().any().any(), "ainda há NaN: coluna sem mediana do treino?"
        if modelo == "svm":
            X = (X - self.media[feats]) / self.escala[feats]
        elif modelo != "rf":
            raise ValueError("modelo deve ser 'rf' ou 'svm'")
        return pd.concat([X, pd.DataFrame(ind, index=X.index)], axis=1) if usar_indicadores and ind else X

def ler_base(conjunto, tarefa="binario", amostra_mod=None, limite=None):
    """Lê uma view do ClickHouse (conjunto: 'treino'|'teste'; tarefa: 'binario'|'multi'). Para o treino completo prefira ler em lotes."""
    onde = f" WHERE cityHash64(id_linha) % {int(amostra_mod)} = 0" if amostra_mod else ""
    return consultar_num(f"SELECT * FROM prep_vw_{conjunto}_{tarefa}{onde}" + (f" LIMIT {int(limite)}" if limite else ""))

# demonstração/sanidade: parâmetros do treino aplicados a amostras de treino e de teste
pp = PreProc.carregar()
a_tr, a_te = ler_base("treino", "binario", AMOSTRA_MOD), ler_base("teste", "binario", AMOSTRA_MOD)
Xtr, Xte = pp.transformar(a_tr, "svm"), pp.transformar(a_te, "svm")
print(f"SVM  treino: média {Xtr[pp.features].mean().abs().mean():.3f} | desvio {Xtr[pp.features].std().mean():.3f}  (esperado ~0 e ~1)")
print(f"SVM  teste : média {Xte[pp.features].mean().abs().mean():.3f} | desvio {Xte[pp.features].std().mean():.3f}  (pode diferir: usa parâmetros do treino)")
print("RF   sem NaN:", not pp.transformar(a_te, "rf").isna().any().any(), "| SVM sem NaN:", not Xte.isna().any().any())

SVM  treino: média 0.003 | desvio 1.138  (esperado ~0 e ~1)
SVM  teste : média 0.348 | desvio 1.116  (pode diferir: usa parâmetros do treino)
RF   sem NaN: True | SVM sem NaN: True


### Exemplo (etapa 8 — imputação e z-score): antes → depois
**Antes:** a feature pode estar `NULL` (era ∞, vazio ou texto). **Depois:**
- **Random Forest:** `NULL` → **mediana do treino** (sem escala).
- **SVM:** `NULL` → mediana do treino e depois `z = (x − μ_treino) / σ_treino`.
μ, σ e a mediana vêm **só do treino** e são reaproveitados no teste (ficam em `prep_parametros` e no JSON versionado). Abaixo, um exemplo com **os seus parâmetros reais**.

In [41]:
def exemplo_parametros():
    P = PARAMS.set_index("coluna")
    cand = [c for c in COLS_NULL_BASE if c in P.index] or [FEATURES_BASE[0]]
    f = cand[0]
    med, mu, sd = float(P.loc[f, "mediana"]), float(P.loc[f, "media"]), float(P.loc[f, "desvio"])
    sd = sd if sd > 0 else 1.0
    r = consultar(f"SELECT toString({q(f)}) AS x FROM prep_final_treino WHERE isNotNull({q(f)}) LIMIT 1")
    x = float(r["x"][0])
    print(f"Feature: {f}   |   parâmetros do TREINO: mediana = {med:.6g}, média μ = {mu:.6g}, desvio σ = {sd:.6g}")
    print(f"\n1) Valor presente x = {x:.6g}")
    print(f"   Random Forest : {x:.6g}   (sem escala)")
    print(f"   SVM (z-score) : ({x:.6g} - {mu:.6g}) / {sd:.6g} = {(x - mu) / sd:.4f}")
    if f in COLS_NULL_BASE:
        print(f"\n2) Valor ausente (NULL: era ∞, vazio ou texto na tabela original)")
        print(f"   Random Forest : NULL -> mediana do treino = {med:.6g}")
        print(f"   SVM (z-score) : NULL -> {med:.6g} -> ({med:.6g} - {mu:.6g}) / {sd:.6g} = {(med - mu) / sd:.4f}")

In [42]:
try:
    exemplo_parametros()
except Exception as _e:
    print('(exemplo ao vivo indisponível agora:', str(_e)[:250], ')')

Feature: flow_bytes_s   |   parâmetros do TREINO: mediana = 689.591, média μ = 5.31489e+07, desvio σ = 2.93725e+08

1) Valor presente x = 6.21e+08
   Random Forest : 6.21e+08   (sem escala)
   SVM (z-score) : (6.21e+08 - 5.31489e+07) / 2.93725e+08 = 1.9333

2) Valor ausente (NULL: era ∞, vazio ou texto na tabela original)
   Random Forest : NULL -> mediana do treino = 689.591
   SVM (z-score) : NULL -> 689.591 -> (689.591 - 5.31489e+07) / 2.93725e+08 = -0.1809


## 9. Validação final das bases e relatório

In [43]:
restaurar_selecao()
exigir("prep_final_treino", "prep_final_teste", "prep_hash", "prep_keep_treino", "prep_grp_treino")
if "aud_const" not in globals(): aud_const = pd.DataFrame(estado_get("etapa:b3_constantes")["res"])
if "aud_dup" not in globals(): aud_dup = pd.DataFrame(estado_get("etapa:b3_duplicadas")["res"])
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)

# 1) originais intactas
for t in ORIGENS: assert n_linhas(t) == ESPERADO_REGISTROS[t]
verifica("Tabelas originais intactas (18 contagens iguais às da EDA)", True)

# 2) contagens
verifica("Treino final = linhas únicas (features+rótulo) do treino", n_linhas("prep_final_treino") == n_linhas("prep_keep_treino"), f"({n_linhas('prep_final_treino'):,})")
verifica("Teste final = teste oficial completo (sem deduplicar)", n_linhas("prep_final_teste") == N_TESTE, f"({N_TESTE:,})")
verifica("Soma de n_repeticoes = treino bruto (nenhuma linha do treino ficou sem contar)", _soma("prep_final_treino", "n_repeticoes") == N_TREINO)

# 3) rótulos
for tab in ["prep_final_treino", "prep_final_teste"]:
    r = consultar_num(f"SELECT countIf(label_binary != toUInt8(label_original != 'BENIGN')) AS bin_incons, countIf(label_multi = '') AS multi_vazio FROM {tab}").iloc[0]
    verifica(f"{tab}: label_binary coerente com label_original e label_multi preenchido", r.bin_incons == 0 and r.multi_vazio == 0)
dist = consultar_num("SELECT 'treino' AS conjunto, label_multi, count() AS n FROM prep_final_treino GROUP BY label_multi UNION ALL SELECT 'teste', label_multi, count() FROM prep_final_teste GROUP BY label_multi")
tabela_dist = dist.pivot_table(index="label_multi", columns="conjunto", values="n", fill_value=0)
tabela_dist["%treino"] = (100 * tabela_dist["treino"] / tabela_dist["treino"].sum()).round(2); tabela_dist["%teste"] = (100 * tabela_dist["teste"] / tabela_dist["teste"].sum()).round(2)
display(tabela_dist)

# 4) inválidos remanescentes: nenhum Inf/NaN; ausentes só nas colunas previstas
for tab in ["prep_final_treino", "prep_final_teste"]:
    agg = ", ".join(f"countIf(isInfinite({q(c)}) OR isNaN({q(c)})) AS {q(c)}" for c in FEATURES_BASE)
    inv = consultar_num(f"SELECT {agg} FROM {tab}").iloc[0]
    nul = consultar_num("SELECT " + ", ".join(f"countIf(isNull({q(c)})) AS {q(c)}" for c in COLS_NULL_BASE) + f" FROM {tab}").iloc[0] if COLS_NULL_BASE else pd.Series(dtype=float)
    verifica(f"{tab}: nenhum Inf/NaN", inv.sum() == 0)
    print("    ausentes (NULL) restantes, a imputar com a mediana do treino:", {k: int(v) for k, v in nul.items()})

# 5) duplicatas no treino final
cols_h = ", ".join([(f"ifNull({q(c)}, 0)" if c in COLS_NULL_BASE else q(c)) for c in FEATURES_BASE] + [f"isNull({q(c)})" for c in COLS_NULL_BASE])
try:
    d = rodar_pesado(f"SELECT count() AS n, uniqExact(sipHash128({cols_h}, label_multi)) AS unicas FROM prep_final_treino @@S@@").iloc[0]
    _nota = ""
except Exception as _e:
    if not (isinstance(_e, ErroConexao) or any(k in str(_e) for k in CODIGOS_RECUPERAVEIS)):
        raise
    # plano B (menos memória): hash de 64 bits; chance de falso "duplicado" ~ n^2 / 2^65 (~2e-6 para 8 milhões de linhas)
    print("  (memória insuficiente com hash de 128 bits; repetindo a checagem com hash de 64 bits)")
    d = rodar_pesado(f"SELECT count() AS n, uniqExact(sipHash64({cols_h}, label_multi)) AS unicas FROM prep_final_treino @@S@@").iloc[0]
    _nota = " [hash de 64 bits]"
verifica("Treino final sem duplicatas (features+rótulo)", d.n == d.unicas, f"({int(d.n):,} linhas, {int(d.unicas):,} únicas){_nota}")

# 5b) features que ficaram constantes depois da deduplicação (informativo)
mm = consultar_num("SELECT " + ", ".join(f"min({q(c)}) AS mn_{i}, max({q(c)}) AS mx_{i}" for i, c in enumerate(FEATURES_BASE)) + " FROM prep_final_treino").iloc[0]
const_pos = [c for i, c in enumerate(FEATURES_BASE) if mm[f"mn_{i}"] == mm[f"mx_{i}"]]
print("   features constantes no treino final deduplicado (informativo):", const_pos or "nenhuma")

# 6) seleção de features registrada e baseada no treino
sel = consultar_num("SELECT * FROM prep_selecao_features")
verifica("Tabela prep_selecao_features contém as 80 features originais", len(sel) == len(FEATURES_ORDEM) and set(sel.coluna) == set(FEATURES_ORDEM), f"({len(sel)} linhas)")
verifica("Seleção principal coincide com FEATURES_BASE", set(sel.loc[sel.em_base == 1, "coluna"]) == set(FEATURES_BASE))
verifica("As 13 features reportadas pela EDA foram auditadas individualmente", set(CONSTANTES_EDA) <= set(aud_const.coluna) and len(CONSTANTES_EDA) == 13)
verifica("Remoção de features duplicadas foi decidida pelo treino", all(x["duplicada_no_treino"] for x in aud_dup[aud_dup.candidata_a_remover.isin(DUPLICADAS)].to_dict("records")))

# 7) consistência de colunas entre treino e teste
colunas = consultar("SELECT `table` AS tabela, name AS coluna, type AS tipo FROM system.columns WHERE database = currentDatabase() AND `table` IN ('prep_final_treino', 'prep_final_teste')")
t1 = colunas[colunas.tabela == "prep_final_treino"].set_index("coluna")["tipo"]; t2 = colunas[colunas.tabela == "prep_final_teste"].set_index("coluna")["tipo"]
comuns = [c for c in t1.index if c in t2.index]
verifica("Features e tipos idênticos entre treino e teste", all(t1[c] == t2[c] for c in comuns) and set(FEATURES_BASE) <= set(comuns))
extras = {"id_linha", "origem_tabela", "label_original", "label_multi", "label_binary", "n_repeticoes", "conflito_rotulo", "conflito_binario", "sobrepoe_treino", "sobrepoe_treino_mesmo_rotulo", "b"}
feats_final = [c for c in t1.index if c not in extras]
verifica("Nenhum identificador/rótulo bruto na matriz de features", not (set(ID_COLS) | {"label"}) & set(feats_final) and all(not c.startswith("p_") for c in feats_final))
print("   features na matriz:", len(FEATURES_BASE), "+ indicadores:", len(COLS_NULL_BASE))

# 8) sobreposição e conflitos (informativo)
s = consultar_num("SELECT count() AS n, sum(sobrepoe_treino) AS sobrepoe, sum(sobrepoe_treino_mesmo_rotulo) AS sobrepoe_mesmo_rotulo FROM prep_final_teste").iloc[0]
c = consultar_num("SELECT sum(conflito_rotulo) AS conflito_rotulo, sum(conflito_binario) AS conflito_binario FROM prep_final_treino").iloc[0]
print(f"   teste com features iguais às do treino: {int(s.sobrepoe):,} ({100 * s.sobrepoe / s.n:.2f}%), com o mesmo rótulo: {int(s.sobrepoe_mesmo_rotulo):,}")
print(f"   treino com conflito de rótulo: {int(c.conflito_rotulo):,} linhas | conflito benigno x ataque: {int(c.conflito_binario):,}")
RELATORIO["validacao"] = {"checagens_ok": int(sum(ok)), "checagens_total": len(ok), "teste_sobrepoe_treino": int(s.sobrepoe), "treino_conflito_rotulo": int(c.conflito_rotulo)}
assert all(ok), "há checagens reprovadas acima"
(ARTEFATOS / "relatorio_preprocessamento.json").write_text(json.dumps(RELATORIO, indent=1, ensure_ascii=False, default=_jd), encoding="utf-8")
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas. Relatório: {ARTEFATOS / 'relatorio_preprocessamento.json'}")
display(painel_status())


✔ Tabelas originais intactas (18 contagens iguais às da EDA) 
✔ Treino final = linhas únicas (features+rótulo) do treino (8,257,343)
✔ Teste final = teste oficial completo (sem deduplicar) (20,364,525)
✔ Soma de n_repeticoes = treino bruto (nenhuma linha do treino ficou sem contar) 
✔ prep_final_treino: label_binary coerente com label_original e label_multi preenchido 
✔ prep_final_teste: label_binary coerente com label_original e label_multi preenchido 


conjunto,teste,treino,%treino,%teste
label_multi,,,,
BENIGN,56965.0,50768.0,0.61,0.28
DrDoS_DNS,0.0,113259.0,1.37,0.00
DrDoS_LDAP,0.0,30099.0,0.36,0.00
DrDoS_MSSQL,0.0,207003.0,2.51,0.00
DrDoS_NTP,0.0,1112936.0,13.48,0.00
DrDoS_NetBIOS,0.0,19490.0,0.24,0.00
DrDoS_SNMP,0.0,115316.0,1.40,0.00
DrDoS_SSDP,0.0,891221.0,10.79,0.00
DrDoS_UDP,0.0,1075337.0,13.02,0.00


  conexão instável (Exception: HTTP 500 [código ClickHouse: 241] corpo='{\n\t"meta":\n\t[\n\t\t{\n\t\t\t"name": "protocol",\n\t\t\t"type": "UInt64"\n\t\t},\n\t\t{\n\t\t\t"name": "flow_duratio); nova tentativa em 5s...
  conexão instável (Exception: HTTP 500 [código ClickHouse: 241] corpo='{\n\t"meta":\n\t[\n\t\t{\n\t\t\t"name": "protocol",\n\t\t\t"type": "UInt64"\n\t\t},\n\t\t{\n\t\t\t"name": "flow_duratio); nova tentativa em 15s...
  conexão instável (Exception: HTTP 500 [código ClickHouse: 241] corpo='{\n\t"meta":\n\t[\n\t\t{\n\t\t\t"name": "protocol",\n\t\t\t"type": "UInt64"\n\t\t},\n\t\t{\n\t\t\t"name": "flow_duratio); nova tentativa em 45s...
✔ prep_final_treino: nenhum Inf/NaN 
    ausentes (NULL) restantes, a imputar com a mediana do treino: {'flow_bytes_s': 24915, 'flow_packets_s': 24915}
✔ prep_final_teste: nenhum Inf/NaN 
    ausentes (NULL) restantes, a imputar com a mediana do treino: {'flow_bytes_s': 807514, 'flow_packets_s': 807514}
  conexão instável (Exception: HTTP 500

,tabela,motor,linhas,tamanho,marca_pronta
0,prep_aud_invalidos,MergeTree,144.0,14.46 KiB,sim
1,prep_aux_ids,MergeTree,70427637.0,3.32 GiB,sim
2,prep_classes_resumo,MergeTree,18.0,1.14 KiB,sim
3,prep_estado,ReplacingMergeTree,535.0,30.65 KiB,não
4,prep_final_teste,MergeTree,20364525.0,1.28 GiB,sim
5,prep_final_treino,MergeTree,8257343.0,800.20 MiB,sim
6,prep_grp_treino,MergeTree,8041864.0,149.28 MiB,sim
7,prep_hash,MergeTree,70427637.0,783.54 MiB,sim
8,prep_keep_treino,MergeTree,8257343.0,200.91 MiB,sim
9,prep_mapa_rotulos,MergeTree,19.0,864.00 B,sim


## 10. O que foi feito em cada caso

1. **Duplicatas de registros** — Removidas **somente no treino** (global entre as tabelas `data_*`), mantendo uma ocorrência por (features + rótulo), a de menor `id_linha`; `n_repeticoes` guarda quantas havia. O teste oficial nunca é deduplicado nem deduplicado em conjunto com o treino. A sobreposição treino×teste é medida (`sobrepoe_treino`, `sobrepoe_treino_mesmo_rotulo` em cada linha do teste) para um experimento extra sem sobreposição. A EDA (59.927.555 de 70.427.637; 85,09%) é só referência; o número efetivo é calculado pelo pipeline.
2. **Conflitos de rótulo** — Grupos com mesmas features e rótulos diferentes são contados (`prep_grp_treino`), marcados (`conflito_rotulo`, `conflito_binario`) e investigados (bloco 4.3). Nada é corrigido nem descartado automaticamente; a análise de sensibilidade (treinar com e sem eles) fica para os experimentos.
3. **Duplicatas de features (colunas)** — Pares exatamente iguais são verificados valor a valor **no treino**; só a segunda coluna do par sai. O teste é só auditoria (alerta se divergir).
4. **Constantes** — As 13 features da EDA são auditadas uma a uma em **todas** as tabelas de treino; são removidas as constantes/sem valor válido no treino (inclusive constantes novas, se houver). Decisões em `prep_selecao_features`, `prep_estado` e no relatório JSON.
5. **Nulos, NaN, infinitos e texto inválido** — Tipos numéricos explícitos (sem `CREATE TABLE ... AS` das originais). Vazio, texto não numérico, `NaN`, `+Inf` e `-Inf` viram `NULL` com o motivo em `*_status`; contagem por coluna, tabela, conjunto e classe; causa investigada (duração zero). Nenhum registro é removido por valor inválido; feature com >50% de ausentes no treino sai da matriz. Imputação pela **mediana do treino** (no `PreProc`), mais coluna `<feature>_ausente` opcional. Nada é trocado por zero nem pelo maior valor finito.
6. **Redundância por correlação** — |r| > 0,95 recalculado no treino deduplicado gera `FEATURES_REDUZIDO` para um experimento comparativo (especialmente SVM); a matriz principal não perde features por isso. O **desempenho** das duas matrizes deve ser comparado na etapa de modelagem.
7. **Identificadores** — `unnamed_0`, `flow_id`, IPs, portas, `timestamp` e o texto bruto de `simillarhttp` ficam só em `prep_aux_ids` (rastreabilidade via `id_linha`); fora da matriz de features.
8. **Rótulos** — `label_original` preservado; `label_binary` e `label_multi` coexistem na mesma base (views binária e multiclasse leem a mesma tabela). `UDP-lag` → `UDPLag` controlado por `UNIFICAR_UDPLAG` (com evidência exibida); demais unificações (`DrDoS_X` × `X`) só por decisão explícita em `MAPA_ROTULOS_EXTRA`. `WebDDoS` não é excluída automaticamente (`CLASSES_FORA_MULTI`).
9. **Escala** — Medianas, médias e desvios **só do treino**, em `prep_parametros` e JSON versionado com SHA-256. `PreProc.transformar(df, "rf")` imputa; `PreProc.transformar(df, "svm")` imputa e aplica z-score com os parâmetros do treino. O balanceamento (Hybrid Sampling) é aplicado depois e **só** ao treino.
10. **Treino/teste** — Divisão oficial preservada (`data_*` = treino, `data2_*` = teste); seleção de features, constantes e parâmetros decididos no treino e apenas aplicados ao teste. Não há variáveis categóricas na matriz (`protocol` é numérico).

**Reexecução:** basta rodar o notebook de cima para baixo. Tabelas e análises já concluídas e válidas são reaproveitadas; o painel (`painel_status()`) mostra o que está pronto. Para refazer algo, use `RECRIAR.add("nome")`.


**v8 — reexecução:** rode as células de cima para baixo. Etapas já concluídas e válidas são reaproveitadas em segundos (mensagens `= ... já está pronta`); etapas interrompidas continuam de onde pararam; o `RECRIAR` só é necessário para forçar um refazer.

## 11. Unificação de classes (o mesmo ataque com nomes diferentes)

**O problema.** O mesmo tipo de ataque aparece com nomes diferentes nos dois dias de captura. No treino se chama `DrDoS_MSSQL`; no teste, `MSSQL`. Se não forem juntados, o modelo nunca vê `MSSQL` no treino, e o teste multiclasse fica injusto por construção. **Para o binário isso não importa**: os dois são "ataque".

**A regra é simples:** une-se o rótulo do treino com o do teste quando o **nome é o mesmo sem o prefixo `DrDoS_`**. Exemplo: `DrDoS_MSSQL` (treino) + `MSSQL` (teste) → `MSSQL`.

**Por que isso é seguro: a documentação do dataset.** A Tabela III do artigo do CICDDoS2019 (Sharafaldin, Lashkari, Hakak e Ghorbani, 2019) lista os ataques executados em cada dia:

| Dia | Ataques |
|---|---|
| Treino (12/jan) | NTP, DNS, **LDAP, MSSQL, NetBIOS**, SNMP, SSDP, **UDP, UDP-Lag**, WebDDoS, **SYN**, TFTP |
| Teste (11/mar) | PortMap, **NetBIOS, LDAP, MSSQL, UDP, UDP-Lag, SYN** |

Os ataques em negrito foram executados **nos dois dias**, com o mesmo nome. O prefixo `DrDoS_` só marca os ataques de reflexão nas figuras e nos rótulos do dataset. Isso também confirma que `UDP-lag` (treino) e `UDPLag` (teste) são o mesmo ataque. Ficam sem par: `NTP`, `DNS`, `SNMP`, `SSDP`, `TFTP` e `WebDDoS` (só no treino) e `Portmap` (só no teste; o artigo chama de "PortScan" no texto e de "PortMap" na tabela e na figura da taxonomia, e diz que é desconhecido de propósito).

**Esta etapa não olha os dados do teste para decidir.** Decide só pelo nome e pela documentação. Uma conferência **opcional e informativa** (`CONFERIR_COM_DADOS = True`) compara as medianas de algumas colunas entre treino e teste de cada par. Ela **não muda nenhuma decisão**: serve para você ver se algo destoa muito. Esperamos alguma diferença natural, porque os dois dias têm máquinas e endereços diferentes (Tabela II do artigo).

**Remoção do prefixo `DrDoS_` (ligada por padrão).** Além de unir os pares, o notebook tira o prefixo de **todos** os rótulos do treino: `DrDoS_DNS` → `DNS`, `DrDoS_NTP` → `NTP`, `DrDoS_SNMP` → `SNMP`, `DrDoS_SSDP` → `SSDP`. Motivos:
- evita nomes misturados (`LDAP` ao lado de `DrDoS_DNS`) em tabelas, gráficos e matrizes de confusão;
- o prefixo **não** é um marcador confiável da família: na taxonomia do artigo, UDP flood é ataque de *exploração*, mas o rótulo do treino é `DrDoS_UDP`, enquanto `Syn` e `UDPLag` (também de exploração) não têm prefixo;
- as Tabelas III e IV do artigo usam os nomes sem prefixo.

Nomes finais: `BENIGN`, `DNS`, `LDAP`, `MSSQL`, `NetBIOS`, `NTP`, `Portmap`, `SNMP`, `SSDP`, `Syn`, `TFTP`, `UDP`, `UDPLag`, `WebDDoS`. **`label_original` continua com o nome original** (com o prefixo), então dá para voltar ao dado de origem. Um par que você mandar **não unificar** (`FORCAR_SEPARAR`) **mantém** o prefixo, para não ser unido por acaso ao rótulo do teste de mesmo nome. Para manter os prefixos em tudo, use `REMOVER_PREFIXO_DRDOS = False`.

**O que é criado (nada existente é alterado):**
- tabela `prep_mapa_rotulos_unif`: cada rótulo do treino ou do teste, o **rótulo final**, se consta na documentação e a decisão (`UNIFICAR`, `NÃO UNIFICAR`, `SEM PAR` ou `REMOVER PREFIXO`), para auditoria;
- views novas `prep_vw_treino_multi_unif` e `prep_vw_teste_multi_unif`, onde `y` já vem unificado. A troca é feita **na leitura**: as tabelas finais, a coluna `label_multi` e as views antigas **continuam iguais**, e `label_original` nunca muda.

**Para mudar uma decisão:** `FORCAR_SEPARAR = {"DrDoS_X"}` (não une) ou `FORCAR_UNIFICAR = {"DrDoS_X": "X"}` (une). Para o caso de a documentação listar mais um ataque nos dois dias, acrescente o nome em `DOCUMENTADAS_NOS_DOIS_DIAS`.

In [44]:
# ---------- 11.1 Candidatos: mesmo nome, sem o prefixo "DrDoS_" ----------
import re
restaurar_selecao()
FORCAR_UNIFICAR = globals().get("FORCAR_UNIFICAR", {})          # decisão manual, ex.: {"DrDoS_X": "X"}  (rótulo do treino -> rótulo do teste)
FORCAR_SEPARAR = globals().get("FORCAR_SEPARAR", set())         # decisão manual, ex.: {"DrDoS_X"}
CONFERIR_COM_DADOS = globals().get("CONFERIR_COM_DADOS", True)  # tabela informativa de medianas treino x teste (não decide nada)
REMOVER_PREFIXO_DRDOS = globals().get("REMOVER_PREFIXO_DRDOS", True)   # tira o prefixo "DrDoS_" de TODOS os rótulos do treino nas views unificadas (label_original não muda)
# Ataques executados nos DOIS dias, segundo a Tabela III do artigo do CICDDoS2019 (nomes sem símbolos e em minúsculas)
DOCUMENTADAS_NOS_DOIS_DIAS = globals().get("DOCUMENTADAS_NOS_DOIS_DIAS", {"ldap", "mssql", "netbios", "udp", "udplag", "syn"})

def _norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

rot_treino = consultar_num("SELECT label_multi AS rotulo, count() AS linhas_treino FROM prep_final_treino GROUP BY label_multi")
rot_teste = consultar_num("SELECT label_multi AS rotulo, count() AS linhas_teste FROM prep_final_teste GROUP BY label_multi")
vis = rot_treino.merge(rot_teste, on="rotulo", how="outer").fillna(0)
vis["onde existe"] = np.where((vis.linhas_treino > 0) & (vis.linhas_teste > 0), "treino e teste", np.where(vis.linhas_treino > 0, "só no treino", "só no teste"))
print("Rótulos (label_multi) nas bases finais:"); display(vis.sort_values(["onde existe", "rotulo"]).astype({"linhas_treino": int, "linhas_teste": int}).reset_index(drop=True))

so_treino, so_teste = sorted(set(rot_treino.rotulo) - set(rot_teste.rotulo)), sorted(set(rot_teste.rotulo) - set(rot_treino.rotulo))
CANDIDATOS = []
for y in so_teste:
    for x in so_treino:
        nx, ny = _norm(x), _norm(y)
        if nx == ny or nx == "drdos" + ny:                       # mesmo nome, com ou sem o prefixo DrDoS_
            CANDIDATOS.append((x, y))
print("\nPares com o mesmo nome (rótulo do treino -> rótulo do teste):", CANDIDATOS or "nenhum")
print("Rótulos só no teste SEM par (ficam como estão):", sorted(set(so_teste) - {y for _, y in CANDIDATOS}) or "nenhum")
print("Rótulos só no treino SEM par (ficam como estão):", sorted(set(so_treino) - {x for x, _ in CANDIDATOS}) or "nenhum")

Rótulos (label_multi) nas bases finais:


,rotulo,linhas_treino,linhas_teste,onde existe
0,LDAP,0,1915122,só no teste
1,MSSQL,0,5787453,só no teste
2,NetBIOS,0,3657497,só no teste
3,Portmap,0,186960,só no teste
4,UDP,0,3867155,só no teste
5,DrDoS_DNS,113259,0,só no treino
6,DrDoS_LDAP,30099,0,só no treino
7,DrDoS_MSSQL,207003,0,só no treino
8,DrDoS_NTP,1112936,0,só no treino
9,DrDoS_NetBIOS,19490,0,só no treino



Pares com o mesmo nome (rótulo do treino -> rótulo do teste): [('DrDoS_LDAP', 'LDAP'), ('DrDoS_MSSQL', 'MSSQL'), ('DrDoS_NetBIOS', 'NetBIOS'), ('DrDoS_UDP', 'UDP')]
Rótulos só no teste SEM par (ficam como estão): ['Portmap']
Rótulos só no treino SEM par (ficam como estão): ['DrDoS_DNS', 'DrDoS_NTP', 'DrDoS_SNMP', 'DrDoS_SSDP', 'TFTP', 'WebDDoS']


In [45]:
# ---------- 11.2 Decisão por nome + documentação (os dados do teste NÃO entram na decisão) ----------
cand_dict = {y: x for x, y in CANDIDATOS}
linhas = []
for y in sorted(set(rot_teste.rotulo) - set(rot_treino.rotulo)):
    x = cand_dict.get(y); doc = int(_norm(y) in DOCUMENTADAS_NOS_DOIS_DIAS)
    if x is None:
        dec, mot = "SEM PAR", "nenhum rótulo do treino tem o mesmo nome (sem o prefixo DrDoS_); fica só no teste"
    elif x in FORCAR_SEPARAR:
        dec, mot = "NÃO UNIFICAR (manual)", "decisão manual em FORCAR_SEPARAR"
    elif doc:
        dec, mot = "UNIFICAR", f"mesmo nome sem o prefixo DrDoS_, e '{y}' consta nos dois dias na Tabela III do artigo do dataset"
    else:
        dec, mot = "NÃO UNIFICAR", f"nome compatível, mas '{y}' não está na lista de ataques dos dois dias (DOCUMENTADAS_NOS_DOIS_DIAS); use FORCAR_UNIFICAR se tiver base para unir"
    linhas.append({"rotulo_treino": x or "", "rotulo_teste": y, "documentado_nos_dois_dias": doc, "decisao": dec, "motivo": mot})
for x, y in FORCAR_UNIFICAR.items():                             # decisões manuais prevalecem
    linhas = [l for l in linhas if l["rotulo_teste"] != y]
    linhas.append({"rotulo_treino": x, "rotulo_teste": y, "documentado_nos_dois_dias": int(_norm(y) in DOCUMENTADAS_NOS_DOIS_DIAS), "decisao": "UNIFICAR (manual)", "motivo": "decisão manual em FORCAR_UNIFICAR"})
tab_unif = pd.DataFrame(linhas, columns=["rotulo_treino", "rotulo_teste", "documentado_nos_dois_dias", "decisao", "motivo"])
MAPA_UNIF = {r.rotulo_treino: r.rotulo_teste for r in tab_unif.itertuples() if str(r.decisao).startswith("UNIFICAR") and r.rotulo_treino}
PREFIXO = "drdos_"
if REMOVER_PREFIXO_DRDOS:
    # Tira o prefixo "DrDoS_" dos demais rótulos do treino. Exceção: pares que você mandou NÃO unificar mantêm o prefixo,
    # senão o nome sem prefixo coincidiria com o do teste e eles seriam unidos por acaso.
    mantidos = {r.rotulo_treino for r in tab_unif.itertuples() if r.rotulo_treino and str(r.decisao).startswith("NÃO UNIFICAR")}
    existentes = set(rot_treino.rotulo) | set(rot_teste.rotulo); novos = []
    for x in sorted(rot_treino.rotulo):
        if str(x).lower().startswith(PREFIXO) and x not in MAPA_UNIF and x not in mantidos:
            novo = str(x)[len(PREFIXO):]
            if novo in existentes or novo in MAPA_UNIF.values():
                print(f"⚠ '{x}' mantém o prefixo: o nome '{novo}' já existe e a remoção uniria rótulos diferentes."); continue
            MAPA_UNIF[x] = novo; novos.append({"rotulo_treino": x, "rotulo_teste": "", "documentado_nos_dois_dias": 0, "decisao": "REMOVER PREFIXO",
                                                "motivo": "tem o prefixo DrDoS_ e não tem par no teste: só o prefixo é removido"})
    if novos: tab_unif = pd.concat([tab_unif, pd.DataFrame(novos)], ignore_index=True)
tab_unif["rotulo_final"] = [MAPA_UNIF.get(r.rotulo_treino, r.rotulo_treino) if r.rotulo_treino else r.rotulo_teste for r in tab_unif.itertuples()]
print("== Decisão por rótulo =="); display(tab_unif[["rotulo_treino", "rotulo_teste", "rotulo_final", "documentado_nos_dois_dias", "decisao", "motivo"]])
print("\nMAPA_UNIF (rótulo do treino -> rótulo final):", MAPA_UNIF or "nenhuma alteração")
if REMOVER_PREFIXO_DRDOS: print("Prefixo DrDoS_ removido dos rótulos do treino (label_original continua com o nome original). Para manter: REMOVER_PREFIXO_DRDOS = False.")

def conferencia_medianas():
    # INFORMATIVA: medianas de algumas colunas, no treino e no teste, para cada par. Não decide nada.
    cols = [c for c in ["flow_duration", "total_fwd_packets", "fwd_packet_length_max", "flow_iat_mean", "min_packet_length", "max_packet_length", "average_packet_size", "packet_length_std"] if c in FEATURES_BASE][:8]
    pares = [(r.rotulo_treino, r.rotulo_teste) for r in tab_unif.itertuples() if r.rotulo_treino and r.rotulo_teste and str(r.decisao).startswith("UNIFICAR")]
    if not pares or not cols: return None
    med = lambda tab, rots: consultar_num("SELECT label_multi AS r, " + ", ".join(f"quantile(0.5)({q(c)}) AS {q(c)}" for c in cols)
                                          + f" FROM {tab} WHERE label_multi IN ({', '.join(lit(r) for r in rots)}) GROUP BY label_multi").set_index("r")
    mt, me = med("prep_final_treino", {x for x, _ in pares}), med("prep_final_teste", {y for _, y in pares})
    out = []
    for x, y in pares:
        if x not in mt.index or y not in me.index: continue
        for c in cols:
            a, b = float(mt.loc[x, c]), float(me.loc[y, c]); r = (b + 1e-9) / (a + 1e-9) if (a or b) else 1.0
            out.append({"par": f"{x} → {y}", "coluna": c, "mediana_treino": round(a, 3), "mediana_teste": round(b, 3), "razao_teste_treino": round(r, 3)})
    d = pd.DataFrame(out); d["parecida"] = d.razao_teste_treino.between(0.5, 2.0)
    return d

if CONFERIR_COM_DADOS and MAPA_UNIF:
    cm = conferencia_medianas()
    if cm is not None and len(cm):
        print("\n== Conferência INFORMATIVA (não decide nada): medianas de colunas-chave, treino × teste ==")
        print("Razão perto de 1 = parecidas; muito diferente (10 vezes, por exemplo) = vale olhar. Alguma diferença é natural: os dois dias têm máquinas e endereços diferentes.")
        display(cm.groupby("par").parecida.mean().mul(100).round(0).rename("% das colunas com medianas parecidas (razão entre 0,5 e 2)").to_frame())
        display(cm)

== Decisão por rótulo ==


,rotulo_treino,rotulo_teste,rotulo_final,documentado_nos_dois_dias,decisao,motivo
0,DrDoS_LDAP,LDAP,LDAP,1,UNIFICAR,"mesmo nome sem o prefixo DrDoS_, e 'LDAP' consta nos dois dias na Tabela III do artigo do dataset"
1,DrDoS_MSSQL,MSSQL,MSSQL,1,UNIFICAR,"mesmo nome sem o prefixo DrDoS_, e 'MSSQL' consta nos dois dias na Tabela III do artigo do dataset"
2,DrDoS_NetBIOS,NetBIOS,NetBIOS,1,UNIFICAR,"mesmo nome sem o prefixo DrDoS_, e 'NetBIOS' consta nos dois dias na Tabela III do artigo do dataset"
3,,Portmap,Portmap,0,SEM PAR,nenhum rótulo do treino tem o mesmo nome (sem o prefixo DrDoS_); fica só no teste
4,DrDoS_UDP,UDP,UDP,1,UNIFICAR,"mesmo nome sem o prefixo DrDoS_, e 'UDP' consta nos dois dias na Tabela III do artigo do dataset"
5,DrDoS_DNS,,DNS,0,REMOVER PREFIXO,tem o prefixo DrDoS_ e não tem par no teste: só o prefixo é removido
6,DrDoS_NTP,,NTP,0,REMOVER PREFIXO,tem o prefixo DrDoS_ e não tem par no teste: só o prefixo é removido
7,DrDoS_SNMP,,SNMP,0,REMOVER PREFIXO,tem o prefixo DrDoS_ e não tem par no teste: só o prefixo é removido
8,DrDoS_SSDP,,SSDP,0,REMOVER PREFIXO,tem o prefixo DrDoS_ e não tem par no teste: só o prefixo é removido



MAPA_UNIF (rótulo do treino -> rótulo final): {'DrDoS_LDAP': 'LDAP', 'DrDoS_MSSQL': 'MSSQL', 'DrDoS_NetBIOS': 'NetBIOS', 'DrDoS_UDP': 'UDP', 'DrDoS_DNS': 'DNS', 'DrDoS_NTP': 'NTP', 'DrDoS_SNMP': 'SNMP', 'DrDoS_SSDP': 'SSDP'}
Prefixo DrDoS_ removido dos rótulos do treino (label_original continua com o nome original). Para manter: REMOVER_PREFIXO_DRDOS = False.

== Conferência INFORMATIVA (não decide nada): medianas de colunas-chave, treino × teste ==
Razão perto de 1 = parecidas; muito diferente (10 vezes, por exemplo) = vale olhar. Alguma diferença é natural: os dois dias têm máquinas e endereços diferentes.


,"% das colunas com medianas parecidas (razão entre 0,5 e 2)"
par,
DrDoS_LDAP → LDAP,75.0
DrDoS_MSSQL → MSSQL,100.0
DrDoS_NetBIOS → NetBIOS,75.0
DrDoS_UDP → UDP,62.0


,par,coluna,mediana_treino,mediana_teste,razao_teste_treino,parecida
0,DrDoS_LDAP → LDAP,flow_duration,39.000,1.000,0.026,False
1,DrDoS_LDAP → LDAP,total_fwd_packets,2.000,2.000,1.000,True
2,DrDoS_LDAP → LDAP,fwd_packet_length_max,1464.000,1472.000,1.005,True
3,DrDoS_LDAP → LDAP,flow_iat_mean,31.000,1.000,0.032,False
4,DrDoS_LDAP → LDAP,min_packet_length,1464.000,1472.000,1.005,True
5,DrDoS_LDAP → LDAP,max_packet_length,1464.000,1472.000,1.005,True
6,DrDoS_LDAP → LDAP,average_packet_size,2184.000,2208.000,1.011,True
7,DrDoS_LDAP → LDAP,packet_length_std,0.000,0.000,1.000,True
8,DrDoS_MSSQL → MSSQL,flow_duration,2.000,1.000,0.500,True
9,DrDoS_MSSQL → MSSQL,total_fwd_packets,2.000,2.000,1.000,True


In [46]:
# ---------- 11.3 Tabela de auditoria, views unificadas e efeito nas classes ----------
def expr_unif(col="label_multi"):
    if not MAPA_UNIF: return f"toString({col})"
    ks = "[" + ",".join(lit(k) for k in MAPA_UNIF) + "]"; vs = "[" + ",".join(lit(v) for v in MAPA_UNIF.values()) + "]"
    return f"transform(toString({col}), {ks}, {vs}, toString({col}))"

versao_unif = datetime.now().strftime("%Y%m%d_%H%M%S")
executar("DROP TABLE IF EXISTS `prep_mapa_rotulos_unif`")
executar("CREATE TABLE prep_mapa_rotulos_unif (rotulo_treino String, rotulo_teste String, rotulo_final String, documentado_nos_dois_dias UInt8, decisao String, motivo String, unificado UInt8, versao String) "
         "ENGINE = MergeTree ORDER BY (rotulo_teste, rotulo_treino)")
if len(tab_unif):
    aud = tab_unif[["rotulo_treino", "rotulo_teste", "rotulo_final", "documentado_nos_dois_dias", "decisao", "motivo"]].copy()
    aud["unificado"] = aud.decisao.str.startswith("UNIFICAR").astype(int); aud["versao"] = versao_unif
    inserir_df("prep_mapa_rotulos_unif", aud)
estado_set("mapa_unif", MAPA_UNIF)
tab_unif.to_csv(ARTEFATOS / "evidencia_unificacao_rotulos.csv", index=False)

feats_v = ", ".join([q(c) for c in FEATURES_BASE] + [q(c + "_ausente") for c in COLS_NULL_BASE])
fora = ", ".join(lit(c) for c in CLASSES_FORA_MULTI); filtro_multi = f" WHERE label_multi NOT IN ({fora})" if CLASSES_FORA_MULTI else ""
for conj, tab in [("treino", "prep_final_treino"), ("teste", "prep_final_teste")]:
    executar(f"DROP VIEW IF EXISTS `prep_vw_{conj}_multi_unif`")
    executar(f"CREATE VIEW prep_vw_{conj}_multi_unif AS SELECT id_linha, {expr_unif()} AS y, {feats_v} FROM {tab}{filtro_multi}")
print("Views criadas: prep_vw_treino_multi_unif e prep_vw_teste_multi_unif (y = rótulo unificado). As views antigas e as tabelas finais não foram alteradas.\n")

d = consultar_num(f"SELECT 'treino' AS conjunto, {expr_unif()} AS classe, count() AS linhas FROM prep_final_treino GROUP BY classe "
                  f"UNION ALL SELECT 'teste', {expr_unif()} AS classe, count() FROM prep_final_teste GROUP BY classe")
cls = d.pivot_table(index="classe", columns="conjunto", values="linhas", aggfunc="sum", fill_value=0).reindex(columns=["treino", "teste"], fill_value=0).astype(int)
cls["onde existe"] = np.where((cls.treino > 0) & (cls.teste > 0), "treino e teste", np.where(cls.treino > 0, "só no treino", "só no teste"))
print("== Classes DEPOIS da unificação =="); display(cls.sort_values(["onde existe", "treino"], ascending=[True, False]))
print("Só no teste:", cls.index[cls["onde existe"] == "só no teste"].tolist() or "nenhuma", "| só no treino:", cls.index[cls["onde existe"] == "só no treino"].tolist() or "nenhuma")
RELATORIO["unificacao"] = {"mapa": MAPA_UNIF, "decisoes": tab_unif[["rotulo_treino", "rotulo_teste", "decisao"]].to_dict("records")}

Views criadas: prep_vw_treino_multi_unif e prep_vw_teste_multi_unif (y = rótulo unificado). As views antigas e as tabelas finais não foram alteradas.

== Classes DEPOIS da unificação ==


conjunto,treino,teste,onde existe
classe,,,
Portmap,0,186960,só no teste
TFTP,4396919,0,só no treino
NTP,1112936,0,só no treino
SSDP,891221,0,só no treino
SNMP,115316,0,só no treino
DNS,113259,0,só no treino
WebDDoS,414,0,só no treino
UDP,1075337,3867155,treino e teste
MSSQL,207003,5787453,treino e teste


Só no teste: ['Portmap'] | só no treino: ['DNS', 'NTP', 'SNMP', 'SSDP', 'TFTP', 'WebDDoS']


## 12. Seleção de atributos (AUC + informação mútua + remoção de colunas gêmeas)

### Por que selecionar atributos?
Temos ~59 colunas. Algumas **não ajudam** a separar normal de ataque (ruído), e outras **repetem a mesma informação** (colunas "gêmeas"). Tirá-las deixa o treino mais rápido (principalmente no SVM) e o resultado mais fácil de explicar. **Isso não garante nota maior**: por isso a seleção é conferida no treino (veja "Cuidados").

### O que significa "selecionar sem treinar"
É um **método de filtro**: avaliamos **cada coluna sozinha**, olhando a relação dela com a classe, sem treinar nenhum modelo. Serve como **triagem**. A confirmação final é treinar com todas as colunas e só com as selecionadas e comparar.

### As três medidas (com exemplos inventados)

| Medida | Pergunta | Exemplo | Como ler |
|---|---|---|---|
| **AUC** | Sorteando uma linha de ataque e uma normal, qual a chance de a coluna ser maior no ataque? | `flow_iat_std`: em 98 de 100 pares, o ataque tem valor maior → AUC = 0,98 | 0,5 = inútil; perto de 1 **ou** de 0 = separa bem. Usamos a **força** = \|AUC − 0,5\| × 2 (de 0 a 1), então 0,98 → 0,96 e 0,02 → 0,96 também |
| **Informação mútua (MI)** | Saber o valor da coluna **diminui** a dúvida sobre a classe? | Uma coluna em que o ataque aparece nos valores **muito baixos e muito altos**, e o normal no meio, tem AUC ≈ 0,5 (parece inútil), mas MI **alta** | 0 = nada; maior = mais útil. Pega relações que não são "maior ou menor" |
| **\|r\| (correlação)** | Duas colunas andam juntas? | `total_fwd_packets` e `total_length_of_fwd_packets` quase sempre sobem juntas → \|r\| = 0,99 | perto de 1 = quase a mesma informação ("gêmeas"). O sinal não importa (por isso as barras) |

**Por que usar AUC e MI juntas?** Cada uma enxerga uma coisa que a outra não vê. O AUC só vê "maior ou menor" e a MI vê qualquer padrão, mas é mais ruidosa. Uma coluna boa em **qualquer uma** merece atenção.

### Passo a passo
1. **Amostra do treino** (só `prep_final_treino`; o teste não entra): **todos os BENIGN** + cerca de 5.000 linhas por tipo de ataque. Os vazios são trocados pela mediana do treino (`PreProc`).
2. **Calcula AUC (força) e MI de cada coluna**, uma vez para o **binário** (BENIGN × ataque) e outra para o **multiclasse** (com os rótulos já unificados da seção 11).
   - No multiclasse, o AUC é "uma classe contra as outras", e guardamos o **maior** valor entre as classes: uma coluna pode ser ótima para um ataque só.
3. **Nota** = média das **posições** (percentis) da coluna no ranking de AUC e no ranking de MI. Usamos posições, e não os valores, porque AUC e MI têm escalas diferentes.
4. **Remove colunas inúteis**: força de AUC < **0,02** *e* MI relativa (à maior MI) < **0,02**. Só sai se for fraca nas **duas**.
5. **Remove colunas gêmeas**: percorrendo da maior para a menor nota, cada coluna é comparada com as já mantidas; se o \|r\| com alguma for ≥ **0,95**, ela é descartada (a de **maior nota fica**).
6. As colunas `<feature>_ausente` seguem a coluna de origem: ficam se ela ficar.

### Exemplo completo com 4 colunas (números inventados)

| Coluna | Força de AUC | MI relativa | Nota (posições) | \|r\| com A | Decisão |
|---|---|---|---|---|---|
| **A** | 0,95 | 1,00 | 1,00 | — | **mantida** |
| **B** | 0,94 | 0,97 | 0,75 | **0,99** | **removida: gêmea de A** (nota menor) |
| **C** | 0,13 | 0,50 | 0,50 | 0,10 | **mantida** (AUC fraca, mas a MI mostra que ajuda) |
| **D** | 0,01 | 0,01 | 0,25 | 0,02 | **removida: inútil** (fraca nas duas) |

### Resultado (nada existente é alterado)
- tabela `prep_selecao_atributos`: todas as notas, a decisão e o motivo de cada coluna (a próxima célula explica como ler);
- views novas: `prep_vw_{treino,teste}_binario_sel` e `prep_vw_{treino,teste}_multi_unif_sel`;
- a função `carregar_selecao("binario" | "multi")` devolve a lista final.

### Cuidados
- **Colunas "boas demais"** (força de AUC ≥ 0,80) ficam marcadas como `atalho_suspeito`. **Não são removidas**: se são um atalho do dataset ou um sinal real, só um teste dirá. Faça a **ablação** no treino: compare o modelo com e sem elas.
- Filtros olham uma coluna por vez e **não captam interações** entre colunas. Por isso, no treino, compare o RF com **todas** as colunas e só com as **selecionadas** numa validação; se a nota se mantém, a seleção está boa.
- A seleção usa só o treino. As linhas da futura validação fazem parte dele, o que causa um viés pequeno. Para ser rigoroso, repita este ranking dentro de cada dobra da validação cruzada.
- **Nunca** escolha atributos olhando o teste.
- Os limiares (0,02 e 0,95) são padrões razoáveis, não regras. Se quiser mais ou menos colunas, ajuste `AUC_FORCA_MIN`, `MI_REL_MIN` e `R_MAX`, ou use `TOP_K`.

In [47]:
# ---------- 12.1 Parâmetros e amostra do treino ----------
AUC_FORCA_MIN = globals().get("AUC_FORCA_MIN", 0.02)          # força de AUC mínima  (|AUC-0,5|*2)
MI_REL_MIN = globals().get("MI_REL_MIN", 0.02)                # MI mínima relativa à maior MI
R_MAX = globals().get("R_MAX", 0.95)                          # |r| a partir do qual duas colunas são "gêmeas"
ATALHO_AUC = globals().get("ATALHO_AUC", 0.80)                # força de AUC acima da qual a coluna é marcada como atalho suspeito
AMOSTRA_SEL_POR_CLASSE = globals().get("AMOSTRA_SEL_POR_CLASSE", 5000)   # ~linhas por tipo de ataque (todos os BENIGN entram)
MI_MAX_LINHAS = globals().get("MI_MAX_LINHAS", 40_000)        # a MI usa no máximo isto (amostra estratificada), por custo
TOP_K = globals().get("TOP_K", None)                          # opcional: manter no máximo K colunas
restaurar_selecao()

def _amostra_sel():
    ct = consultar_num(f"SELECT {expr_unif()} AS r, count() AS n FROM prep_final_treino GROUP BY r")
    ks = list(ct.r); ms = [1 if k == "BENIGN" else max(1, -(-int(n) // AMOSTRA_SEL_POR_CLASSE)) for k, n in zip(ct.r, ct.n)]
    filtro = f"cityHash64(id_linha) % transform({expr_unif()}, [{','.join(lit(k) for k in ks)}], [{','.join(str(m) for m in ms)}], 1) = 0"
    feats = ", ".join(q(c) for c in FEATURES_BASE)
    return rodar_pesado(f"SELECT label_binary, {expr_unif()} AS label_unif, {feats} FROM prep_final_treino WHERE {filtro} @@S@@")

### Como ler as tabelas da próxima célula

| Coluna | O que é |
|---|---|
| `posicao` | Ordem pela nota (1 = a melhor coluna) |
| `auc_forca` | Força do AUC, de 0 a 1 (0 = não separa; 1 = separa perfeitamente, em qualquer sentido) |
| `mi` e `mi_rel` | Informação mútua, e a MI relativa à maior MI (1,0 = a melhor) |
| `score` | Nota final: média das posições em AUC e MI (maior = melhor) |
| `decisao` | `mantida`, `removida_inutil` (fraca nas duas medidas), `removida_gemea` (quase igual a outra coluna de nota maior) ou `removida_top_k` (se você definiu `TOP_K`) |
| `gemea_de` | Para as gêmeas removidas: qual coluna **ficou no lugar** dela |
| `atalho_suspeito` | 1 = força de AUC ≥ 0,80: coluna "boa demais"; **mantida**, mas vale a ablação |

**Exemplo de leitura:** a linha `total_length_of_fwd_packets | removida_gemea | gemea_de = total_fwd_packets` quer dizer: essa coluna é praticamente a mesma informação de `total_fwd_packets`, que tem nota maior, então **só `total_fwd_packets` fica**.
Há duas listas porque o **binário** e o **multiclasse** têm notas diferentes: uma coluna pode ser ótima para separar um ataque específico (multiclasse) e pouco útil para separar "normal × ataque" (binário).

In [49]:
%pip install scikit-learn

   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------- ----------------------- 3.4/8.2 MB 17.6 MB/s eta 0:00:01
   ------------------------ --------------- 5.0/8.2 MB 12.0 MB/s eta 0:00:01
   ------------------------------- -------- 6.6/8.2 MB 10.3 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 10.1 MB/s  0:00:00
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ------ --------------------------------- 5.5/36.6 MB 26.4 MB/s eta 0:00:02
   ----------- ---------------------------- 10.7/36.6 MB 25.1 MB/s eta 0:00:02
   ---------------- ----------------------- 15.5/36.6 MB 24.5 MB/s eta 0:00:01
   ---------------------- ----------------- 21.0/36.6 MB 24.8 MB/s eta 0:00:01
   ----------------------------- ---------- 26.7/36.6 MB 25.1 MB/s eta 0:00:01
   ----------------------------------- ---- 32.2/36.6 MB 25.5 MB/s eta 0:00:01
   ---------------------------------------  36.4/36.6 MB 25.7 MB/s eta 0:00:01
   -

In [50]:
# ---------- 12.2 AUC, informação mútua, correlação e seleção ----------
def _selecionar(forca, mi, R):
    pa, pm = forca.rank(pct=True), mi.rank(pct=True); score = (pa + pm) / 2
    mi_rel = mi / mi.max() if mi.max() > 0 else mi * 0
    mant, linhas = [], []
    for pos, c in enumerate(score.sort_values(ascending=False).index, 1):
        inutil = forca[c] < AUC_FORCA_MIN and mi_rel[c] < MI_REL_MIN
        if inutil:
            dec, gem, mot = "removida_inutil", "", f"força de AUC {forca[c]:.3f} < {AUC_FORCA_MIN} e MI relativa {mi_rel[c]:.3f} < {MI_REL_MIN}"
        else:
            g = next((m for m in mant if R.loc[c, m] >= R_MAX), None)
            if g is not None: dec, gem, mot = "removida_gemea", g, f"|r| = {R.loc[c, g]:.3f} com {g}, que tem nota maior"
            elif TOP_K and len(mant) >= TOP_K: dec, gem, mot = "removida_top_k", "", f"fora das {TOP_K} melhores"
            else: dec, gem, mot = "mantida", "", ""; mant.append(c)
        linhas.append({"feature": c, "posicao": pos, "auc_forca": round(float(forca[c]), 4), "mi": round(float(mi[c]), 5), "mi_rel": round(float(mi_rel[c]), 4), "score": round(float(score[c]), 4),
                       "decisao": dec, "gemea_de": gem, "motivo": mot, "atalho_suspeito": int(forca[c] >= ATALHO_AUC)})
    return linhas

def _calc_selecao_atributos():
    from sklearn.metrics import roc_auc_score
    from sklearn.feature_selection import mutual_info_classif
    df = _amostra_sel(); feats = list(pp.features)
    X = pp.transformar(df, "rf")[feats]
    yb = df["label_binary"].astype(int).values; yu = df["label_unif"].astype(str).values
    auc_b = pd.Series({c: roc_auc_score(yb, X[c].values) for c in feats}); forca_b = (auc_b - 0.5).abs() * 2
    classes = [k for k in sorted(set(yu)) if 2 <= (yu == k).sum() <= len(yu) - 2]
    forca_m = pd.DataFrame({k: {c: abs(roc_auc_score(yu == k, X[c].values) - 0.5) * 2 for c in feats} for k in classes}).max(axis=1)
    frac = min(1.0, MI_MAX_LINHAS / len(df)); idx = df.groupby("label_unif", group_keys=False).sample(frac=frac, random_state=0).index
    Xs = X.loc[idx].values; cod = pd.factorize(pd.Series(yu, index=df.index).loc[idx])[0]
    mi_b = pd.Series(mutual_info_classif(Xs, yb[df.index.get_indexer(idx)], random_state=0), index=feats)
    mi_m = pd.Series(mutual_info_classif(Xs, cod, random_state=0), index=feats)
    R = X.corr().abs().fillna(0)
    return {"binario": _selecionar(forca_b, mi_b, R), "multi": _selecionar(forca_m, mi_m, R),
            "amostra": {"linhas": int(len(df)), "benign": int((yb == 0).sum()), "linhas_mi": int(len(idx)), "classes": {k: int((yu == k).sum()) for k in sorted(set(yu))}}}

_sig_sel = assinatura(SIG_FEATS, n_linhas("prep_final_treino"), sorted(MAPA_UNIF.items()), AUC_FORCA_MIN, MI_REL_MIN, R_MAX, ATALHO_AUC, AMOSTRA_SEL_POR_CLASSE, MI_MAX_LINHAS, TOP_K, "sel1")
_res_sel, _ = etapa("b10_selecao_atributos", _sig_sel, _calc_selecao_atributos)
sel_bin, sel_multi = pd.DataFrame(_res_sel["binario"]), pd.DataFrame(_res_sel["multi"])
SEL_BIN, SEL_MULTI = sel_bin.loc[sel_bin.decisao == "mantida", "feature"].tolist(), sel_multi.loc[sel_multi.decisao == "mantida", "feature"].tolist()
a = _res_sel["amostra"]; print(f"Amostra do treino: {a['linhas']:,} linhas ({a['benign']:,} BENIGN); MI calculada em {a['linhas_mi']:,}.")
for nome, t, sel in (("BINÁRIO", sel_bin, SEL_BIN), ("MULTICLASSE (rótulos unificados)", sel_multi, SEL_MULTI)):
    print(f"\n== {nome}: {len(sel)} de {len(t)} colunas mantidas ==")
    print(t.decisao.value_counts().to_string())
    display(t.head(15)[["posicao", "feature", "auc_forca", "mi", "score", "decisao", "gemea_de", "atalho_suspeito"]])
    rem = t[t.decisao != "mantida"]
    if len(rem): print("Removidas:"); display(rem[["feature", "auc_forca", "mi_rel", "decisao", "gemea_de", "motivo"]].reset_index(drop=True))
    at = t[t.atalho_suspeito == 1].feature.tolist()
    if at: print("⚠ Atalhos suspeitos (força de AUC >= %.2f; mantidos, mas faça a ablação no treino):" % ATALHO_AUC, at)

  conexão instável (RemoteDisconnected: Remote end closed connection without response); nova tentativa em 5s...
  ClickHouse com merges internos usando 658.21 MiB; aguardando terminarem (máx 600s)...
  ✔ servidor livre; seguindo.
  conexão instável (RemoteDisconnected: Remote end closed connection without response); nova tentativa em 5s...
Amostra do treino: 104,730 linhas (50,768 BENIGN); MI calculada em 39,999.

== BINÁRIO: 45 de 61 colunas mantidas ==
decisao
mantida            45
removida_gemea     15
removida_inutil     1


,posicao,feature,auc_forca,mi,score,decisao,gemea_de,atalho_suspeito
0,1,bwd_packets_s,0.8441,0.47667,0.9426,mantida,,1
1,2,avg_fwd_segment_size,0.6901,0.49182,0.9098,mantida,,0
2,3,inbound,0.8441,0.46376,0.9016,mantida,,1
3,4,fwd_packet_length_min,0.7410,0.47883,0.8852,removida_gemea,avg_fwd_segment_size,0
4,5,fwd_packet_length_mean,0.6901,0.48980,0.8852,removida_gemea,avg_fwd_segment_size,0
5,6,min_packet_length,0.7498,0.47645,0.8689,removida_gemea,avg_fwd_segment_size,0
6,7,average_packet_size,0.6620,0.47958,0.8607,removida_gemea,avg_fwd_segment_size,0
7,8,packet_length_mean,0.6513,0.47870,0.8361,removida_gemea,avg_fwd_segment_size,0
8,9,bwd_header_length,0.7973,0.39578,0.8279,mantida,,0
9,10,total_backward_packets,0.8082,0.39156,0.8279,mantida,,1


Removidas:


,feature,auc_forca,mi_rel,decisao,gemea_de,motivo
0,fwd_packet_length_min,0.7410,0.9736,removida_gemea,avg_fwd_segment_size,"|r| = 0.988 com avg_fwd_segment_size, que tem nota maior"
1,fwd_packet_length_mean,0.6901,0.9959,removida_gemea,avg_fwd_segment_size,"|r| = 1.000 com avg_fwd_segment_size, que tem nota maior"
2,min_packet_length,0.7498,0.9688,removida_gemea,avg_fwd_segment_size,"|r| = 0.985 com avg_fwd_segment_size, que tem nota maior"
3,average_packet_size,0.6620,0.9751,removida_gemea,avg_fwd_segment_size,"|r| = 0.978 com avg_fwd_segment_size, que tem nota maior"
4,packet_length_mean,0.6513,0.9733,removida_gemea,avg_fwd_segment_size,"|r| = 0.971 com avg_fwd_segment_size, que tem nota maior"
5,bwd_packet_length_mean,0.5923,0.5570,removida_gemea,avg_bwd_segment_size,"|r| = 1.000 com avg_bwd_segment_size, que tem nota maior"
6,fwd_iat_max,0.3505,0.6945,removida_gemea,flow_iat_max,"|r| = 0.993 com flow_iat_max, que tem nota maior"
7,flow_packets_s,0.0974,0.9575,removida_gemea,fwd_packets_s,"|r| = 0.998 com fwd_packets_s, que tem nota maior"
8,fwd_iat_mean,0.3698,0.6246,removida_gemea,flow_iat_mean,"|r| = 0.969 com flow_iat_mean, que tem nota maior"
9,fwd_iat_total,0.3448,0.6204,removida_gemea,flow_duration,"|r| = 0.998 com flow_duration, que tem nota maior"


⚠ Atalhos suspeitos (força de AUC >= 0.80; mantidos, mas faça a ablação no treino): ['bwd_packets_s', 'inbound', 'total_backward_packets']

== MULTICLASSE (rótulos unificados): 45 de 61 colunas mantidas ==
decisao
mantida            45
removida_gemea     15
removida_inutil     1


,posicao,feature,auc_forca,mi,score,decisao,gemea_de,atalho_suspeito
0,1,avg_fwd_segment_size,0.9392,1.34200,0.9754,mantida,,1
1,2,min_packet_length,0.9405,1.33356,0.9672,removida_gemea,avg_fwd_segment_size,1
2,3,fwd_packet_length_mean,0.9392,1.33778,0.9508,removida_gemea,avg_fwd_segment_size,1
3,4,fwd_packet_length_min,0.9399,1.33312,0.9508,removida_gemea,avg_fwd_segment_size,1
4,5,average_packet_size,0.9298,1.33859,0.9344,removida_gemea,avg_fwd_segment_size,1
5,6,packet_length_mean,0.9372,1.33067,0.9098,removida_gemea,avg_fwd_segment_size,1
6,7,total_length_of_fwd_packets,0.9356,1.28474,0.8852,mantida,,1
7,8,fwd_packet_length_max,0.9136,1.30334,0.8770,mantida,,1
8,9,flow_packets_s,0.8766,1.12315,0.8279,mantida,,1
9,10,act_data_pkt_fwd,0.9392,0.69202,0.8115,mantida,,1


Removidas:


,feature,auc_forca,mi_rel,decisao,gemea_de,motivo
0,min_packet_length,0.9405,0.9937,removida_gemea,avg_fwd_segment_size,"|r| = 0.985 com avg_fwd_segment_size, que tem nota maior"
1,fwd_packet_length_mean,0.9392,0.9969,removida_gemea,avg_fwd_segment_size,"|r| = 1.000 com avg_fwd_segment_size, que tem nota maior"
2,fwd_packet_length_min,0.9399,0.9934,removida_gemea,avg_fwd_segment_size,"|r| = 0.988 com avg_fwd_segment_size, que tem nota maior"
3,average_packet_size,0.9298,0.9975,removida_gemea,avg_fwd_segment_size,"|r| = 0.978 com avg_fwd_segment_size, que tem nota maior"
4,packet_length_mean,0.9372,0.9916,removida_gemea,avg_fwd_segment_size,"|r| = 0.971 com avg_fwd_segment_size, que tem nota maior"
5,fwd_iat_mean,0.8855,0.7171,removida_gemea,flow_iat_mean,"|r| = 0.969 com flow_iat_mean, que tem nota maior"
6,fwd_packets_s,0.8491,0.8327,removida_gemea,flow_packets_s,"|r| = 0.998 com flow_packets_s, que tem nota maior"
7,fwd_iat_total,0.8544,0.7381,removida_gemea,flow_duration,"|r| = 0.998 com flow_duration, que tem nota maior"
8,fwd_iat_max,0.8457,0.7262,removida_gemea,flow_iat_max,"|r| = 0.993 com flow_iat_max, que tem nota maior"
9,fwd_iat_std,0.8377,0.6179,removida_gemea,flow_iat_std,"|r| = 0.975 com flow_iat_std, que tem nota maior"


⚠ Atalhos suspeitos (força de AUC >= 0.80; mantidos, mas faça a ablação no treino): ['avg_fwd_segment_size', 'min_packet_length', 'fwd_packet_length_mean', 'fwd_packet_length_min', 'average_packet_size', 'packet_length_mean', 'total_length_of_fwd_packets', 'fwd_packet_length_max', 'flow_packets_s', 'act_data_pkt_fwd', 'flow_iat_mean', 'max_packet_length', 'flow_bytes_s', 'fwd_iat_mean', 'fwd_packets_s', 'flow_duration', 'total_fwd_packets', 'fwd_iat_total', 'flow_iat_std', 'flow_iat_max', 'fwd_iat_max', 'fwd_iat_std', 'bwd_packets_s', 'inbound', 'total_backward_packets', 'ack_flag_count', 'idle_std']


In [51]:
# ---------- 12.3 Tabela de auditoria, views com atributos selecionados e função de leitura ----------
versao_sel = datetime.now().strftime("%Y%m%d_%H%M%S")
executar("DROP TABLE IF EXISTS `prep_selecao_atributos`")
executar("CREATE TABLE prep_selecao_atributos (tarefa String, feature String, posicao UInt16, auc_forca Float64, mi Float64, mi_rel Float64, score Float64, decisao String, gemea_de String, "
         "motivo String, atalho_suspeito UInt8, versao String) ENGINE = MergeTree ORDER BY (tarefa, posicao)")
for tarefa, t in (("binario", sel_bin), ("multi", sel_multi)):
    z = t.copy(); z.insert(0, "tarefa", tarefa); z["versao"] = versao_sel
    inserir_df("prep_selecao_atributos", z[["tarefa", "feature", "posicao", "auc_forca", "mi", "mi_rel", "score", "decisao", "gemea_de", "motivo", "atalho_suspeito", "versao"]])

def carregar_selecao(tarefa="binario"):
    # lista de atributos selecionados, em ordem de nota (tarefa: 'binario' ou 'multi')
    return consultar(f"SELECT feature FROM prep_selecao_atributos WHERE tarefa = {lit(tarefa)} AND decisao = 'mantida' ORDER BY posicao")["feature"].tolist()

def _feats_view(sel):
    return ", ".join([q(c) for c in sel] + [q(c + "_ausente") for c in COLS_NULL_BASE if c in sel])

for conj, tab in [("treino", "prep_final_treino"), ("teste", "prep_final_teste")]:
    executar(f"DROP VIEW IF EXISTS `prep_vw_{conj}_binario_sel`")
    executar(f"CREATE VIEW prep_vw_{conj}_binario_sel AS SELECT id_linha, label_binary AS y, {_feats_view(SEL_BIN)} FROM {tab}")
    executar(f"DROP VIEW IF EXISTS `prep_vw_{conj}_multi_unif_sel`")
    executar(f"CREATE VIEW prep_vw_{conj}_multi_unif_sel AS SELECT id_linha, {expr_unif()} AS y, {_feats_view(SEL_MULTI)} FROM {tab}{filtro_multi}")
(ARTEFATOS / "prep_selecao_atributos_ULTIMO.json").write_text(json.dumps({"versao": versao_sel, "binario": SEL_BIN, "multi": SEL_MULTI, "mapa_unificacao": MAPA_UNIF,
    "parametros": {"AUC_FORCA_MIN": AUC_FORCA_MIN, "MI_REL_MIN": MI_REL_MIN, "R_MAX": R_MAX, "AMOSTRA_SEL_POR_CLASSE": AMOSTRA_SEL_POR_CLASSE, "TOP_K": TOP_K}}, indent=1, ensure_ascii=False), encoding="utf-8")
estado_set("selecao_atributos", {"binario": SEL_BIN, "multi": SEL_MULTI})
RELATORIO["selecao_atributos"] = {"binario": len(SEL_BIN), "multi": len(SEL_MULTI), "candidatas": len(FEATURES_BASE)}
print(f"Views criadas: prep_vw_{{treino,teste}}_binario_sel ({len(SEL_BIN)} atributos) e prep_vw_{{treino,teste}}_multi_unif_sel ({len(SEL_MULTI)} atributos).")
print("As views antigas (prep_vw_*_binario e *_multi) e as tabelas finais não foram alteradas.")

Views criadas: prep_vw_{treino,teste}_binario_sel (45 atributos) e prep_vw_{treino,teste}_multi_unif_sel (45 atributos).
As views antigas (prep_vw_*_binario e *_multi) e as tabelas finais não foram alteradas.


## 13. Quais views usar em cada experimento

| Experimento | View de treino | View de teste | Atributos |
|---|---|---|---|
| **Binário, todos os atributos** (linha de base) | `prep_vw_treino_binario` | `prep_vw_teste_binario` | todos (`FEATURES_BASE`) |
| **Binário, atributos selecionados** | `prep_vw_treino_binario_sel` | `prep_vw_teste_binario_sel` | `carregar_selecao("binario")` |
| **Multiclasse, classes unificadas, todos os atributos** | `prep_vw_treino_multi_unif` | `prep_vw_teste_multi_unif` | todos |
| **Multiclasse, unificada e selecionada** | `prep_vw_treino_multi_unif_sel` | `prep_vw_teste_multi_unif_sel` | `carregar_selecao("multi")` |
| Multiclasse **sem** unificar (só para comparar) | `prep_vw_treino_multi` | `prep_vw_teste_multi` | todos |

Como ler no Python (a coluna `y` é a resposta; `id_linha` não entra na entrada do modelo):
```python
pp = PreProc.carregar()
sel = carregar_selecao("binario")
tr = ler_base("treino", "binario_sel", amostra_mod=20)           # ~5% do treino
X = pp.transformar(tr, "rf", features=sel)                        # "svm" para o SVM; sel = só as selecionadas
y = tr["y"]
```
Para comparar RF e SVM de forma justa, use **o mesmo conjunto de atributos** nos dois.

In [52]:
# ---------- 13.1 Validação das etapas 11 e 12 ----------
ok2 = []
def verifica2(nome, cond, detalhe=""):
    ok2.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)

lab_tr = set(rot_treino.rotulo)
verifica2("Unificação: todo rótulo mapeado existe no treino e o destino não existe no treino", all(k in lab_tr and v not in lab_tr for k, v in MAPA_UNIF.items()), f"({len(MAPA_UNIF)} mapeamentos)")
verifica2("Unificação: nenhum destino é origem de outro mapeamento (sem cadeias)", not (set(MAPA_UNIF.values()) & set(MAPA_UNIF)))
n_tr, n_te = n_linhas("prep_final_treino"), n_linhas("prep_final_teste")
exc_tr = int(consultar_num(f"SELECT count() AS n FROM prep_final_treino WHERE label_multi IN ({', '.join(lit(c) for c in CLASSES_FORA_MULTI)})").n[0]) if CLASSES_FORA_MULTI else 0
verifica2("Views multi_unif: mesmas linhas das bases finais (menos classes excluídas do multiclasse)", n_linhas("prep_vw_treino_multi_unif") == n_tr - exc_tr and n_linhas("prep_vw_teste_multi_unif") <= n_te)
verifica2("Views binario_sel: mesmas linhas das bases finais", n_linhas("prep_vw_treino_binario_sel") == n_tr and n_linhas("prep_vw_teste_binario_sel") == n_te, f"({n_tr:,} / {n_te:,})")
dist_u = consultar_num("SELECT y, count() AS n FROM prep_vw_treino_multi_unif GROUP BY y")
verifica2("Treino unificado: soma das classes = linhas da view", int(dist_u.n.sum()) == n_linhas("prep_vw_treino_multi_unif"), f"({len(dist_u)} classes)")
if REMOVER_PREFIXO_DRDOS:
    mantidos_sep = {r.rotulo_treino for r in tab_unif.itertuples() if r.rotulo_treino and str(r.decisao).startswith("NÃO UNIFICAR")}
    ficaram = sorted(y for y in dist_u.y.astype(str) if y.lower().startswith("drdos_"))
    verifica2("Rótulos unificados sem o prefixo DrDoS_ (exceto os que você mandou manter separados)", set(ficaram) <= mantidos_sep, f"(com prefixo: {ficaram or 'nenhum'})")
    verifica2("label_original preservado (continua com o prefixo nas bases finais)", int(consultar_num("SELECT countIf(label_original LIKE 'DrDoS\\_%') AS n FROM prep_final_treino").n[0]) > 0 or not any(str(x).lower().startswith("drdos_") for x in rot_treino.rotulo))
for nome, sel in (("binario", SEL_BIN), ("multi", SEL_MULTI)):
    verifica2(f"Seleção {nome}: lista não vazia e contida em FEATURES_BASE", len(sel) > 0 and set(sel) <= set(FEATURES_BASE), f"({len(sel)} de {len(FEATURES_BASE)})")
    verifica2(f"Seleção {nome}: carregar_selecao() devolve a mesma lista", carregar_selecao(nome) == sel)
cols_v = consultar("SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = 'prep_vw_treino_binario_sel'")["name"].tolist()
esperadas = ["id_linha", "y"] + SEL_BIN + [c + "_ausente" for c in COLS_NULL_BASE if c in SEL_BIN]
verifica2("View binario_sel: colunas = id_linha, y, atributos selecionados e seus indicadores", cols_v == esperadas, f"({len(cols_v)} colunas)")
proib = {"label_original", "label_multi", "label_binary", "origem_tabela", "n_repeticoes", "conflito_rotulo", "conflito_binario", "sobrepoe_treino", "sobrepoe_treino_mesmo_rotulo", "b"}
verifica2("Nenhuma view nova expõe rótulo bruto ou marca de controle como entrada", not (proib & set(cols_v)))
assert all(ok2), "há checagens reprovadas acima"
print(f"\n✔ {sum(ok2)}/{len(ok2)} checagens das etapas 11 e 12 aprovadas.")

✔ Unificação: todo rótulo mapeado existe no treino e o destino não existe no treino (8 mapeamentos)
✔ Unificação: nenhum destino é origem de outro mapeamento (sem cadeias) 
✔ Views multi_unif: mesmas linhas das bases finais (menos classes excluídas do multiclasse) 
✔ Views binario_sel: mesmas linhas das bases finais (8,257,343 / 20,364,525)
✔ Treino unificado: soma das classes = linhas da view (13 classes)
✔ Rótulos unificados sem o prefixo DrDoS_ (exceto os que você mandou manter separados) (com prefixo: nenhum)
✔ label_original preservado (continua com o prefixo nas bases finais) 
✔ Seleção binario: lista não vazia e contida em FEATURES_BASE (45 de 61)
✔ Seleção binario: carregar_selecao() devolve a mesma lista 
✔ Seleção multi: lista não vazia e contida em FEATURES_BASE (45 de 61)
✔ Seleção multi: carregar_selecao() devolve a mesma lista 
✔ View binario_sel: colunas = id_linha, y, atributos selecionados e seus indicadores (48 colunas)
✔ Nenhuma view nova expõe rótulo bruto ou marca 

## 14. Overview das bases finais (treino e teste)
Tamanho, quantidade de linhas e colunas, distribuição das classes, marcas de repetição/conflito/sobreposição e ausentes. Somente leitura.

In [53]:
def overview_bases_finais():
    gb = lambda x: f"{x / 2 ** 30:,.2f} GiB" if x >= 2 ** 30 else f"{x / 2 ** 20:,.1f} MiB"
    n_feat = len(FEATURES_BASE)
    BASES = (("prep_final_treino", "treino", N_TREINO), ("prep_final_teste", "teste", N_TESTE))
    faltam = [t for t, _, _ in BASES if not existe(t)]
    if faltam:
        print("Bases ainda não existem:", faltam); return
    # 1) tamanho e estrutura
    resumo = {}
    for t, conj, bruto in BASES:
        p = consultar_num(f"SELECT sum(rows) AS linhas, count() AS partes, sum(bytes_on_disk) AS disco, sum(data_uncompressed_bytes) AS descomp "
                          f"FROM system.parts WHERE database = currentDatabase() AND `table` = {lit(t)} AND active").iloc[0]
        cols = consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(t)}")["name"]
        n_ind = int(cols.str.endswith("_ausente").sum()); linhas = int(p.linhas)
        resumo[t] = {"conjunto": conj, "linhas": f"{linhas:,}", "linhas na origem (bruto)": f"{bruto:,}", "variação vs origem": f"{(linhas / bruto - 1) * 100:+.2f}%",
                     "colunas (total)": len(cols), "features numéricas": n_feat, "indicadores *_ausente": n_ind, "partes no disco": int(p.partes),
                     "tamanho em disco (comprimido)": gb(float(p.disco)), "tamanho descomprimido": gb(float(p.descomp)), "razão de compressão": f"{p.descomp / p.disco:.1f}x",
                     "RAM estimada p/ carregar no pandas (float64)": gb(linhas * (n_feat + n_ind) * 8)}
    print("== 1) Tamanho e estrutura =="); display(pd.DataFrame(resumo))
    # 2) classes
    _cls = [("label_binary", "binário (0 = BENIGN, 1 = ataque)"), ("label_multi", "multiclasse")]
    if "MAPA_UNIF" in globals() and "expr_unif" in globals() and MAPA_UNIF: _cls.append((expr_unif(), "multiclasse DEPOIS da unificação (prep_vw_*_multi_unif)"))
    for col, titulo in _cls:
        d = consultar_num(f"SELECT 'treino' AS conjunto, {col} AS classe, count() AS n FROM prep_final_treino GROUP BY {col} "
                          f"UNION ALL SELECT 'teste', {col}, count() FROM prep_final_teste GROUP BY {col}")
        t_ = d.pivot_table(index="classe", columns="conjunto", values="n", aggfunc="sum", fill_value=0).reindex(columns=["treino", "teste"], fill_value=0)
        for c in ("treino", "teste"):
            t_[f"% {c}"] = (100 * t_[c] / t_[c].sum()).round(2)
        print(f"\n== 2) Classes — {titulo} =="); display(t_.sort_values("treino", ascending=False).astype({"treino": int, "teste": int}))
    # 3) por tabela de origem
    o = consultar_num("SELECT * FROM (SELECT 'treino' AS conjunto, origem_tabela, count() AS linhas FROM prep_final_treino GROUP BY origem_tabela "
                      "UNION ALL SELECT 'teste', origem_tabela, count() FROM prep_final_teste GROUP BY origem_tabela) ORDER BY conjunto, linhas DESC")
    print("\n== 3) Linhas por tabela de origem =="); display(o.astype({"linhas": int}))
    # 4) marcas
    tr = consultar_num("SELECT sum(n_repeticoes) AS copias_representadas, min(n_repeticoes) AS min_rep, round(avg(n_repeticoes), 2) AS media_rep, max(n_repeticoes) AS max_rep, "
                       "sum(conflito_rotulo) AS conflito_rotulo, sum(conflito_binario) AS conflito_binario FROM prep_final_treino").iloc[0]
    te = consultar_num("SELECT sum(sobrepoe_treino) AS sobrepoe_treino, sum(sobrepoe_treino_mesmo_rotulo) AS sobrepoe_mesmo_rotulo FROM prep_final_teste").iloc[0]
    print("\n== 4) Marcas ==")
    print(f"  Treino: {int(tr.copias_representadas):,} linhas brutas representadas (n_repeticoes: mín {int(tr.min_rep)}, média {tr.media_rep}, máx {int(tr.max_rep):,}); "
          f"em conflito de rótulo: {int(tr.conflito_rotulo):,}; conflito BENIGN×ataque: {int(tr.conflito_binario):,}")
    print(f"  Teste : {int(te.sobrepoe_treino):,} linhas ({te.sobrepoe_treino / N_TESTE * 100:.2f}%) têm features idênticas a alguma do treino; "
          f"{int(te.sobrepoe_mesmo_rotulo):,} também com o mesmo rótulo (multi)")
    # 5) ausentes (ficam NULL nas bases finais; a imputação é feita com os parâmetros do treino)
    if COLS_NULL_BASE:
        linhas = []
        for t, conj, _ in BASES:
            r = consultar_num("SELECT count() AS n, " + ", ".join(f"countIf(isNull({q(c)})) AS {q(c)}" for c in COLS_NULL_BASE) + f" FROM {t}").iloc[0]
            for c in COLS_NULL_BASE:
                linhas.append({"conjunto": conj, "feature": c, "ausentes": int(r[c]), "% ausentes": round(100 * r[c] / r["n"], 3)})
        print("\n== 5) Features com valores ausentes (NULL) =="); display(pd.DataFrame(linhas))
    # 6) objetos prep_*
    tabs = consultar("SELECT name, engine FROM system.tables WHERE database = currentDatabase() AND name LIKE 'prep_%' ORDER BY name")
    tam = consultar_num("SELECT `table` AS name, sum(rows) AS linhas, sum(bytes_on_disk) AS disco FROM system.parts WHERE database = currentDatabase() AND active AND `table` LIKE 'prep_%' GROUP BY `table`")
    d = tabs.merge(tam, on="name", how="left"); d["tamanho em disco"] = d["disco"].map(lambda x: "" if pd.isna(x) else gb(float(x))); d["linhas"] = d["linhas"].map(lambda x: "" if pd.isna(x) else f"{int(x):,}")
    print("\n== 6) Tabelas e views prep_* =="); display(d[["name", "engine", "linhas", "tamanho em disco"]])
    try:
        livre = consultar("SELECT formatReadableSize(min(free_space)) AS livre, formatReadableSize(min(total_space)) AS total FROM system.disks").iloc[0]
        print(f"\nDisco do servidor: {livre['livre']} livres de {livre['total']}.")
    except Exception:
        pass


overview_bases_finais()

== 1) Tamanho e estrutura ==


,prep_final_treino,prep_final_teste
conjunto,treino,teste
linhas,"8,257,343","20,364,525"
linhas na origem (bruto),"50,063,112","20,364,525"
variação vs origem,-83.51%,+0.00%
colunas (total),72,71
features numéricas,61,61
indicadores *_ausente,2,2
partes no disco,361,44
tamanho em disco (comprimido),800.2 MiB,1.28 GiB
tamanho descomprimido,2.09 GiB,5.63 GiB



== 2) Classes — binário (0 = BENIGN, 1 = ataque) ==


conjunto,treino,teste,% treino,% teste
classe,,,,
1,8206575,20307560,99.39,99.72
0,50768,56965,0.61,0.28



== 2) Classes — multiclasse ==


conjunto,treino,teste,% treino,% teste
classe,,,,
TFTP,4396919,0,53.25,0.00
DrDoS_NTP,1112936,0,13.48,0.00
DrDoS_UDP,1075337,0,13.02,0.00
DrDoS_SSDP,891221,0,10.79,0.00
DrDoS_MSSQL,207003,0,2.51,0.00
Syn,155515,4891500,1.88,24.02
DrDoS_SNMP,115316,0,1.40,0.00
DrDoS_DNS,113259,0,1.37,0.00
UDPLag,89066,1873,1.08,0.01


... (5 outputs left out)

,conjunto,feature,ausentes,% ausentes
0,treino,flow_bytes_s,24915,0.302
1,treino,flow_packets_s,24915,0.302
2,teste,flow_bytes_s,807514,3.965
3,teste,flow_packets_s,807514,3.965



== 6) Tabelas e views prep_* ==


,name,engine,linhas,tamanho em disco
0,prep_aud_invalidos,MergeTree,144,0.0 MiB
1,prep_aux_ids,MergeTree,"70,427,637",3.32 GiB
2,prep_classes_resumo,MergeTree,18,0.0 MiB
3,prep_estado,ReplacingMergeTree,536,0.0 MiB
4,prep_final_teste,MergeTree,"20,364,525",1.28 GiB
5,prep_final_treino,MergeTree,"8,257,343",800.2 MiB
6,prep_grp_treino,MergeTree,"8,041,864",149.3 MiB
7,prep_hash,MergeTree,"70,427,637",783.5 MiB
8,prep_keep_treino,MergeTree,"8,257,343",200.9 MiB
9,prep_mapa_rotulos,MergeTree,19,0.0 MiB



Disco do servidor: 2.59 GiB livres de 18.61 GiB.
